# Information flow in the MOS 6502: pairwise and multivariate transfer entropy against known circuit structure

**RQ1.** To what extent does pairwise transfer entropy identify known directed signal-flow relationships in MOS 6502 transistor-level simulation data?
**RQ2.** How does multivariate transfer entropy using IDTxl improve on pairwise transfer entropy for identifying directed computational structure in a high-fidelity microprocessor?

This is the final, single-run notebook. Run it top to bottom. Every expensive step checkpoints and resumes. Section 17 checks that the run is complete and internally consistent before any number is reported.

## Document map

| Part | Sections | Thesis chapter |
|---|---|---|
| I. Data and core estimates | 0 configuration, JVM and bit-correct significance; 1 recording and reduction; 2 MI, AIS and pairwise TE (JIDT); 3 structural reference; 4 provenance | Data and Methods |
| II. Validation | 5 analysis setup and toolkit; 6 estimator validation (units audit, empirical null); 7 synthetic ground-truth logic circuits | Methods validation |
| III. RQ1 — pairwise TE | 8.1–8.9 | Results I |
| IV. RQ2 — multivariate TE | 9 IDTxl multivariate TE (with IDTxl bivariate TE as the matched pairwise comparator), all active targets | Results II |
| V. Ground-truth 6502 | 10 validated simulator; 11 interventions and lesions; 12 state vs rising edges; 13 functional modules; 14 synergy (PID); 15 NMOS synthetic circuits; 16 instruction-aligned dynamics | Results III / Discussion |
| VI. Integrity and outputs | 17 run health and results at a glance; 18 results ledger, figure and table index, provenance; 19 evidence package | Appendix |

## Pre-specified hypotheses and analysis plan

Hypotheses were fixed before the final run. Every other analysis is labelled exploratory.

| | Hypothesis | Primary test |
|---|---|---|
| H1 (RQ1) | Pairwise TE (lag 1) ranks wired transistor pairs above unwired pairs | Physical-level ROC-AUC > 0.5 and AP lift > 1, with 95% target-cluster bootstrap CI (8.1) |
| H2 (RQ1) | Because switch-level logic settles within a step, contemporaneous dependence (lag 0) identifies wiring better than lag 1 | Paired bootstrap contrast lag 0 − lag 1 (8.1); simulated chip (12); NMOS circuits (15) |
| H3 (RQ1) | TE asymmetry identifies signal-flow direction beyond nuisance asymmetries | Orientation accuracy at matched histories; grouped cross-validated increment over rate and entropy (8.3) |
| H4 (RQ1) | In the rising-edge representation, pull-down edges are suppressive | Fraction of δ < 0 with Wilson CI, pull-down vs unwired (8.4); synthetic inverting gates (7) |
| H5 (RQ2) | IDTxl multivariate TE has higher precision than pairwise TE by removing indirect and shared-parent false positives | Paired target bootstrap of precision and MCC differences on identical targets (9.4); false-positive anatomy (9.5); synthetic circuits (7, 15) |
| H6 (RQ2) | Allowing lag-0 sources improves multivariate recovery | 9.7; 12; 15 |
| H7 | Wiring and causal influence differ; information measures track causal influence | Wiring vs intervention overlap; AUC against causal references; lesion criticality (11, 12) |
| H8 | Information networks respect functional modules | Within/between ratio vs permutation null; Louvain NMI/ARI (13) |

**Primary metrics.** Physical-level ROC-AUC and AP lift (the base rate is 0.035%, so AP lift and precision at fixed budgets are more informative than accuracy), with 95% target-cluster bootstrap intervals. Network decisions use bit-correct χ² p-values with BH FDR (pairwise), IDTxl's network-level FDR (multivariate) or link-level FDR (bivariate). Thresholds are never tuned against the netlist.

## Required outputs (what goes into the thesis)

| Chapter | Figures (`thesis_v5/figures`) | Tables (`thesis_v5/tables`) |
|---|---|---|
| Data/Methods | `fig05_coverage` | `jidt_units_audit`, `null_calibration_by_target` |
| Validation | `fig06_null_calibration`, `fig07_synthetic_convergence`, `fig15_nmos` | `synthetic_summary`, `synthetic_recall_by_edge_class`, `synthetic_effect_sign`, `synthetic_idtxl_units_effect`, `nmos_summary`, `synthetic_calibration` |
| RQ1 | `fig08_1_roc_pr`, `fig08_1_score_distributions`, `fig08_2_pvalue_distributions`, `fig08_3_direction`, `fig08_4_signed_effect`, `fig08_5_phase_delay`, `fig08_7_convergence`, `fig08_9_layout_maps` | `rq1_discrimination`, `rq1_paired_contrasts`, `rq1_te_by_relation`, `rq1_decision_rules`, `rq1_direction`, `rq1_direction_incremental`, `rq1_signed_effect`, `rq1_signed_scores`, `rq1_phase_delay_sweep`, `rq1_shared_driver`, `rq1_convergence`, `rq1_incremental_prediction` |
| RQ2 | `fig09_5_fp_anatomy`, `fig09_6_embedding_runtime` | `rq2_network_comparison`, `rq2_paired_differences`, `rq2_false_positive_anatomy`, `rq2_source_lag_sensitivity` |
| Ground truth | `fig12_representation`, `fig13_module_te`, `fig14_pid`, `fig16_instruction_aligned` | `sim_wiring_vs_intervention`, `sim_lesion_criticality`, `sim_representation_scores`, `sim_idtxl`, `modules`, `pid_summary`, `instruction_aligned_profile`, `instruction_profile_by_opcode` |

## Corrections in this version

* **Bit-correct analytical inference.** JIDT's discrete calculators return bits but their analytic χ² null treats the value as nats. All p-values and null means use $2N\ln 2\,\hat I_{\text{bits}}\sim\chi^2_{df}$ (Section 0.2), stored as log-p. IDTxl's analytic surrogates are corrected the same way (Section 5).
* **IDTxl fixes.** NumPy-2 compatibility; a vectorised, output-identical helper (about 13× faster); link-wise bivariate TE, because IDTxl's bivariate omnibus test cannot reject at this scale.
* **AIS history selection** uses the correct null, with base 1.0.

## How to run

1. Folder layout: `<project>/data/raw/donkeykong.5000.ts.spikes.npy`, `<project>/data/visual6502/`, `<project>/vendor/jidt/infodynamics.jar`, and this notebook in `<project>/notebooks/`. `segdefs.js` is downloaded automatically in Part V if it is absent.
2. Install IDTxl into the environment: `pip install "git+https://github.com/pwollstadt/IDTxl.git" statsmodels h5py networkx`.
3. Optional smoke test: start Jupyter with `TE6502_QUICK=1` (minutes). Section 17 marks such a run as **not** thesis results.
4. Thesis run: unset `TE6502_QUICK`, run all cells, and keep the machine awake (`caffeinate -i` on macOS). RQ2 analyses every active target (about 4–5 h of the total); completed targets are reused if the run is interrupted.

## 0. Configuration and environment

One configuration controls the entire notebook. All investigations are enabled; flags remain available for subsequent targeted reruns. Existing results are resumed only if their manifest matches. If settings differ, a deterministic separate directory is selected. This protects earlier runs without requiring manual path edits.

In [ ]:
import importlib.util
required_modules = ['numpy', 'scipy', 'pandas', 'matplotlib', 'jpype', 'sklearn', 'IPython']
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError("Select the Honours (ARM64, JIDT) kernel; missing modules: " + ", ".join(missing_modules))
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import json
import os
import platform
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jpype
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/raw/donkeykong.5000.ts.spikes.npy").exists())
DATA_PATH = ROOT / "data/raw/donkeykong.5000.ts.spikes.npy"
JAR_PATH = ROOT / "vendor/jidt/infodynamics.jar"  # Original, unmodified JIDT.

AIS_BIAS_BASE = 1.0  # Selection penalty at history k is (base ** k) * bit-correct analytical mean.
if not np.isfinite(AIS_BIAS_BASE) or AIS_BIAS_BASE < 1:
    raise ValueError("AIS_BIAS_BASE must be finite and at least 1.")
N_WORKERS = 6
K_VALUES = np.arange(1, 13)
SOURCE_HISTORY = 1
DELAY = 1
ROW_STRIDE = 3
ROW_PHASE = None  # None: largest event-count phase; or explicitly 0, 1, 2.
if ROW_PHASE not in (None, 0, 1, 2):
    raise ValueError("ROW_PHASE must be None, 0, 1 or 2.")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = dict(method_version=5, analytic_null="chi2_of_2N_ln2_I_bits", data_sha256=sha256_file(DATA_PATH),
                jar_sha256=sha256_file(JAR_PATH), k_values=K_VALUES.tolist(),
                ais="per_target_raw_minus_base_to_k_analytical_mean_argmax_first",
                ais_bias_base=AIS_BIAS_BASE, ais_penalty="base**k * analytical_mean",
                mi="JIDT_discrete_binary_lag0", source_history=SOURCE_HISTORY,
                delay=DELAY, embedding_spacings=1, crop="start_to_last_nonzero_inclusive",
                row_stride=ROW_STRIDE, row_phase=ROW_PHASE,
                phase_rule="max_total_events_first_tie" if ROW_PHASE is None else "explicit",
                grouping="exact_reduced_sequences_exclude_constants")
# All follow-on experiments are enabled. These limits bound the extra workload.
EXT_RUN={name:True for name in ['phase_delay','surrogates','history','windows','time_bootstrap','local_te','direction','incremental_prediction']}
EXT_CFG=dict(seed=2026,positive_pairs=64,negative_pairs=192,negative_audit_pairs=30000,
             max_original_rows=60000,phase_delays=[1,2,3,4],fixed_k=[1,2,4],
             bias_bases=[1.0,1.05,1.10,1.15],k_grid=list(range(1,13)),
             surrogate_pairs=8,surrogates=999,block_length=60,windows=4,
             bootstrap_draws=200,time_bootstrap_draws=50,local_pairs=8,
             alpha=.05,top_budget=1000,near_fractions=[.005,.01,.02,.05])

base_label = format(AIS_BIAS_BASE, '.8g').replace('.', '')
# Keep established names x105/x110 for the two common settings.
base_label = '105' if AIS_BIAS_BASE == 1.05 else ('110' if AIS_BIAS_BASE == 1.10 else base_label)
OUT = ROOT / f"results/v5_bitcorrect_every_third_mi_ais_te_k{K_VALUES.min()}_{K_VALUES.max()}_L{SOURCE_HISTORY}_d{DELAY}_ais_x{base_label}"
# This notebook's own path (some sections hash their source code for provenance).
NOTEBOOK_NAME = "Honours_Every_Third_MI_AIS_TE_v5.ipynb"
NOTEBOOK_PATH = next((p for p in [Path.cwd() / NOTEBOOK_NAME, ROOT / "notebooks" / NOTEBOOK_NAME]
                      if p.exists()), None)
if NOTEBOOK_PATH is None:
    _found = sorted([*Path.cwd().glob("Honours_Every_Third*.ipynb"), *(ROOT / "notebooks").glob("Honours_Every_Third*.ipynb")],
                    key=lambda p: p.stat().st_mtime)
    NOTEBOOK_PATH = _found[-1] if _found else Path.cwd() / NOTEBOOK_NAME

# Surrogate audit, direction and incremental prediction: enabled, with bounded workloads.
EXT_CFG.update(surrogate_history_grid=[1,4], surrogate_max_samples=4000,
               surrogate_block_lengths=[20,60,180], surrogate_shift_guards=[20,180],
               surrogate_period=3, synthetic_rates=[0.01,0.1], synthetic_histories=[1,4],
               synthetic_couplings=[0.,0.1,0.3], synthetic_trials=30,
               synthetic_samples=2000, synthetic_surrogates=199,
               direction_pairs=64, direction_matched_histories=[1,4],
               direction_delays=[1,2,3,4], prediction_folds=5,
               prediction_C=1.0)


In [ ]:
manifest_path = OUT / "manifest.json"
if manifest_path.exists() and json.loads(manifest_path.read_text()) != manifest:
    saved_manifest = json.loads(manifest_path.read_text())
    differences = [f"{key}: saved={saved_manifest.get(key)!r}, configured={manifest.get(key)!r}"
                   for key in sorted(set(saved_manifest) | set(manifest))
                   if saved_manifest.get(key) != manifest.get(key)]
    run_key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:12]
    OUT = OUT.with_name(OUT.name + "_" + run_key)
    manifest_path = OUT / "manifest.json"
    print("Using a separate run directory because settings changed:\n" + "\n".join(differences))
OUT.mkdir(parents=True, exist_ok=True)
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != manifest:
        raise ValueError(f"Manifest collision or corrupted run directory: {OUT}")
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))

def save_npz(path, **arrays):
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    temporary.replace(path)

print("Results:", OUT)
print("AIS bias base:", AIS_BIAS_BASE, "Workers:", N_WORKERS)
print("Enabled investigations:", EXT_RUN)


### 0.1 Verify the JVM before loading the recording

The existing resolver checks for a real JVM library compatible with the kernel architecture. Java observation arrays are prepared after reduction.

In [ ]:
# Resolve an existing JVM library explicitly: JAVA_HOME can point to a removed SDKMAN JDK.
import subprocess
import sys

def resolve_jvm():
    candidates = []
    homes = []
    if os.environ.get("JAVA_HOME"):
        homes.append(Path(os.environ["JAVA_HOME"]).expanduser())
    if sys.platform == "darwin":
        discovery = subprocess.run(["/usr/libexec/java_home"], capture_output=True, text=True)
        if discovery.returncode == 0 and discovery.stdout.strip():
            homes.append(Path(discovery.stdout.strip()))
        for base in [Path.home()/"Library/Java/JavaVirtualMachines",
                     Path("/Library/Java/JavaVirtualMachines")]:
            homes.extend(sorted(base.glob("*/Contents/Home")))
    for home in homes:
        for relative in ["lib/server/libjvm.dylib", "lib/server/libjvm.so",
                         "jre/lib/amd64/server/libjvm.so", "bin/server/jvm.dll"]:
            candidates.append(home / relative)
    try:
        candidates.append(Path(jpype.getDefaultJVMPath()))
    except (OSError, RuntimeError):
        pass
    rejected = []
    for library in dict.fromkeys(candidates):
        if not library.is_file():
            continue
        if sys.platform == "darwin":
            # The notebook kernel and Java must have the same architecture.
            description = subprocess.run(["/usr/bin/file", "-L", str(library)],
                                         capture_output=True, text=True, check=True).stdout
            architecture = platform.machine()
            if architecture not in description:
                rejected.append(f"{library} (kernel is {architecture})")
                continue
        return library
    raise FileNotFoundError(
        "No existing JVM library matching this Python kernel was found. "
        f"Kernel: {sys.executable} ({platform.machine()}). "
        "Set JAVA_HOME to a compatible JDK and rerun this cell. "
        "Architecture mismatches: " + "; ".join(rejected))

if not JAR_PATH.is_file():
    raise FileNotFoundError(f"JIDT jar not found: {JAR_PATH}")
if not jpype.isJVMStarted():
    jvm_library = resolve_jvm()
    print(f"Python kernel: {sys.executable} ({platform.machine()})")
    print(f"Using JVM: {jvm_library}")
    jpype.startJVM(str(jvm_library), "-ea", "-Xmx4g",
                  classpath=[str(JAR_PATH)], convertStrings=True)

AIS = jpype.JClass("infodynamics.measures.discrete.ActiveInformationCalculatorDiscrete")
MI = jpype.JClass("infodynamics.measures.discrete.MutualInformationCalculatorDiscrete")
TE = jpype.JClass("infodynamics.measures.discrete.TransferEntropyCalculatorDiscrete")

JInts = jpype.JArray(jpype.JInt)


### 0.2 Bit-correct analytical significance (v5)

For a discrete plug-in estimate $\hat I$ (MI, AIS, TE or conditional TE) under the null of conditional independence, $2N\ln 2\,\hat I_{\text{bits}}\sim\chi^2_{df}$ asymptotically, so the null mean in bits is $df/(2N\ln 2)$. Degrees of freedom follow JIDT: TE with target history $k$, one binary source and $m$ binary conditionals has $df=2^{k+m}$; AIS with history $k$ has $df=2^k-1$. The audit below checks the loaded JIDT jar against these formulas; every later cell uses the helpers defined here rather than JIDT's `computeSignificance()`.

In [ ]:
import math
from scipy.stats import chi2 as _chi2

def df_te(k, l=1, m=0):
    """Discrete TE degrees of freedom: binary target history k, source history l, m binary conditionals."""
    return int((2 ** int(l) - 1) * 2 ** int(k) * 2 ** int(m))

def df_ais(k):
    return int(2 ** int(k) - 1)

def analytic_null_mean_bits(df, n):
    return float(df) / (2.0 * float(n) * math.log(2.0))

def bitcorrect_logp(value_bits, n, df):
    """Natural-log p-value of a bit-valued plug-in estimate; never underflows."""
    return float(_chi2.logsf(max(2.0 * float(n) * math.log(2.0) * float(value_bits), 0.0), int(df)))

def bitcorrect_p(value_bits, n, df):
    return math.exp(bitcorrect_logp(value_bits, n, df))

# Audit the loaded JIDT jar on a weakly coupled synthetic pair.
_rng = np.random.default_rng(11); _n = 20000
_x = _rng.integers(0, 2, _n); _y = np.r_[0, np.where(_rng.random(_n - 1) < .02, _x[:-1], _rng.integers(0, 2, _n - 1))]
_audit = []
for _k in (1, 3, 6):
    _c = TE(2, _k, 1, 1, 1, 1); _c.initialise(); _c.addObservations(JInts(_x.astype(np.int32)), JInts(_y.astype(np.int32)))
    _v = float(_c.computeAverageLocalOfObservations()); _m = _c.computeSignificance(); _N = int(_c.getNumObservations())
    _audit.append(dict(k=_k, te_bits=_v, jidt_p=float(_m.pValue), bitcorrect_p=bitcorrect_p(_v, _N, df_te(_k)),
                       jidt_null_mean=float(_m.getMeanOfDistribution()), null_mean_bits=analytic_null_mean_bits(df_te(_k), _N)))
_ratio = np.median([a['jidt_null_mean'] / a['null_mean_bits'] for a in _audit])
for a in _audit: print(a)
print(f"JIDT null mean / bit-correct null mean = {_ratio:.6f} (ln 2 = {math.log(2):.6f}); "
      + ("JIDT treats bit-valued estimates as nats: using the bit-correct helpers." if abs(_ratio - math.log(2)) < 1e-6
         else "JIDT appears bit-consistent; helpers agree with it."))
(OUT / "jidt_units_audit.json").write_text(json.dumps(dict(rows=_audit, ratio=_ratio), indent=2))


## 1. EDA

### 1.1 Inspect the full recording and choose the retained interval

The input contains binary **off-to-on event indicators**, not continuous transistor voltage or on/off state. A zero means no recorded event at that sample. We use sample index because no sampling interval is supplied here.

Scan every row and verify binary values. Plot activity over the entire recording and identify the final nonzero row. A trailing all-zero segment is evidence for defining an active-period analysis, **not proof of why the recording went quiet**. Keep every row from the beginning through the final event, including internal quiet periods.

In [ ]:
# Scan in blocks to avoid copying the entire recording into memory.
# STOP is exclusive: retain the final event and all earlier quiet samples.
raw = np.load(DATA_PATH, mmap_mode="r")

if not (raw.ndim == 2):
    raise ValueError('Validation failed: raw.ndim == 2')
row_events = np.zeros(raw.shape[0], dtype=np.int64)
column_events = np.zeros(raw.shape[1], dtype=np.int64)
for start in range(0, len(raw), 10000):
    block = raw[start:start + 10000]
    if not (np.all((block == 0) | (block == 1))):
        raise ValueError("Expected binary event indicators.")
    row_events[start:start + len(block)] = block.sum(axis=1, dtype=np.int64)
    column_events += block.sum(axis=0, dtype=np.int64)
nonzero_rows = np.flatnonzero(row_events)
if not (nonzero_rows.size > 0):
    raise ValueError('Validation failed: nonzero_rows.size > 0')

STOP = int(nonzero_rows[-1]) + 1
N = STOP
print(f"Original: {raw.shape[0]:,} rows × {raw.shape[1]:,} physical columns; dtype={raw.dtype}")
print(f"Retained interval: [0, {STOP}) = {STOP:,} rows")
print(f"Trailing all-zero rows: {len(raw)-STOP:,} ({(len(raw)-STOP)/len(raw):.2%})")
print(f"First event row: {nonzero_rows[0]:,}; last event row: {STOP-1:,}")

bin_size = 1000
bin_starts = np.arange(0, len(raw), bin_size)
bin_lengths = np.minimum(bin_size, len(raw) - bin_starts)
mean_events = np.add.reduceat(row_events, bin_starts) / bin_lengths
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)

axes[0].plot(bin_starts, mean_events)
axes[0].axvline(STOP, color="red", linestyle="--", label="End of retained interval")
axes[0].set(xlabel="Sample index", ylabel="Mean events/sample", title="Full recording (1,000-sample bins)")
axes[0].legend()
axes[1].plot(bin_starts, mean_events)
axes[1].set(xlim=(max(0, STOP-20000), min(len(raw), STOP+20000)),
            xlabel="Sample index", ylabel="Mean events/sample", title="Activity around the cutoff")
axes[1].axvline(STOP, color="red", linestyle="--")
plt.show()
save_npz(OUT / "recording_audit.npz", original_shape=raw.shape, stop=STOP,
         row_events=row_events, column_events=column_events)

### 1.2 Test the proposed three-row / instruction-cycle pattern

Before reduction, compare all three row-index residues (`row % 3`), activity through time, short-lag dependence of total row activity, and inter-event gaps within physical columns. A dominant phase, stable phase concentration across windows, and gaps at multiples of three support a three-row representation. The notebook reports these quantities from the data when run; it does not insert assumed results.

A recording pattern alone cannot identify a CPU instruction boundary. Here “cycle” means the **observed three-row pattern**. If events occupy other phases, taking every third row loses those events; the audit below quantifies that cost. A higher event density is not evidence of improved network recovery by itself.


In [ ]:
import csv
CYCLE_DIR = OUT / "cycle_eda"
CYCLE_DIR.mkdir(exist_ok=True)
phase_counts = np.array([row_events[p:STOP:ROW_STRIDE].sum() for p in range(ROW_STRIDE)])
phase_lengths = np.array([len(range(p, STOP, ROW_STRIDE)) for p in range(ROW_STRIDE)])
phase_rows = []
for p in range(ROW_STRIDE):
    counts = row_events[p:STOP:ROW_STRIDE]
    phase_rows.append(dict(phase=p, rows=len(counts), events=int(phase_counts[p]),
        event_share=float(phase_counts[p]/phase_counts.sum()),
        mean_events_per_row=float(counts.mean()) if len(counts) else 0.,
        active_row_fraction=float(np.mean(counts>0)) if len(counts) else 0.))
with (OUT / "cycle_phase_summary.csv").open("w", newline="") as stream:
    writer=csv.DictWriter(stream, fieldnames=list(phase_rows[0]))
    writer.writeheader(); writer.writerows(phase_rows)
for row in phase_rows: print(row)

# Windows contain multiples of 3 rows and preserve the global phase origin.
cycle_window = 30000
cycle_starts = np.arange(0, STOP, cycle_window)
window_phase_events = np.array([[row_events[s+p:min(s+cycle_window,STOP):3].sum()
                                for p in range(3)] for s in cycle_starts])
window_totals = window_phase_events.sum(axis=1, keepdims=True)
window_phase_share = np.divide(window_phase_events, window_totals,
    out=np.full(window_phase_events.shape,np.nan), where=window_totals>0)
cycle_lags = np.arange(1, min(61, STOP))
x = row_events[:STOP].astype(float); x -= x.mean()
variance = np.mean(x*x)
cycle_acf = np.array([np.mean(x[:-lag]*x[lag:])/variance if variance else np.nan
                     for lag in cycle_lags])
# Pool within-column gaps only; never join separate physical series.
gap_modulo = np.zeros(3, dtype=np.int64)
short_gap_counts = np.zeros(30, dtype=np.int64)
for column in range(raw.shape[1]):
    gaps = np.diff(np.flatnonzero(raw[:STOP,column]))
    gap_modulo += np.bincount(gaps % 3, minlength=3)
    short = gaps[(gaps>=1)&(gaps<=30)]
    short_gap_counts += np.bincount(short, minlength=31)[1:31]
print("Within-column gap counts modulo 3:", gap_modulo)
print("Fraction of gaps divisible by 3:",
      float(gap_modulo[0]/gap_modulo.sum()) if gap_modulo.sum() else np.nan)
fig, axes = plt.subplots(2, 3, figsize=(16,8), constrained_layout=True)
axes[0,0].bar(range(3), phase_counts/phase_counts.sum())
axes[0,0].set(xticks=range(3), xlabel="Original row modulo 3", ylabel="Share of events", title="Phase concentration")
for p in range(3): axes[0,1].plot(cycle_starts,window_phase_share[:,p],label=f"phase {p}")
axes[0,1].set(xlabel="Original row",ylabel="Window event share",title="Phase stability")
axes[0,1].legend()
axes[0,2].stem(cycle_lags,cycle_acf)
axes[0,2].set(xlabel="Lag in original rows",ylabel="Normalised autocovariance",title="Aggregate activity dependence")
axes[1,0].bar(np.arange(1,31),short_gap_counts)
axes[1,0].set(xlabel="Within-column gap (original rows)",ylabel="Count",title="Event spacings")
busy=int(np.argmax(row_events[:STOP])); lo=max(0,busy-45); hi=min(STOP,lo+90)
axes[1,1].scatter(np.arange(lo,hi),row_events[lo:hi],c=np.arange(lo,hi)%3,cmap="viridis")
axes[1,1].set(xlabel="Original row",ylabel="Events",title="Busy window: phase-coloured rows")
axes[1,2].bar(range(3),gap_modulo)
axes[1,2].set(xticks=range(3),xlabel="Within-column gap modulo 3",ylabel="Count",title="Three-row gap structure")
fig.savefig(CYCLE_DIR / "three_row_evidence.png",dpi=160,bbox_inches="tight")
plt.show()
save_npz(OUT / "cycle_diagnostics.npz",phase_events=phase_counts,phase_lengths=phase_lengths,
         window_starts=cycle_starts,window_phase_events=window_phase_events,
         window_phase_share=window_phase_share,lags=cycle_lags,aggregate_acf=cycle_acf,
         gap_modulo=gap_modulo,short_gap_counts=short_gap_counts)


### 1.3 Retain one phase every three rows and audit concentration

Retain `raw[selected_phase:STOP:3]`, with the selected phase chosen once for the entire recording. This targets the most event-rich phase by default, without changing the phase from window to window. If most events fall in that phase, it reduces repeated quiet samples and increases event density. It is a coarser, phase-specific representation, not a guaranteed lossless compression.

The full-rate phase diagnostics remain available for judging the decision. Report event-retention fraction alongside the row reduction and density ratio. Columns and groups are then rebuilt on this exact reduced input so MI, AIS and TE all use identical samples. Do not compare full-rate and reduced-rate k or delay as equal physical times.


In [ ]:
selected_phase = int(np.argmax(phase_counts)) if ROW_PHASE is None else int(ROW_PHASE)
original_rows = np.arange(selected_phase, STOP, ROW_STRIDE, dtype=np.int64)
reduced = raw[selected_phase:STOP:ROW_STRIDE]
N = len(reduced)
if N <= int(K_VALUES.max()) + 1:
    raise ValueError("Too few reduced observations for the requested AIS histories.")
reduced_column_events = reduced.sum(axis=0, dtype=np.int64)
reduced_row_events = row_events[original_rows]
retained_events = int(reduced_column_events.sum())
total_events = int(phase_counts.sum())
retention = retained_events/total_events
full_density = total_events/(STOP*raw.shape[1])
reduced_density = retained_events/(N*raw.shape[1])
print(f"Selected phase: {selected_phase}; original rows {selected_phase}, {selected_phase+3}, ...")
print(f"Rows: {STOP:,} → {N:,}; retained {N/STOP:.2%}")
print(f"Events retained: {retained_events:,}/{total_events:,} ({retention:.2%})")
print(f"Events discarded: {total_events-retained_events:,}")
print(f"Event density: {full_density:.6g} → {reduced_density:.6g}; ratio {reduced_density/full_density:.3f}")
if retention < 1:
    print("This phase-specific reduction discards events; use the audit when interpreting its suitability.")
print("Delay 1 = 3 original rows; AIS k=1..12 spans 3..36 original row intervals.")
save_npz(OUT / "reduction_audit.npz",stride=ROW_STRIDE,phase=selected_phase,
         original_rows=original_rows,original_active_rows=STOP,reduced_rows=N,
         original_events=total_events,retained_events=retained_events,
         discarded_events=total_events-retained_events,event_retention=retention,
         full_density=full_density,reduced_density=reduced_density,
         reduced_column_events=reduced_column_events)


### 1.4 Exact activity groups after reduction

Group physical columns with **exactly identical reduced sequences**, excluding columns constant in the reduced recording. Reduction can merge previously distinct signals or make them constant, so the original 801-group count is not assumed. Preserve physical IDs and use this updated mapping for all structural evaluation. Grouping reuses identical estimates; it does not average signals.


In [ ]:
# Exact sequence grouping saves duplicate calculations while preserving physical IDs.
# Group membership is about observed activity, not equivalence of physical wiring.
group_path = OUT / "groups.npz"
if not group_path.exists():
    packed = np.packbits(reduced, axis=0)
    lookup, representatives = {}, []
    physical_to_group = np.full(raw.shape[1], -1, dtype=np.int32)
    for column in range(raw.shape[1]):
        if reduced_column_events[column] in (0, N):
            continue
        key = packed[:, column].tobytes()  # Exact bytes, not approximate similarity.
        if key not in lookup:
            lookup[key] = len(representatives)
            representatives.append(column)
        physical_to_group[column] = lookup[key]
    representatives = np.asarray(representatives, dtype=np.int32)
    save_npz(group_path, representatives=representatives, physical_to_group=physical_to_group)
    del packed, lookup
with np.load(group_path) as saved:
    representatives = saved["representatives"]
    physical_to_group = saved["physical_to_group"]
G = len(representatives)
if G < 2:
    raise ValueError("Need at least two nonconstant reduced activity groups.")
# [group, time] gives contiguous series; all retained observations are included.
data = np.ascontiguousarray(reduced[:, representatives].T, dtype=np.uint8)
if not (data.shape == (G, N)):
    raise ValueError('Validation failed: data.shape == (G, N)')
group_sizes = np.bincount(physical_to_group[physical_to_group >= 0], minlength=G)
event_rates = data.mean(axis=1)
print(f"Analysis: {G} groups × {N:,} samples; {G*(G-1):,} directed non-self pairs")
print(f"Zero-lag MI needs only {G*(G-1)//2:,} unordered pairs because it is symmetric.")
print("Event-rate min / median / max:", np.min(event_rates), np.median(event_rates), np.max(event_rates))
print("Excluded constant physical columns:", np.sum(physical_to_group < 0))

# Representative examples span low, median and high event rates.
sorted_groups = np.argsort(event_rates)
examples = sorted_groups[[0, G//2, G-1]]
window_start = int(np.argmax(reduced_row_events))
window_start = max(0, min(window_start-250, N-1000))
window_stop = min(window_start+1000, N)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
axes[0, 0].imshow(data[:80, window_start:window_stop], aspect="auto", interpolation="nearest", cmap="binary",
                  extent=[window_start, window_stop, 80, 0])
axes[0, 0].set(xlabel="Reduced sample index", ylabel="Group index", title="Binary events: first 80 groups, busy window")

In [ ]:
axes[0, 1].hist(event_rates, bins=40)
axes[0, 1].set(xlabel="Events / retained sample", ylabel="Groups", title="Event-rate distribution")
lags = np.arange(1, 101)
for group in examples:
    x = data[group].astype(float)
    mean = x.mean()
    acf = [np.mean((x[:-lag]-mean)*(x[lag:]-mean))/(mean*(1-mean)) for lag in lags]
    axes[1, 0].plot(lags, acf, label=f"group {group} (t{representatives[group]})")
    gaps = np.diff(np.flatnonzero(data[group]))
    if len(gaps):
        axes[1, 1].step(np.sort(gaps), np.arange(1, len(gaps)+1)/len(gaps), where="post", label=f"group {group}")
axes[1, 0].set(xlabel="Lag (reduced samples)", ylabel="Normalised autocovariance", title="Temporal dependence: three rate-ranked examples")
axes[1, 0].legend(fontsize=8)
axes[1, 1].set(xlabel="Inter-event interval (reduced samples)", ylabel="Empirical cumulative fraction", xscale="log", title="Inter-event intervals")
axes[1, 1].legend(fontsize=8)
plt.show()

# Window-level activity makes changes over the retained interval visible.
starts = np.arange(0, N, 5000)
lengths = np.minimum(5000, N-starts)
window_rates = np.add.reduceat(data, starts, axis=1, dtype=np.int64)/lengths
plt.figure(figsize=(12, 4))
plt.imshow(window_rates, aspect="auto", origin="lower", extent=[0, N, 0, G])
plt.xlabel("Reduced sample index"); plt.ylabel("Group index"); plt.title("Event rate in 5,000-sample windows")
plt.colorbar(label="Events / sample"); plt.show()
save_npz(OUT / "temporal_features.npz", event_rates=event_rates, group_sizes=group_sizes,
         window_starts=starts, window_lengths=lengths, window_rates=window_rates, example_groups=examples)

### 1.5 Prepare shared JIDT input arrays and verify estimator direction

Each worker uses an independent calculator. Synthetic delayed-copy checks run before the recording estimates.

In [ ]:
java_series = [JInts(np.asarray(series, dtype=np.int32)) for series in data]
print("Prepared", len(java_series), "Java arrays once.")

# Small direction and estimator checks before the real calculations.
rng = np.random.default_rng(2026)
x = rng.integers(0, 2, 10000, dtype=np.int32)
y = np.r_[np.int32(0), x[:-1]]

check = TE(2, 1, 1, 1, 1, 1)
check.initialise(); check.addObservations(JInts(x), JInts(y))
if not (check.computeAverageLocalOfObservations() > 0.98):
    raise ValueError('Validation failed: check.computeAverageLocalOfObservations() > 0.98')
check.initialise(); check.addObservations(JInts(y), JInts(x))
if not (check.computeAverageLocalOfObservations() < 0.01):
    raise ValueError('Validation failed: check.computeAverageLocalOfObservations() < 0.01')
check_mi = MI(2, 2, 0)
check_mi.initialise(); check_mi.addObservations(JInts(x), JInts(x))
if not (check_mi.computeAverageLocalOfObservations() > 0.98):
    raise ValueError('Validation failed: check_mi.computeAverageLocalOfObservations() > 0.98')
print("Synthetic MI and delayed-copy TE checks passed.")

def run_targets(folder, function):
    folder.mkdir(exist_ok=True)
    pending = [target for target in range(G) if not (folder / f"{target:04d}.npz").exists()]
    started = time.perf_counter()
    if pending:
        with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
            futures = {pool.submit(function, target): target for target in pending}
            for count, future in enumerate(as_completed(futures), 1):
                target = futures[future]
                result = future.result()
                save_npz(folder / f"{target:04d}.npz", target=target, **result)
                if count == 1 or count % 20 == 0 or count == len(pending):
                    print(f"{folder.name}: {G-len(pending)+count}/{G} targets; session {time.perf_counter()-started:.1f}s", flush=True)
    else:
        print(folder.name, ": all targets already saved")
    return dict(new_targets=len(pending), session_seconds=time.perf_counter()-started, workers=N_WORKERS)

In [ ]:
group_id = min(2, G - 1)
members = np.flatnonzero(physical_to_group == group_id)
print("Representative:", f"t{representatives[group_id]}")
print("Members:", [f"t{x}" for x in members])


## 2. Estimate MI, AIS and pairwise TE

These are the full-recording estimates on the selected every-third-row representation. All later sections consume these outputs. Checkpoints are scoped to the manifest-validated run directory.

### 2.1 Zero-lag mutual information

Use `MutualInformationCalculatorDiscrete(2, 2, 0)`, as in the supplied network notebook. The binary data need no binning or continuous estimator. MI measures contemporaneous association, so it is symmetric and does not identify direction. Compute each unordered pair once, then mirror it; this produces the same estimates as Joe's two directed loops. Diagonal entries are `NaN` because self-pairs are outside this experiment. No threshold is applied.

Time alignment is preserved: MI pairs source[t] with target[t] for every retained timestamp. Zero-lag MI does not itself model temporal history; it measures simultaneous association.

In [ ]:
# MI has no direction at lag zero: calculate one triangle, then mirror its values.
# Each worker owns its calculator; only read-only input arrays are shared.
def mi_target(target):
    started = time.perf_counter()
    calc = MI(2, 2, 0)
    sources = np.arange(target, dtype=np.int32)
    values = np.empty(len(sources))

    for i, source in enumerate(sources):
        calc.initialise()
        calc.addObservations(java_series[int(source)], java_series[target])
        values[i] = float(calc.computeAverageLocalOfObservations())
   
    if not (np.all(np.isfinite(values))):
        raise ValueError('Validation failed: np.all(np.isfinite(values))')
    
    return dict(sources=sources, mi_bits=values, seconds=time.perf_counter()-started)

MI_DIR = OUT / "mi_targets"
mi_timing = run_targets(MI_DIR, mi_target)
mi_matrix = np.full((G, G), np.nan)
for target in range(G):
    with np.load(MI_DIR / f"{target:04d}.npz") as result:
        sources = result["sources"]
        mi_matrix[sources, target] = result["mi_bits"]
        mi_matrix[target, sources] = result["mi_bits"]
save_npz(OUT / "mi.npz", mi_bits=mi_matrix, representatives=representatives)
print(f"MI complete: {G*(G-1)//2:,} unordered pairs; maximum {np.nanmax(mi_matrix):.6f} bits")

In [ ]:
if (OUT / "mi.npz").exists():
    with np.load(OUT / "mi.npz") as saved:
        mi_matrix = saved["mi_bits"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    image = axes[0].imshow(mi_matrix, origin="lower", aspect="auto")
    axes[0].set(xlabel="Target group", ylabel="Source group", title="Zero-lag discrete MI")
    fig.colorbar(image, ax=axes[0], label="MI (bits)")
    axes[1].hist(mi_matrix[np.triu_indices(G, 1)], bins=60)
    axes[1].set(xlabel="MI (bits)", ylabel="Unordered pairs", yscale="log", title="MI distribution")
    plt.show()

### 2.2 AIS and target-history selection

For each target, estimate binary AIS over `K_VALUES` and obtain the JIDT analytical null mean. Preserve standard corrected AIS as `raw AIS − analytical mean`. Select the first maximum of:

`raw_AIS(k) − AIS_BIAS_BASE**k × analytical_mean(k)`

The configured base is **AIS_BIAS_BASE (default 1.0)**: with the bit-correct null mean, base 1.0 is the standard bias-corrected AIS. Bases above 1 are a history-dependent regularisation heuristic, not a calibrated correction. The sensitivity plot compares alternative bases without changing the selected run. Histories are expressed in retained samples; one retained interval spans three original rows by default.


In [ ]:
# AIS asks how informative a target history is about its own next sample.
# Preserve Joe's correction; select histories using the extra base**k penalty.
# Choose k separately for each target; do not pool targets or optimise against wiring labels.
def ais_target(target):
    started = time.perf_counter()
    raw_ais = np.empty(len(K_VALUES))
    bias = np.empty(len(K_VALUES))
    jidt_bias = np.empty(len(K_VALUES))
    
    for i, k in enumerate(K_VALUES):
        calc = AIS(2, int(k))
        calc.initialise()
        calc.addObservations(java_series[target])
        raw_ais[i] = float(calc.computeAverageLocalOfObservations())
        jidt_bias[i] = float(calc.computeSignificance().getMeanOfDistribution())  # audit only (nats-scaled)
        bias[i] = analytic_null_mean_bits(df_ais(int(k)), int(calc.getNumObservations()))
    corrected = raw_ais - bias  # Preserve the standard bias-corrected AIS.
    selection_score = raw_ais - np.power(AIS_BIAS_BASE, K_VALUES) * bias
    
    if not (np.all(np.isfinite(corrected))):
        raise ValueError('Validation failed: np.all(np.isfinite(corrected))')
    
    optimal_k = int(K_VALUES[np.argmax(selection_score)])
    
    return dict(raw_ais_bits=raw_ais, analytical_mean_bits=bias, jidt_analytical_mean_raw=jidt_bias,
                corrected_ais_bits=corrected, selection_score_bits=selection_score,
                bias_base=AIS_BIAS_BASE, bias_multipliers=np.power(AIS_BIAS_BASE, K_VALUES), optimal_k=optimal_k, seconds=time.perf_counter()-started)

AIS_DIR = OUT / "ais_targets"
ais_timing = run_targets(AIS_DIR, ais_target)
raw_ais = np.empty((G, len(K_VALUES)))
ais_bias = np.empty_like(raw_ais)
corrected_ais = np.empty_like(raw_ais)
selection_scores = np.empty_like(raw_ais)
optimal_k = np.empty(G, dtype=np.int32)

for target in range(G):
    with np.load(AIS_DIR / f"{target:04d}.npz") as result:
        raw_ais[target] = result["raw_ais_bits"]
        ais_bias[target] = result["analytical_mean_bits"]
        corrected_ais[target] = result["corrected_ais_bits"]
        selection_scores[target] = result["selection_score_bits"]
        if float(result["bias_base"]) != AIS_BIAS_BASE:
            raise ValueError("AIS penalty changed: use a fresh output directory.")
        optimal_k[target] = result["optimal_k"]
save_npz(OUT / "ais.npz", k_values=K_VALUES, raw_ais_bits=raw_ais, analytical_mean_bits=ais_bias,
         corrected_ais_bits=corrected_ais, selection_score_bits=selection_scores,
         bias_base=AIS_BIAS_BASE, bias_multipliers=np.power(AIS_BIAS_BASE, K_VALUES), optimal_k=optimal_k, representatives=representatives)
print("Selected k counts:", dict(zip(*np.unique(optimal_k, return_counts=True))))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].bar(K_VALUES, np.bincount(optimal_k, minlength=13)[1:13])
axes[0].set(xticks=K_VALUES, xlabel="Selected target k", ylabel="Groups", title="Target-specific AIS selection")
for group in examples:
    axes[1].plot(K_VALUES, selection_scores[group], marker="o", label=f"group {group}, k={optimal_k[group]}")
axes[1].set(xlabel="Target history k", ylabel="Penalised AIS selection score (bits)", xticks=K_VALUES)
axes[1].legend(); plt.show()
# Sensitivity uses the same stored AIS curves; it does not rerun AIS or TE.
# These are diagnostics, not an automatic search for a preferred k distribution.
import csv
penalty_rows = []
fig, ax = plt.subplots(figsize=(10,4), constrained_layout=True)
for base in sorted(set([1.,1.05,1.10,1.15,1.25,AIS_BIAS_BASE])):
    selected = K_VALUES[np.argmax(raw_ais-np.power(base,K_VALUES)*ais_bias, axis=1)]
    counts = np.array([np.sum(selected==k) for k in K_VALUES])
    ax.plot(K_VALUES,counts,marker="o",label=f"penalty {base:g}^k × bias")
    penalty_rows.append(dict(base_x=base,median_k=float(np.median(selected)),
        fraction_at_k_max=float(np.mean(selected==K_VALUES[-1])),
        fraction_changed_from_original=float(np.mean(selected != K_VALUES[np.argmax(corrected_ais,axis=1)]))))
ax.set(xlabel="Selected target history k",ylabel="Groups",xticks=K_VALUES,
       title="AIS history-selection sensitivity to x^k × analytical bias")
ax.legend(); fig.savefig(OUT / "ais_penalty_sensitivity.png",dpi=160); plt.show()
with (OUT / "ais_penalty_sensitivity.csv").open("w",newline="") as stream:
    writer=csv.DictWriter(stream,fieldnames=list(penalty_rows[0]))
    writer.writeheader(); writer.writerows(penalty_rows)
for row in penalty_rows: print(row)
print("Standard corrected AIS and the penalised selection score are saved separately.")


### 2.3 Directed pairwise TE and analytical p-values

For target d, use `TransferEntropyCalculatorDiscrete(2, optimal_k[d], 1, 1, 1, 1)`. This is Joe's pairwise calculation with his target-specific AIS selection. Analytical p-values are bit-correct (Section 0.2) and also stored as natural-log p (`log_p`); JIDT's nats-treated p-values are stored as `jidt_p` for audit. Call `initialise()`, `addObservations(source, destination)`, and `computeAverageLocalOfObservations()` for every directed non-self pair.

Save the no-argument analytical p-value as the network notebook does. **No edge selection, multiplicity correction or significance conclusions are made here.** The analytical approximation can be unreliable for sparse/restricted states; k≤12 is not a validity guarantee. Matrices hold raw TE and provisional analytical p-values, with `NaN` on the untested diagonal. Each target uses N−k[d] embedded observations; all targets receive the same reduced recording.

With source history and delay both 1, TE uses source[t−1] to predict target[t], conditional on target[t−1], …, target[t−k]. The first k samples supply history, leaving N−k usable outcomes. Samples follow one fixed phase, three original rows apart; they are never shuffled or compressed to event-only times. Delay 1 therefore means three original row intervals. No surrogate permutations are run here.

In [ ]:
# Each source is tested against the target using that target's AIS-selected k.
# Store raw TE and the analytical p-value separately: neither is a binary edge decision.
# Self-pairs stay NaN so an untested pair cannot be mistaken for measured zero TE.
def te_target(target):
    started = time.perf_counter()
    k = int(optimal_k[target])
    calc = TE(2, k, 1, SOURCE_HISTORY, 1, DELAY)
    values = np.full(G, np.nan)
    p_values = np.full(G, np.nan)
    log_p = np.full(G, np.nan)
    jidt_p = np.full(G, np.nan)
   
    for source in range(G):
        if source == target:
            continue
        calc.initialise()
        calc.addObservations(java_series[source], java_series[target])
        values[source] = float(calc.computeAverageLocalOfObservations())
        n_obs = int(calc.getNumObservations())
        log_p[source] = bitcorrect_logp(values[source], n_obs, df_te(k))
        p_values[source] = math.exp(log_p[source])
        jidt_p[source] = float(calc.computeSignificance().pValue)  # audit only (nats-treated null)
        if not (int(calc.getNumObservations()) == N-k):
            raise ValueError('Validation failed: int(calc.getNumObservations()) == N-k')
   
    tested = np.arange(G) != target
    if not (np.all(np.isfinite(values[tested]))):
        raise ValueError('Validation failed: np.all(np.isfinite(values[tested]))')
    if not (np.all(np.isfinite(p_values[tested]) & (p_values[tested] >= 0) & (p_values[tested] <= 1))):
        raise ValueError('Validation failed: np.all(np.isfinite(p_values[tested]) & (p_values[tested] >= 0) & (p_values[tested] <= 1))')
    return dict(te_bits=values, analytical_p=p_values, log_p=log_p, jidt_p=jidt_p, k=k, observations=N-k,
                seconds=time.perf_counter()-started)

TE_DIR = OUT / "te_targets"
te_timing = run_targets(TE_DIR, te_target)
te_matrix = np.full((G, G), np.nan)
analytical_p = np.full((G, G), np.nan)
te_log_p = np.full((G, G), np.nan)
te_jidt_p = np.full((G, G), np.nan)
for target in range(G):
    with np.load(TE_DIR / f"{target:04d}.npz") as result:
        if not (int(result["k"]) == optimal_k[target]):
            raise ValueError('Validation failed: int(result["k"]) == optimal_k[target]')
        te_matrix[:, target] = result["te_bits"]
        analytical_p[:, target] = result["analytical_p"]
        te_log_p[:, target] = result["log_p"]
        te_jidt_p[:, target] = result["jidt_p"]
save_npz(OUT / "te.npz", te_bits=te_matrix, analytical_p=analytical_p, log_p=te_log_p, jidt_p=te_jidt_p,
         optimal_k=optimal_k, observations=N-optimal_k, representatives=representatives)
print(f"TE complete: {G*(G-1):,} directed pairs; maximum {np.nanmax(te_matrix):.6f} bits")

In [ ]:
if (OUT / "te.npz").exists():
    with np.load(OUT / "te.npz") as saved:
        te_matrix = saved["te_bits"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    image = axes[0].imshow(te_matrix, origin="lower", aspect="auto")
    axes[0].set(xlabel="Target group", ylabel="Source group", title="Raw pairwise TE; target-specific k")
    fig.colorbar(image, ax=axes[0], label="TE (bits)")
    axes[1].hist(te_matrix[np.isfinite(te_matrix)], bins=60)
    axes[1].set(xlabel="TE (bits)", ylabel="Directed pairs", yscale="log", title="Raw TE distribution")
    plt.show()

## 3. Structural reference (terminal-to-gate wiring)

Labels are built from the netlist independently of any score. A directed physical pair (source transistor → target transistor) is **wired** when a channel terminal of the source is the gate node of the target, excluding the supply nodes (VCC 657, VSS 558) and self-pairs. Each activity-group pair carries the exact numbers of wired and unwired physical pairs it represents, so evaluation can be done at the physical level without expanding millions of rows. Within-group connections cannot be tested (identical series) and are reported as an observability limit.

In [ ]:
# Load saved estimates and build structural labels independently of their scores.
# Count physical positives/negatives within each group pair to avoid expanding millions of rows.
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import csv, html, json, re
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display, HTML, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/visual6502/transdefs.js").exists())
ANALYSIS = OUT / "analysis_pairwise"
if not OUT.exists():
    raise FileNotFoundError(f"Run the estimation notebook first: {OUT}")
ANALYSIS.mkdir(exist_ok=True)
N_WORKERS = 6

def table(rows, filename=None, limit=30):
    if not rows:
        print("No eligible observations.")
        return
    keys = list(rows[0])
    if filename:
        with (ANALYSIS / filename).open("w", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=keys)
            writer.writeheader(); writer.writerows(rows)
    def format_value(value):
        return f"{value:.6g}" if isinstance(value, (float, np.floating)) else str(value)
    heading = "".join(f"<th>{html.escape(k)}</th>" for k in keys)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(format_value(row[k]))}</td>" for k in keys) + "</tr>" for row in rows[:limit])
    display(HTML("<table><thead><tr>"+heading+"</tr></thead><tbody>"+body+"</tbody></table>"))
    if len(rows)>limit: print(f"Showing {limit}/{len(rows)} rows; full table saved when a filename is given.")

def figure(name):
    plt.gcf().savefig(ANALYSIS / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.gcf().savefig(ANALYSIS / f"{name}.pdf", bbox_inches="tight")
    plt.show()

with np.load(OUT / "groups.npz") as z:
    representatives=z["representatives"]; physical_to_group=z["physical_to_group"]
with np.load(OUT / "mi.npz") as z: MI_scores=z["mi_bits"]
with np.load(OUT / "te.npz") as z:
    TE_scores=z["te_bits"]; P_values=z["analytical_p"]
    chosen_k=z["optimal_k"]; observations=z["observations"]
    if not (np.array_equal(z["representatives"], representatives)):
        raise ValueError('Validation failed: np.array_equal(z["representatives"], representatives)')

In [ ]:
with np.load(OUT / "ais.npz") as z:
    AIS_corrected=z["corrected_ais_bits"]; AIS_raw=z["raw_ais_bits"]; AIS_bias=z["analytical_mean_bits"]
with np.load(OUT / "temporal_features.npz") as z: rates=z["event_rates"]
with np.load(OUT / "reduction_audit.npz") as z:
    retained_rows=int(z["reduced_rows"])
    original_row_stride=int(z["stride"]); selected_phase=int(z["phase"])
if not np.array_equal(observations, retained_rows-chosen_k):
    raise ValueError("TE observation counts disagree with the reduced recording.")
G=len(representatives); physical_count=len(physical_to_group)
if not (MI_scores.shape == TE_scores.shape == P_values.shape == (G,G)):
    raise ValueError('Validation failed: MI_scores.shape == TE_scores.shape == P_values.shape == (G,G)')
if not (np.allclose(MI_scores,MI_scores.T,equal_nan=True)):
    raise ValueError('Validation failed: np.allclose(MI_scores,MI_scores.T,equal_nan=True)')
eligible=~np.eye(G,dtype=bool)
if not (np.isfinite(TE_scores[eligible]).all() and np.isfinite(MI_scores[eligible]).all()):
    raise ValueError('Validation failed: np.isfinite(TE_scores[eligible]).all() and np.isfinite(MI_scores[eligible]).all()')
if not (np.isfinite(P_values[eligible]).all() and np.all((P_values[eligible]>=0)&(P_values[eligible]<=1))):
    raise ValueError('Validation failed: np.isfinite(P_values[eligible]).all() and np.all((P_values[eligible]>=0)&(P_values[eligible]<=1))')
source_index,target_index=np.nonzero(eligible)
group_sizes=np.bincount(physical_to_group[physical_to_group>=0],minlength=G)
pair_count=group_sizes[:,None]*group_sizes[None,:]
np.fill_diagonal(pair_count,0)

matches=re.findall(r"\['t(\d+)',\s*(\d+),\s*(\d+),\s*(\d+)", (ROOT / "data/visual6502/transdefs.js").read_text())
transistors={int(i):(int(g),int(a),int(b)) for i,g,a,b in matches}
if not (sorted(transistors)==list(range(physical_count))):
    raise ValueError('Validation failed: sorted(transistors)==list(range(physical_count))')
gate,c1,c2=np.asarray([transistors[i] for i in range(physical_count)]).T
gate_targets={}
for target, gate_id in enumerate(gate): gate_targets.setdefault(int(gate_id),[]).append(target)
positive_count=np.zeros((G,G),dtype=np.int64)
coverage=dict(all_structural_connections=0,inactive_endpoint_connections=0,within_group_connections=0,evaluated_connections=0)
for source in range(physical_count):
    destinations=set()
    for terminal in (c1[source],c2[source]):
        if terminal not in (558,657): destinations.update(gate_targets.get(int(terminal),[]))
    for target in destinations:
        if source==target: continue
        coverage["all_structural_connections"]+=1
        sg,tg=physical_to_group[source],physical_to_group[target]
        if sg<0 or tg<0: coverage["inactive_endpoint_connections"]+=1
        elif sg==tg: coverage["within_group_connections"]+=1
        else:
            positive_count[sg,tg]+=1
            coverage["evaluated_connections"]+=1
negative_count=pair_count-positive_count
if not (np.all(negative_count>=0)):
    raise ValueError('Validation failed: np.all(negative_count>=0)')
if not (sum(coverage[k] for k in list(coverage)[1:])==coverage["all_structural_connections"]):
    raise ValueError('Validation failed: sum(coverage[k] for k in list(coverage)[1:])==coverage["all_structural_connections"]')

In [ ]:
table([{"Coverage":k,"Count":v} for k,v in coverage.items()],"coverage.csv")
mixed=(positive_count>0)&(negative_count>0)
print(f"Evaluated: {eligible.sum():,} group pairs, {pair_count.sum():,} physical pairs.")
print(f"Mixed-label group pairs: {mixed.sum():,}; positives excluded within groups: {coverage['within_group_connections']:,}")
print("The evaluated positive count excludes within-group connections, whose TE was not tested.")
np.savez_compressed(ANALYSIS / "structural_counts.npz",positive_count=positive_count,negative_count=negative_count,
                    pair_count=pair_count,representatives=representatives)

## 4. Recording provenance and physical-column mapping

These checks document what each recorded bit and row mean before interpreting TE
as a circuit relationship. A filename, binary encoding, or matching column count
does not establish those meanings. Fill in the evidence fields below from the
recording generator or its documentation. Unfilled fields remain **unverified**;
they do not silently become assumptions supported by the data.

The current analysis assumes column `i` is transistor `ti`. An optional CSV with
columns `column,transistor_id` checks that assumption against an independently
obtained mapping. If it disagrees, the existing structural evaluation must be
rebuilt; this appendix does not relabel already-computed results.

The AIS range and previously selected histories are unchanged. Clock and workload
checks below are bounded diagnostics, not another embedding search or full network
inference. All analysis code is included inline in this notebook.

In [ ]:
CHECK_CFG = dict(
    seed=2026,
    max_retained_samples=20000,  # Prefix length for additional diagnostics only.
    clock_pairs=32,
    mte_edges=32,
    phase_period_original_rows=3,
)
RECORDING_EVIDENCE = dict(
    bit_meaning='',             # E.g. transition indicator; quote the generator definition.
    row_meaning='',             # E.g. simulator half-cycle; include sampling details.
    evidence_files=[],         # Paths to generator source / documentation.
    reviewed=False,            # Set True only after checking those files yourself.
    mapping_csv=None,          # Independently established column,transistor_id CSV.
)
# Optional measured clock, one categorical integer per ORIGINAL recording row.
# This must be aligned to DATA_PATH, not to the already reduced array.
CLOCK_NPY = None
CLOCK_DESCRIPTION = ''
# Additional recordings of the SAME physical circuit. Each entry needs a mapping
# CSV and an explicit sampling statement so columns can be aligned by identity.
# Example: dict(name='other_game', path='/absolute/other.npy',
#               mapping_csv='/absolute/other_mapping.csv',
#               same_row_semantics=True, description='Different workload, same simulator')
OTHER_RECORDINGS = []

import platform, importlib.metadata, zipfile
from datetime import datetime, timezone

CHECK_DIR = OUT / 'remaining_checks'
CHECK_DIR.mkdir(exist_ok=True)

def checks_json(path, value):
    def safe(x):
        if isinstance(x, dict): return {str(k): safe(v) for k,v in x.items()}
        if isinstance(x, (list, tuple)): return [safe(v) for v in x]
        if isinstance(x, np.ndarray): return safe(x.tolist())
        if isinstance(x, np.generic): return safe(x.item())
        if isinstance(x, float) and not np.isfinite(x): return None
        if isinstance(x, Path): return str(x)
        return x
    Path(path).write_text(json.dumps(safe(value), indent=2, allow_nan=False))

def checks_mapping(path, width):
    frame = pd.read_csv(path, dtype=str)
    if not {'column','transistor_id'} <= set(frame):
        raise ValueError('Mapping requires column and transistor_id columns.')
    cols = pd.to_numeric(frame.column, errors='raise').to_numpy(float)
    ids = pd.to_numeric(frame.transistor_id.str.replace(r'^t', '', regex=True), errors='raise').to_numpy(float)
    if not np.isfinite(cols).all() or not np.isfinite(ids).all() or (cols != cols.astype(int)).any() or (ids != ids.astype(int)).any():
        raise ValueError('Mapping indices must be finite integers.')
    cols, ids = cols.astype(int), ids.astype(int)
    if sorted(cols.tolist()) != list(range(width)) or len(set(ids)) != width or (ids < 0).any():
        raise ValueError('Mapping must uniquely cover every column and transistor ID.')
    return ids[np.argsort(cols)]

checks_evidence = []
for name in RECORDING_EVIDENCE['evidence_files']:
    path = Path(name).expanduser().resolve()
    checks_evidence.append(dict(path=str(path), sha256=sha256_file(path)))
checks_map_verified = False
checks_mapping_status = 'unverified: no independent mapping supplied'
if RECORDING_EVIDENCE['mapping_csv']:
    checks_ids = checks_mapping(RECORDING_EVIDENCE['mapping_csv'], raw.shape[1])
    checks_map_verified = bool(np.array_equal(checks_ids, np.arange(raw.shape[1])))
    checks_mapping_status = ('supplied mapping agrees with column i = ti' if checks_map_verified
                             else 'MISMATCH: rebuild physical labels before interpreting precision')
checks_semantics_reviewed = bool(RECORDING_EVIDENCE['reviewed'] and checks_evidence
    and RECORDING_EVIDENCE['bit_meaning'] and RECORDING_EVIDENCE['row_meaning'])
checks_provenance = dict(config=CHECK_CFG, evidence=RECORDING_EVIDENCE,
    evidence_file_hashes=checks_evidence, semantics_reviewed=checks_semantics_reviewed,
    mapping_status=checks_mapping_status, mapping_agrees=checks_map_verified,
    data_sha256=sha256_file(DATA_PATH), original_shape=list(np.load(DATA_PATH,mmap_mode='r').shape),
    original_rows_used=int(STOP), stride=int(ROW_STRIDE), phase=int(selected_phase),
    retained_rows=int(len(original_rows)), discarded_tail_rows=int(np.load(DATA_PATH,mmap_mode='r').shape[0]-STOP),
    k_values=K_VALUES.tolist(), source_history=int(SOURCE_HISTORY), delay=int(DELAY))
if RECORDING_EVIDENCE['mapping_csv']:
    checks_provenance['mapping_sha256'] = sha256_file(Path(RECORDING_EVIDENCE['mapping_csv']))
checks_json(CHECK_DIR/'provenance.json', checks_provenance)
print(checks_mapping_status)
print('Recording semantics reviewed:', checks_semantics_reviewed)
display(pd.DataFrame([checks_provenance], columns=['original_rows_used','stride','phase','retained_rows','discarded_tail_rows']))

# Part II — Validation

The settings below control Parts II–V. After a kernel restart you can rerun Section 0 and continue from here, because everything is reloaded from the run directory.

**Units.** All information is in bits. Analytical significance uses $2N\ln 2\,\hat I_{\text{bits}}\sim\chi^2_{df}$.

In [ ]:
# 5.0 Configuration for Parts II–V. Change scope here; everything downstream reads RQ.
import os
TE6502_QUICK = os.environ.get('TE6502_QUICK', '0') == '1'
RQ = dict(
    seed=2026, alpha=0.05,
    bootstrap_draws=100 if TE6502_QUICK else 1000,
    budgets_items=[100, 500, 1000, 2000],          # group-pair prediction budgets
    # 6 empirical null calibration (circular target shifts)
    null_targets=4 if TE6502_QUICK else 40, null_shifts=10 if TE6502_QUICK else 100, null_min_shift=500,
    # 7 synthetic ground truth
    syn_n_nodes=24 if TE6502_QUICK else 40, syn_inputs=4 if TE6502_QUICK else 5,
    syn_noise=0.02, syn_k_max=3, syn_T=[1000, 3000] if TE6502_QUICK else [1000, 3000, 10000, 30000],
    syn_reps=2 if TE6502_QUICK else 20, syn_idtxl_reps=1 if TE6502_QUICK else 10,
    syn_idtxl_T_max=1000 if TE6502_QUICK else 10000, syn_idtxl_max_lag_target=3,
    # 36 RQ1
    direction_common_k=[1, 4] if TE6502_QUICK else [1, 2, 4, 8],
    phase_sweep=[0, 1, 2], delay_sweep=[0, 1] if TE6502_QUICK else [0, 1, 2, 3, 4],   # delay 0 = contemporaneous conditional MI
    run_stride1=not TE6502_QUICK, stride1_delays=[1, 2, 3, 6], stride1_k=3,
    hub_counts=[1, 2],
    # 37 RQ2 (IDTxl)
    idtxl_quick_targets=6,                          # smoke-test mode only; the thesis run analyses every active target
    idtxl_methods=['multivariate', 'bivariate'],
    idtxl_nofix_targets=2 if TE6502_QUICK else 10,  # sensitivity: uncorrected analytic null on a subset
    idtxl_max_lag_target=12, idtxl_max_lag_sources=1, idtxl_n_perm=200,
    idtxl_lag_targets=2 if TE6502_QUICK else 20, idtxl_lag_max=1 if TE6502_QUICK else 3,   # 9.7 source-lag sensitivity
    # Workers: at most CPU cores - 1, at most 6, and about 1.5 GB of RAM each (IDTxl on the full recording uses ~1.3 GB per worker).
    workers=max(1, min(6, (os.cpu_count() or 2) - 1,
                       int((os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')) / 2**30 // 1.5) - 1 if hasattr(os, 'sysconf') else 6)),
    idtxl_max_heap='2g',
)
print('Quick smoke-test mode' if TE6502_QUICK else 'Thesis configuration', '| workers:', RQ['workers'])

## 5. Analysis setup and toolkit

The next cell writes `te6502_toolkit.py` next to this notebook so that worker processes can import it. It contains: bit-correct analytical nulls; an exact plug-in TE estimator that scores every source of a target in one sparse product (validated against JIDT to machine precision in 8.1); tie-aware weighted ROC/PR and budget metrics; target-cluster bootstrap; BH/BY on log-p values; the synthetic logic-circuit generator; and IDTxl helpers (NumPy-2 compatibility, a vectorised, output-identical replacement for IDTxl's slowest helper, and an estimator whose analytic surrogates are in bits).

In [ ]:
%%writefile te6502_toolkit.py
"""te6502_toolkit: bit-correct analytical inference, fast binary TE, weighted
evaluation, synthetic ground-truth benchmarks and IDTxl helpers for the MOS 6502
honours thesis (RQ1 pairwise TE; RQ2 multivariate TE with IDTxl).

Information is in bits throughout. Chi-square statistics are formed in nats,
G = 2 N ln(2) I_bits ~ chi2(df) under independence (likelihood-ratio
asymptotics; Barnett & Bossomaier 2012, Phys. Rev. Lett. 109, 138105).
"""
from __future__ import annotations

import math
import numpy as np
from scipy import sparse
from scipy.stats import chi2

LN2 = math.log(2.0)
TOOLKIT_VERSION = '5.0'

# ---------------------------------------------------------------------------
# 1. Analytical null distributions in bits
# ---------------------------------------------------------------------------

def df_te(k, l=1, m=0, base=2):
    """Degrees of freedom of discrete (conditional) TE: target history k,
    source history l, m binary conditioning variables (JIDT convention)."""
    return int((base ** int(l) - 1) * (base - 1) * base ** int(k) * base ** int(m))


def df_ais(k, base=2):
    return int((base ** int(k) - 1) * (base - 1))


def null_mean_bits(df, n):
    """Expected plug-in estimate (bits) under independence: df / (2 N ln 2)."""
    return float(df) / (2.0 * float(n) * LN2)


def log_p_from_bits(value_bits, n, df):
    """Natural-log upper-tail p-value. Finite for finite input (no underflow)."""
    stat = np.maximum(2.0 * float(n) * LN2 * np.asarray(value_bits, dtype=float), 0.0)
    return chi2.logsf(stat, df)


def p_from_bits(value_bits, n, df):
    return np.exp(log_p_from_bits(value_bits, n, df))


def jidt_units_audit(jp, n=20000, seed=7):
    """Compare the loaded JIDT jar's analytic null with the bit-correct null."""
    J = jp.JArray(jp.JInt)
    TE = jp.JClass('infodynamics.measures.discrete.TransferEntropyCalculatorDiscrete')
    AIS = jp.JClass('infodynamics.measures.discrete.ActiveInformationCalculatorDiscrete')
    rng = np.random.default_rng(seed)
    x = rng.integers(0, 2, n)
    y = np.r_[0, np.where(rng.random(n - 1) < 0.02, x[:-1], rng.integers(0, 2, n - 1))]
    rows = []
    for k in (1, 2, 4, 6):
        c = TE(2, k, 1, 1, 1, 1); c.initialise(); c.addObservations(J(x.tolist()), J(y.tolist()))
        v = float(c.computeAverageLocalOfObservations()); m = c.computeSignificance()
        N = int(c.getNumObservations()); df = df_te(k)
        rows.append(dict(measure='TE', k=k, n=N, df=df, value_bits=v, jidt_p=float(m.pValue),
                         p_bit_correct=float(p_from_bits(v, N, df)),
                         p_if_bits_treated_as_nats=float(chi2.sf(2 * N * v, df)),
                         jidt_null_mean=float(m.getMeanOfDistribution()),
                         null_mean_bits=null_mean_bits(df, N), null_mean_nats=df / (2 * N)))
        a = AIS(2, k); a.initialise(); a.addObservations(J(y.tolist()))
        va = float(a.computeAverageLocalOfObservations()); Na = int(a.getNumObservations()); dfa = df_ais(k)
        rows.append(dict(measure='AIS', k=k, n=Na, df=dfa, value_bits=va, jidt_p=np.nan, p_bit_correct=np.nan,
                         p_if_bits_treated_as_nats=np.nan,
                         jidt_null_mean=float(a.computeSignificance().getMeanOfDistribution()),
                         null_mean_bits=null_mean_bits(dfa, Na), null_mean_nats=dfa / (2 * Na)))
    te_rows = [r for r in rows if r['measure'] == 'TE']
    ratio = float(np.median([r['jidt_null_mean'] / r['null_mean_bits'] for r in rows]))
    close = lambda a, b: abs(a - b) <= 1e-9 + 1e-6 * abs(b)
    nats = all(close(r['jidt_p'], r['p_if_bits_treated_as_nats']) for r in te_rows)
    bits = all(close(r['jidt_p'], r['p_bit_correct']) for r in te_rows)
    convention = ('values_in_bits_null_in_nats' if abs(ratio - LN2) < 1e-6 and nats
                  else 'bit_consistent' if abs(ratio - 1) < 1e-6 and bits else 'unrecognised')
    return dict(rows=rows, null_mean_ratio_jidt_over_bits=ratio, jidt_p_matches_nats_treatment=bool(nats),
                jidt_p_matches_bit_correct=bool(bits), convention=convention)


# ---------------------------------------------------------------------------
# 2. Exact plug-in TE for binary series: every source into one target at once
# ---------------------------------------------------------------------------

def _encode_history(y, k, start, stop):
    code = np.zeros(stop - start, dtype=np.int64)
    for lag in range(1, int(k) + 1):
        code = (code << 1) | y[start - lag:stop - lag].astype(np.int64)
    return code


def as_sparse_sources(X):
    return X if sparse.issparse(X) else sparse.csr_matrix(np.asarray(X, dtype=np.float64))


def te_all_sources(X, y, k, delay=1, cond=None, cond_delay=1, return_details=False):
    """Plug-in TE(X_g -> y | y^(k)[, cond]) in bits for every row g of X.

    Source sample x[t-delay] predicts y[t]; target history is y[t-k..t-1]. With
    delay=1 this equals JIDT TransferEntropyCalculatorDiscrete(2,k,1,1,1,1)
    (verified to 1e-15). cond: optional (C, T) binary series sampled at
    t-cond_delay (conditional TE). Details include the signed effect
    delta = sum_s w_s [P(y=1|s,x=1) - P(y=1|s,x=0)] (negative = inhibitory).
    """
    y = np.asarray(y).astype(np.int64)
    T = y.shape[0]
    c_lag = 0 if cond is None else int(cond_delay)
    start = max(int(k), int(delay), c_lag)
    stop = T; n = stop - start
    state = _encode_history(y, int(k), start, stop)
    n_cond_bits = 0
    if cond is not None:
        for row in np.atleast_2d(np.asarray(cond)).astype(np.int64):
            state = (state << 1) | row[start - c_lag:stop - c_lag]
            n_cond_bits += 1
    _, state = np.unique(state, return_inverse=True)
    state = state.reshape(-1)
    S = int(state.max()) + 1
    j = state * 2 + y[start:stop]
    M = sparse.csr_matrix((np.ones(n), (np.arange(start, stop) - int(delay), j)), shape=(T, 2 * S))
    Xsp = as_sparse_sources(X)
    C1 = np.asarray((Xsp @ M).todense()).reshape(Xsp.shape[0], S, 2)
    Ntot = np.bincount(j, minlength=2 * S).astype(np.float64).reshape(S, 2)
    C0 = Ntot[None] - C1
    ns = Ntot.sum(axis=1)
    out = np.zeros(C1.shape[0])
    for C in (C0, C1):
        n_sx = C.sum(axis=2, keepdims=True)
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = (C * ns[None, :, None]) / (n_sx * Ntot[None])
            out += np.where(C > 0, C * np.log2(ratio), 0.0).sum(axis=(1, 2))
    te = np.maximum(out / n, 0.0)
    if not return_details:
        return te, n
    n1 = C1.sum(axis=2); n0 = C0.sum(axis=2)
    with np.errstate(divide='ignore', invalid='ignore'):
        p1 = C1[:, :, 1] / n1; p0 = C0[:, :, 1] / n0
        both = (n1 > 0) & (n0 > 0)
        w = np.where(both, n1 + n0, 0.0); wsum = w.sum(axis=1)
        delta = np.where(wsum > 0, np.where(both, (p1 - p0) * w, 0.0).sum(axis=1) / np.where(wsum > 0, wsum, 1), np.nan)
        h = -np.where(Ntot > 0, Ntot * np.log2(Ntot / ns[:, None]), 0.0).sum() / n
    return te, n, dict(delta=delta, observed_states=S, cond_bits=n_cond_bits, df=df_te(k, 1, n_cond_bits),
                       target_conditional_entropy_bits=float(h))


def te_matrix(data, ks, delay=1, cond=None, details=False, progress=0):
    """Directed TE matrix [source, target] (bits), diagonal NaN.
    cond: None or callable t -> (C, T) array of conditioning series."""
    data = np.asarray(data)
    G = data.shape[0]; Xsp = as_sparse_sources(data)
    ks = np.broadcast_to(np.asarray(ks, dtype=int), (G,))
    te = np.full((G, G), np.nan); nobs = np.zeros(G, dtype=np.int64)
    delta = np.full((G, G), np.nan) if details else None
    hcond = np.full(G, np.nan) if details else None
    for t in range(G):
        c = None if cond is None else cond(t)
        res = te_all_sources(Xsp, data[t], int(ks[t]), delay=delay, cond=c, return_details=details)
        te[:, t] = res[0]; nobs[t] = res[1]
        if details:
            delta[:, t] = res[2]['delta']; hcond[t] = res[2]['target_conditional_entropy_bits']
            delta[t, t] = np.nan
        te[t, t] = np.nan
        if progress and ((t + 1) % progress == 0 or t == G - 1):
            print(f'    TE targets {t + 1}/{G}', flush=True)
    if details:
        return te, nobs, delta, hcond
    return te, nobs


def conditional_entropy_bits(y, k):
    y = np.asarray(y).astype(np.int64); start = int(k)
    j = _encode_history(y, k, start, len(y)) * 2 + y[start:]
    cnt = np.bincount(j, minlength=2 ** (int(k) + 1)).astype(float).reshape(-1, 2)
    n = cnt.sum(); ns = cnt.sum(1)
    with np.errstate(divide='ignore', invalid='ignore'):
        return float(-np.where(cnt > 0, cnt * np.log2(cnt / ns[:, None]), 0).sum() / n)


def ais_select_k(series, k_max=6, base=1.0):
    """Bias-corrected AIS history selection in bits: argmax raw AIS - base^k null mean."""
    y = np.asarray(series).astype(np.int64); best = (-np.inf, 1)
    for k in range(1, int(k_max) + 1):
        j = _encode_history(y, k, k, len(y)) * 2 + y[k:]
        cnt = np.bincount(j, minlength=2 ** (k + 1)).astype(float).reshape(-1, 2)
        n = cnt.sum(); ps = cnt.sum(1); py = cnt.sum(0)
        with np.errstate(divide='ignore', invalid='ignore'):
            ais = np.where(cnt > 0, cnt / n * np.log2(cnt * n / (ps[:, None] * py[None, :])), 0).sum()
        score = ais - base ** k * null_mean_bits(df_ais(k), n)
        if score > best[0]:
            best = (score, k)
    return best[1]


# ---------------------------------------------------------------------------
# 3. Weighted, tie-aware evaluation (physical pairs aggregated by group pair)
# ---------------------------------------------------------------------------

def _trapz(y, x):
    return float(np.trapezoid(y, x)) if hasattr(np, 'trapezoid') else float(np.trapz(y, x))


def weighted_curves(score, pos, neg):
    """Tie-aware ROC/PR with positive/negative weights per scored item."""
    score = np.asarray(score, float); pos = np.asarray(pos, float); neg = np.asarray(neg, float)
    keep = np.isfinite(score) & ((pos + neg) > 0)
    score, pos, neg = score[keep], pos[keep], neg[keep]
    P, Nn = pos.sum(), neg.sum()
    if len(score) == 0 or P == 0 or Nn == 0:
        return dict(auc=np.nan, ap=np.nan, prevalence=np.nan, ap_lift=np.nan, fpr=np.r_[0.], tpr=np.r_[0.],
                    precision=np.r_[1.], recall=np.r_[0.])
    order = np.argsort(-score, kind='stable')
    s, p, q = score[order], pos[order], neg[order]
    ends = np.r_[np.flatnonzero(np.diff(s) != 0), len(s) - 1]
    tp = np.cumsum(p)[ends]; fp = np.cumsum(q)[ends]
    tpr = np.r_[0, tp / P]; fpr = np.r_[0, fp / Nn]
    precision = tp / np.maximum(tp + fp, 1e-300); recall = tp / P
    ap = float(np.sum(np.diff(np.r_[0, recall]) * precision))
    prev = P / (P + Nn)
    return dict(auc=_trapz(tpr, fpr), ap=ap, prevalence=prev, ap_lift=ap / prev,
                fpr=fpr, tpr=tpr, precision=np.r_[precision[0], precision], recall=np.r_[0, recall])


def precision_at_budget(score, pos, neg, budget, unit='items'):
    """Expected precision/recall for the top `budget` items (or physical pairs);
    boundary ties included fractionally (uniform random tie-breaking)."""
    score = np.asarray(score, float); pos = np.asarray(pos, float); neg = np.asarray(neg, float)
    keep = np.isfinite(score) & ((pos + neg) > 0)
    score, pos, neg = score[keep], pos[keep], neg[keep]
    size = np.ones_like(score) if unit == 'items' else pos + neg
    order = np.argsort(-score, kind='stable')
    s, p, q, w = score[order], pos[order], neg[order], size[order]
    bounds = np.r_[0, np.flatnonzero(np.diff(s) != 0) + 1, len(s)]
    tp = fp = used = 0.0
    for a, b in zip(bounds[:-1], bounds[1:]):
        bw = w[a:b].sum(); frac = 1.0 if used + bw <= budget else max(0.0, (budget - used) / bw)
        tp += frac * p[a:b].sum(); fp += frac * q[a:b].sum(); used += frac * bw
        if frac < 1.0 or used >= budget:
            break
    P = pos.sum(); prev = P / (P + neg.sum()) if P + neg.sum() > 0 else np.nan
    prec = tp / (tp + fp) if tp + fp > 0 else np.nan
    return dict(budget=budget, expected_TP=tp, expected_FP=fp, precision=prec,
                recall=tp / P if P > 0 else np.nan, lift=prec / prev if prec == prec and prev else np.nan)


def binary_metrics(pred, pos, neg):
    pred = np.asarray(pred, bool); pos = np.asarray(pos, float); neg = np.asarray(neg, float)
    TP = pos[pred].sum(); FP = neg[pred].sum(); FN = pos[~pred].sum(); TN = neg[~pred].sum()
    prec = TP / (TP + FP) if TP + FP > 0 else np.nan
    rec = TP / (TP + FN) if TP + FN > 0 else np.nan
    f1 = 2 * TP / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else np.nan
    den = math.sqrt(max((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN), 0.0))
    prev = (TP + FN) / (TP + FP + FN + TN) if (TP + FP + FN + TN) > 0 else np.nan
    return dict(TP=TP, FP=FP, FN=FN, TN=TN, predicted=TP + FP, precision=prec, recall=rec, F1=f1,
                FPR=FP / (FP + TN) if FP + TN > 0 else np.nan,
                MCC=(TP * TN - FP * FN) / den if den > 0 else np.nan,
                precision_lift=prec / prev if prec == prec and prev else np.nan)


def bootstrap_targets(stat_fn, n_targets, draws=500, seed=2026):
    """Cluster bootstrap over target groups: stat_fn(multiplicity) -> dict."""
    rng = np.random.default_rng(seed); rows = []
    for d in range(int(draws)):
        mult = np.bincount(rng.integers(0, n_targets, n_targets), minlength=n_targets).astype(float)
        try:
            rows.append(dict(draw=d, **stat_fn(mult)))
        except (ValueError, ZeroDivisionError, FloatingPointError):
            continue
    return rows


def interval(rows, key):
    v = np.asarray([r[key] for r in rows if np.isfinite(r.get(key, np.nan))], float)
    if len(v) == 0:
        return (np.nan, np.nan, 0)
    return (float(np.quantile(v, .025)), float(np.quantile(v, .975)), len(v))


def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def bh_adjust(p, method='BH'):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.isfinite(p)
    q = p[ok]; m = len(q)
    if m == 0:
        return out
    order = np.argsort(q); factor = np.sum(1.0 / np.arange(1, m + 1)) if method == 'BY' else 1.0
    adj = np.minimum.accumulate((q[order] * m * factor / np.arange(1, m + 1))[::-1])[::-1]
    res = np.empty(m); res[order] = np.minimum(adj, 1.0); out[ok] = res
    return out


def bh_adjust_logp(log_p, method='BH'):
    """BH/BY on natural-log p-values without underflow; returns log adjusted p."""
    lp = np.asarray(log_p, float); out = np.full_like(lp, np.nan); ok = np.isfinite(lp)
    q = lp[ok]; m = len(q)
    if m == 0:
        return out
    order = np.argsort(q); factor = math.log(np.sum(1.0 / np.arange(1, m + 1))) if method == 'BY' else 0.0
    adj = q[order] + math.log(m) + factor - np.log(np.arange(1, m + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    res = np.empty(m); res[order] = np.minimum(adj, 0.0); out[ok] = res
    return out


def latex_table(df, path, caption, label, float_format='{:.4g}'):
    """Write a booktabs LaTeX table without optional pandas dependencies."""
    def fmt(v):
        if isinstance(v, (float, np.floating)):
            return '--' if not np.isfinite(v) else float_format.format(v)
        return str(v).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')
    cols = list(df.columns)
    lines = [r'\begin{table}[htbp]', r'\centering', r'\small', rf'\caption{{{caption}}}', rf'\label{{{label}}}',
             r'\begin{tabular}{' + 'l' * len(cols) + '}', r'\toprule',
             ' & '.join(fmt(c) for c in cols) + r' \\', r'\midrule']
    for _, row in df.iterrows():
        lines.append(' & '.join(fmt(row[c]) for c in cols) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    with open(path, 'w') as fh:
        fh.write('\n'.join(lines) + '\n')


# ---------------------------------------------------------------------------
# 4. Synthetic ground truth: clocked noisy Boolean logic networks
# ---------------------------------------------------------------------------

GATES = {
    'BUF': (1, lambda a: a[0]),
    'NOT': (1, lambda a: 1 - a[0]),
    'AND': (2, lambda a: a[0] & a[1]),
    'OR': (2, lambda a: a[0] | a[1]),
    'NAND': (2, lambda a: 1 - (a[0] & a[1])),
    'NOR': (2, lambda a: 1 - (a[0] | a[1])),
    'XOR': (2, lambda a: a[0] ^ a[1]),
    'MAJ3': (3, lambda a: ((a[0].astype(np.int16) + a[1] + a[2]) >= 2).astype(np.int8)),
}
INVERTING = {'NOT', 'NAND', 'NOR'}


def make_logic_network(n_nodes=40, n_inputs=5, gate_mix=None, seed=0, clock=True, clock_fraction=0.3,
                       feedback_fraction=0.15):
    """Random logic circuit with known parents (mostly feed-forward plus feedback).
    Nodes 0..n_inputs-1 are persistent stochastic inputs; node n_inputs is a
    period-2 clock if clock=True; remaining nodes are gates with one-step delay."""
    rng = np.random.default_rng(seed)
    gate_mix = gate_mix or {'BUF': .10, 'NOT': .15, 'AND': .15, 'OR': .15, 'NAND': .15,
                            'NOR': .15, 'XOR': .10, 'MAJ3': .05}
    names = list(gate_mix); probs = np.asarray(list(gate_mix.values()), float); probs /= probs.sum()
    first = n_inputs + (1 if clock else 0)
    gates = ['INPUT'] * n_inputs + (['CLOCK'] if clock else []) + list(rng.choice(names, n_nodes - first, p=probs))
    parents = [[] for _ in range(n_nodes)]
    for v in range(first, n_nodes):
        arity = GATES[gates[v]][0]
        pool = np.arange(v)
        if v > first + 3 and rng.random() < feedback_fraction:
            pool = np.arange(n_nodes)[np.arange(n_nodes) != v]
        chosen = [int(c) for c in rng.choice(pool, size=min(arity, len(pool)), replace=False)]
        if clock and arity >= 2 and n_inputs not in chosen and rng.random() < clock_fraction:
            chosen[-1] = n_inputs
        parents[v] = chosen
    A = np.zeros((n_nodes, n_nodes), dtype=np.int8); inv = np.zeros_like(A)
    for v, ps in enumerate(parents):
        for p in ps:
            A[p, v] = 1; inv[p, v] = int(gates[v] in INVERTING)
    return dict(adjacency=A, gates=gates, parents=parents, inverting=inv, n_inputs=n_inputs,
                clock_node=n_inputs if clock else None)


def simulate_logic_network(net, T, noise=0.02, input_persistence=0.8, seed=0, representation='state'):
    """Synchronous one-step-delay update with independent bit-flip noise.
    representation='state' -> node states; 'rising' -> 0->1 event indicators
    (the representation of the 6502 recording)."""
    rng = np.random.default_rng(seed)
    gates = net['gates']; n = len(gates)
    x = np.zeros((n, T), dtype=np.int8); x[:, 0] = rng.integers(0, 2, n)
    inputs = np.array([v for v, g in enumerate(gates) if g == 'INPUT'], dtype=int)
    clocks = np.array([v for v, g in enumerate(gates) if g == 'CLOCK'], dtype=int)
    by_type = {}
    for v, g in enumerate(gates):
        if g in GATES:
            by_type.setdefault(g, []).append(v)
    by_type = {g: (np.asarray(v), np.asarray([net['parents'][u] for u in v])) for g, v in by_type.items()}
    keep = rng.random((len(inputs), T)) < input_persistence
    fresh = rng.integers(0, 2, (len(inputs), T)).astype(np.int8)
    flips = rng.random((n, T)) < noise
    for t in range(1, T):
        prev = x[:, t - 1]; cur = x[:, t]
        if len(inputs):
            cur[inputs] = np.where(keep[:, t], prev[inputs], fresh[:, t])
        if len(clocks):
            cur[clocks] = 1 - prev[clocks]
        for g, (nodes, par) in by_type.items():
            cur[nodes] = GATES[g][1](prev[par].T)
        cur[flips[:, t]] ^= 1
    if representation == 'rising':
        ev = np.zeros_like(x); ev[:, 1:] = ((x[:, 1:] == 1) & (x[:, :-1] == 0)).astype(np.int8)
        return ev
    return x


def edge_categories(A):
    A = np.asarray(A) > 0; I = np.eye(len(A), dtype=bool); Ai = A.astype(int)
    return dict(direct=A & ~I, reverse_only=A.T & ~A & ~I,
                two_step=((Ai @ Ai) > 0) & ~A & ~I,
                shared_parent=((Ai.T @ Ai) > 0) & ~A & ~A.T & ~I,
                shared_child=((Ai @ Ai.T) > 0) & ~A & ~A.T & ~I)


def classify_false_positives(pred, A):
    cats = edge_categories(A); pred = np.asarray(pred, bool) & ~np.eye(len(A), dtype=bool)
    fp = pred & ~cats['direct']; out = dict(false_positives=int(fp.sum()))
    done = np.zeros_like(fp)
    for name in ['reverse_only', 'two_step', 'shared_parent', 'shared_child']:
        m = fp & cats[name] & ~done; out[f'fp_{name}'] = int(m.sum()); done |= m
    out['fp_unrelated'] = int((fp & ~done).sum())
    return out


def metrics_from_prediction(pred, A, label):
    A = np.asarray(A) > 0; off = ~np.eye(len(A), dtype=bool)
    m = binary_metrics(np.asarray(pred, bool)[off], A[off].astype(float), (~A)[off].astype(float))
    m.update(classify_false_positives(pred, A)); m['method'] = label
    return m


# ---------------------------------------------------------------------------
# 5. IDTxl: speed patch, bit-correct analytic surrogates, workers
# ---------------------------------------------------------------------------

def ensure_jvm(jar_path=None, max_heap='3g'):
    """Start the JVM with an explicit heap before IDTxl does (IDTxl otherwise
    starts it with the platform default heap)."""
    import jpype
    if jpype.isJVMStarted():
        return False
    if jar_path is None:
        import os, idtxl
        jar_path = os.path.join(os.path.dirname(idtxl.__file__), 'infodynamics.jar')
    jpype.startJVM(jpype.getDefaultJVMPath(), '-ea', f'-Xmx{max_heap}', classpath=[str(jar_path)])
    return True


def _fast_combine_discrete_dimensions(a, numBins):
    """Vectorised, output-identical replacement for
    idtxl.idtxl_utils.combine_discrete_dimensions (a per-sample Python loop
    that dominates IDTxl run time on long discrete recordings)."""
    a = np.asarray(a)
    if a.ndim == 1:
        return a
    dims = a.shape[1]
    if dims * math.log2(max(int(numBins), 2)) >= 62:
        raise ArithmeticError('Combination of numBins and number of dimensions of a '
                              'leads to overflow in making unidimensional array')
    weights = int(numBins) ** np.arange(dims - 1, -1, -1, dtype=np.int64)
    return (a.astype(np.int64) @ weights).astype(np.int_)


def patch_idtxl():
    """NumPy-2 compatibility plus the vectorised helper (idempotent)."""
    if not hasattr(np, 'issubclass_'):
        np.issubclass_ = issubclass  # IDTxl calls np.issubclass_, removed in NumPy 2.0
    import idtxl.idtxl_utils as _u
    _u.combine_discrete_dimensions = _fast_combine_discrete_dimensions
    return True


_BITS_ESTIMATOR = None


def make_bits_correct_estimator():
    """IDTxl estimator whose analytic surrogates are on the same (bit) scale as
    its estimates. JIDT discrete calculators return bits but draw analytic null
    values as chi2/(2N) (nats); IDTxl compares them directly whenever
    permute_in_time=True, which makes every test anti-conservative."""
    global _BITS_ESTIMATOR
    if _BITS_ESTIMATOR is not None:
        return _BITS_ESTIMATOR
    patch_idtxl()
    from idtxl.estimators_jidt import JidtDiscreteCMI

    class JidtDiscreteCMIBits(JidtDiscreteCMI):
        UNIT_SCALE = 1.0 / LN2

        def estimate_surrogates_analytic(self, n_perm=200, **data):
            return np.asarray(super().estimate_surrogates_analytic(n_perm=n_perm, **data)) * self.UNIT_SCALE

    _BITS_ESTIMATOR = JidtDiscreteCMIBits
    return JidtDiscreteCMIBits


def idtxl_settings(max_lag_target=12, max_lag_sources=1, min_lag_sources=1, alpha=0.05,
                   n_perm=200, perm_block=60, analytic_bits_fix=True, alpha_omnibus=None):
    """IDTxl settings for binary data with the bit-correct analytic null."""
    patch_idtxl()
    return dict(cmi_estimator=make_bits_correct_estimator() if analytic_bits_fix else 'JidtDiscreteCMI',
                discretise_method='none', n_discrete_bins=2,
                max_lag_target=int(max_lag_target), tau_target=1,
                max_lag_sources=int(max_lag_sources), min_lag_sources=int(min_lag_sources), tau_sources=1,
                alpha_max_stat=alpha, alpha_min_stat=alpha, alpha_omnibus=alpha if alpha_omnibus is None else alpha_omnibus, alpha_max_seq=alpha,
                n_perm_max_stat=n_perm, n_perm_min_stat=n_perm, n_perm_omnibus=n_perm, n_perm_max_seq=n_perm,
                permute_in_time=True, perm_type='block', block_size=int(perm_block),
                fdr_correction=False, verbose=False, write_ckp=False)


_LINKWISE = None


def make_linkwise_bivariate():
    """IDTxl BivariateTE whose final step skips the joint omnibus test.

    IDTxl's bivariate omnibus test estimates TE from *all* individually selected
    sources jointly. With many sources the analytic null has a degrees-of-freedom
    count far beyond the sample size (it can never reject, silently emptying the
    network) and JIDT allocates count arrays exponential in the number of sources
    (out-of-memory). Link-wise inference does not need a joint test: this class
    keeps IDTxl's per-source sequential maximum statistics and bivariate single-link
    TE unchanged, and FDR is applied at link level afterwards.
    """
    global _LINKWISE
    if _LINKWISE is not None:
        return _LINKWISE
    patch_idtxl()
    from idtxl.bivariate_te import BivariateTE
    import idtxl.network_inference as _ni

    class BivariateTELinkwise(BivariateTE):
        def _test_final_conditional(self, data):
            original = _ni.stats.omnibus_test
            _ni.stats.omnibus_test = lambda setup, d: (True, float('nan'), float('nan'))
            try:
                return super()._test_final_conditional(data)
            finally:
                _ni.stats.omnibus_test = original

    _LINKWISE = BivariateTELinkwise
    return BivariateTELinkwise


def _quiet(fn, *args, **kwargs):
    """Run an IDTxl call without its progress printing and benign cast warnings."""
    import io, contextlib, warnings
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()), warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return fn(*args, **kwargs)


def _single_summary(res, target, method, seconds, n_candidates=None, n_obs=None):
    s = res.get_single_target(int(target), fdr=False)
    arr = lambda v: [] if v is None else [float(u) for u in np.atleast_1d(v)]
    te_sel = arr(s.selected_sources_te); n_tv = len(s.selected_vars_target or [])
    # Exact bit-correct analytic log-p for each selected source: one binary source
    # variable given the selected target past (bivariate) or the full conditioning set.
    n_cond = n_tv + (0 if method == 'bivariate' else max(len(s.selected_vars_sources or []) - 1, 0))
    logp = [float(log_p_from_bits(v, n_obs, df_te(n_cond))) if n_obs else None for v in te_sel]
    return dict(target=int(target), method=method, seconds=float(seconds), n_candidates=n_candidates, n_obs=n_obs,
                selected_sources_logp_analytic=logp,
                selected_vars_sources=[(int(a), int(b)) for a, b in (s.selected_vars_sources or [])],
                selected_vars_target=[(int(a), int(b)) for a, b in (s.selected_vars_target or [])],
                selected_sources_te=arr(s.selected_sources_te), selected_sources_pval=arr(s.selected_sources_pval),
                omnibus_te=None if s.omnibus_te is None else float(s.omnibus_te),
                omnibus_pval=None if s.omnibus_pval is None else float(s.omnibus_pval),
                omnibus_sign=bool(s.omnibus_sign) if s.omnibus_sign is not None else False)


def idtxl_target_worker(job):
    """Process-pool worker for one 6502 target. job: dict(data_path, target,
    method, settings, sources, out_path, max_heap). Writes a JSON summary."""
    import json, time, os
    ensure_jvm(max_heap=job.get('max_heap', '3g'))
    patch_idtxl()
    from idtxl.data import Data
    if job['method'] == 'multivariate':
        from idtxl.multivariate_te import MultivariateTE as Analysis
    else:
        Analysis = make_linkwise_bivariate()
    values = np.load(job['data_path'], mmap_mode='r')
    data = _quiet(Data, np.asarray(values, dtype=np.int32), dim_order='ps', normalise=False)
    kw = dict(job['settings'])
    settings = idtxl_settings(**kw)
    sources = job.get('sources') or [u for u in range(values.shape[0]) if u != int(job['target'])]
    t0 = time.perf_counter()
    res = _quiet(Analysis().analyse_single_target, settings=settings, data=data, target=int(job['target']),
                 sources=sources)
    lags = kw.get('max_lag_sources', 1) - kw.get('min_lag_sources', 1) + 1
    summary = _single_summary(res, job['target'], job['method'], time.perf_counter() - t0,
                              n_candidates=len(sources) * lags,
                              n_obs=values.shape[1] - max(kw.get('max_lag_target', 12), kw.get('max_lag_sources', 1)))
    tmp = job['out_path'] + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(summary, fh)
    os.replace(tmp, job['out_path'])
    return summary


def network_fdr_omnibus(summaries, alpha=0.05):
    """IDTxl-style network FDR: BH across targets' omnibus p-values; a target's
    selected sources are kept only if its omnibus test survives."""
    if not summaries:
        return {}
    p = np.asarray([1.0 if s['omnibus_pval'] is None else s['omnibus_pval'] for s in summaries], float)
    q = bh_adjust(p)
    return {s['target']: bool(qq <= alpha and len(s['selected_vars_sources']) > 0) for s, qq in zip(summaries, q)}


def network_fdr_links(summaries, alpha=0.05):
    """Link-level BH across all candidate sources of all targets (IDTxl
    correct_by_target=False), using exact bit-correct analytic p-values of the
    selected sources; unselected candidates enter with p = 1."""
    m = int(sum(s.get('n_candidates') or 0 for s in summaries))
    items = [(s['target'], u, lp) for s in summaries
             for (u, _), lp in zip(s['selected_vars_sources'], s.get('selected_sources_logp_analytic') or [])
             if lp is not None]
    if not items or m == 0:
        return set()
    lps = np.asarray([lp for *_, lp in items]); order = np.argsort(lps)
    thresh = np.log(alpha) + np.log(np.arange(1, len(lps) + 1)) - np.log(m)
    passed = np.flatnonzero(lps[order] <= thresh)
    k = passed.max() + 1 if len(passed) else 0
    return {(items[i][1], items[i][0]) for i in order[:k]}


def assemble_network(summaries, method, n, alpha=0.05):
    """Adjacency [source, target] from IDTxl summaries: omnibus network FDR for
    multivariate TE; link-level FDR for bivariate TE."""
    A = np.zeros((n, n), bool)
    if method == 'bivariate':
        for u, t in network_fdr_links(summaries, alpha):
            A[u, t] = True
        return A
    keep = network_fdr_omnibus(summaries, alpha)
    for s in summaries:
        if keep.get(s['target']):
            for u, _ in s['selected_vars_sources']:
                A[u, s['target']] = True
    return A


def synthetic_task(args):
    """One synthetic replicate: simulate a circuit, then pairwise TE (exact,
    bit-correct BH) and optionally IDTxl bivariate/multivariate TE.
    Returns dict(rows=[method metrics], edges=[per-true-edge records])."""
    import time
    a = dict(args)
    net = make_logic_network(a['n_nodes'], a['n_inputs'], seed=a['seed'], clock=True)
    X = simulate_logic_network(net, a['T'], noise=a['noise'], seed=a['seed'] + 7919, representation=a['representation'])
    A = net['adjacency'].astype(bool); n = len(A); off = ~np.eye(n, dtype=bool)
    constant = X.min(axis=1) == X.max(axis=1)
    ks = np.asarray([1 if constant[v] else ais_select_k(X[v], a['k_max']) for v in range(n)])
    te, nobs, delta, _ = te_matrix(X, ks, details=True)
    te = np.where(np.isfinite(te), te, 0.0)
    logp = np.full((n, n), np.nan)
    for t in range(n):
        logp[:, t] = log_p_from_bits(te[:, t], nobs[t], df_te(ks[t]))
    q = np.full((n, n), np.nan); q[off] = np.exp(bh_adjust_logp(logp[off]))
    base = dict(seed=a['seed'], T=a['T'], representation=a['representation'], n_nodes=n, true_edges=int(A[off].sum()))
    pw = (q <= a['alpha']) & off
    curve = weighted_curves(te[off], A[off].astype(float), (~A[off]).astype(float))
    m = metrics_from_prediction(pw, A, 'Pairwise TE (bit-correct, BH)'); m.update(base, AUC=curve['auc'], AP=curve['ap'])
    rows = [m]
    order = np.argsort(-te[off]); top = np.zeros(off.sum(), bool); top[order[:base['true_edges']]] = True
    pb = np.zeros((n, n), bool); pb[off] = top
    m = metrics_from_prediction(pb, A, 'Pairwise TE (top-|E| budget)'); m.update(base); rows.append(m)
    edges = []
    for s_, t_ in zip(*np.nonzero(A & off)):
        edges.append(dict(**base, source=int(s_), target=int(t_), gate=net['gates'][t_],
                          inverting=bool(net['inverting'][s_, t_]), clock_source=bool(s_ == net['clock_node']),
                          pairwise_te=float(te[s_, t_]), reverse_te=float(te[t_, s_]),
                          pairwise_detected=bool(pw[s_, t_]), delta=float(delta[s_, t_])))
    if a.get('run_idtxl'):
        ensure_jvm(max_heap=a.get('max_heap', '2g')); patch_idtxl()
        from idtxl.data import Data
        data = _quiet(Data, X.astype(np.int32), dim_order='ps', normalise=False)
        for method, fix in a['idtxl_methods']:
            if method == 'multivariate':
                from idtxl.multivariate_te import MultivariateTE as Analysis
            else:
                Analysis = make_linkwise_bivariate()
            settings = idtxl_settings(max_lag_target=a['idtxl_max_lag_target'], max_lag_sources=1,
                                      analytic_bits_fix=fix, alpha=a['alpha'])
            t0 = time.perf_counter(); summaries = []
            for t in range(n):
                srcs = [u for u in range(n) if u != t and not constant[u]]
                if constant[t] or not srcs:
                    continue
                r = _quiet(Analysis().analyse_single_target, settings=settings, data=data, target=t, sources=srcs)
                summaries.append(_single_summary(r, t, method, 0.0, n_candidates=len(srcs),
                                                 n_obs=a['T'] - a['idtxl_max_lag_target']))
            pred = assemble_network(summaries, method, n, a['alpha'])
            label = f"IDTxl {method} TE" + ('' if fix else ' (uncorrected analytic null)')
            m = metrics_from_prediction(pred, A, label); m.update(base, seconds=time.perf_counter() - t0)
            rows.append(m)
            if fix:
                for e in edges:
                    e[f'idtxl_{method}_detected'] = bool(pred[e['source'], e['target']])
    return dict(rows=rows, edges=edges)


# ---------------------------------------------------------------------------
# 7. Local information dynamics, PID, generic references, modules
# ---------------------------------------------------------------------------

def local_te(x, y, k, delay=1):
    """Local (pointwise) TE in bits for t = start..T-1, plug-in probabilities
    from the whole series: log2 p(y_t | s_t, x_{t-delay}) / p(y_t | s_t)."""
    y = np.asarray(y).astype(np.int64); x = np.asarray(x).astype(np.int64)
    start = max(int(k), int(delay)); s = _encode_history(y, k, start, len(y))
    yn = y[start:]; xs = x[start - delay:len(y) - delay]
    j3 = (s * 2 + xs) * 2 + yn; j2 = s * 2 + yn
    c3 = np.bincount(j3); c_sx = np.bincount(s * 2 + xs); c2 = np.bincount(j2); c_s = np.bincount(s)
    p_full = c3[j3] / c_sx[s * 2 + xs]; p_hist = c2[j2] / c_s[s]
    return np.log2(p_full / p_hist), start


def local_ais(y, k):
    y = np.asarray(y).astype(np.int64); s = _encode_history(y, k, k, len(y)); yn = y[k:]
    j = s * 2 + yn; cj = np.bincount(j); cs = np.bincount(s); cy = np.bincount(yn, minlength=2)
    return np.log2((cj[j] / cs[s]) / (cy[yn] / len(yn))), k


def _mi_discrete(a, b):
    a = np.asarray(a).astype(np.int64); b = np.asarray(b).astype(np.int64)
    j = a * (b.max() + 1) + b; cj = np.bincount(j).astype(float); n = len(a)
    ca = np.bincount(a).astype(float); cb = np.bincount(b).astype(float)
    nz = cj > 0; ja = np.flatnonzero(nz) // (b.max() + 1); jb = np.flatnonzero(nz) % (b.max() + 1)
    return float(np.sum(cj[nz] / n * np.log2(cj[nz] * n / (ca[ja] * cb[jb]))))


def pid_two_sources(x1, x2, y):
    """Two-source partial information decomposition (bits).
    Redundancy by Williams & Beer (2010) I_min and by minimum mutual information
    (MMI; Barrett 2015). Returns both decompositions and the joint MI."""
    x1 = np.asarray(x1).astype(np.int64); x2 = np.asarray(x2).astype(np.int64); y = np.asarray(y).astype(np.int64)
    i1, i2 = _mi_discrete(x1, y), _mi_discrete(x2, y); i12 = _mi_discrete(x1 * 2 + x2, y)
    n = len(y); py = np.bincount(y, minlength=2) / n
    def spec(x):
        out = np.zeros(2)
        for yy in (0, 1):
            if py[yy] == 0: continue
            m = y == yy; total = 0.0
            for xx in np.unique(x):
                pxy = np.mean(x[m] == xx); pyx = np.mean(y[x == xx] == yy)
                if pxy > 0 and pyx > 0: total += pxy * np.log2(pyx / py[yy])
            out[yy] = total
        return out
    s1, s2 = spec(x1), spec(x2)
    red_imin = float(np.sum(py * np.minimum(s1, s2))); red_mmi = min(i1, i2)
    res = dict(I_x1=i1, I_x2=i2, I_joint=i12)
    for name, red in [('Imin', red_imin), ('MMI', red_mmi)]:
        u1, u2 = i1 - red, i2 - red
        res.update({f'{name}_redundancy': red, f'{name}_unique1': u1, f'{name}_unique2': u2,
                    f'{name}_synergy': i12 - red - u1 - u2})
    return res


def group_identical(X):
    """Group identical non-constant rows of X [units, T]. Returns (representatives, unit_to_group)."""
    X = np.asarray(X); packed = np.packbits(X.astype(bool), axis=1)
    lookup, reps = {}, []; ug = np.full(len(X), -1, dtype=np.int64)
    for u in range(len(X)):
        if X[u].min() == X[u].max():
            continue
        key = packed[u].tobytes()
        if key not in lookup:
            lookup[key] = len(reps); reps.append(u)
        ug[u] = lookup[key]
    return np.asarray(reps, dtype=np.int64), ug


def group_reference(A, unit_to_group, G):
    """Aggregate a unit-level directed reference A [units, units] (dense/sparse,
    nonnegative weights) to group-pair positive counts and negative counts."""
    A = sparse.csr_matrix(A); act = np.flatnonzero(unit_to_group >= 0)
    P = sparse.csr_matrix((np.ones(len(act)), (act, unit_to_group[act])), shape=(A.shape[0], G))
    pos = np.asarray((P.T @ (A > 0).astype(float) @ P).todense()); np.fill_diagonal(pos, 0)
    size = np.bincount(unit_to_group[act], minlength=G).astype(float)
    pairs = np.outer(size, size); np.fill_diagonal(pairs, 0)
    return pos, pairs - pos


def rank_metrics(score, pos, neg, budgets=(100, 1000), level='physical'):
    """AUC, AP lift and precision lift at item budgets for a [G, G] score.
    Non-finite scores mark excluded items (e.g. non-intervened sources)."""
    G = len(score); el = ~np.eye(G, dtype=bool); s = np.asarray(score, float)[el]
    p, n = (pos[el], neg[el]) if level == 'physical' else ((pos[el] > 0).astype(float), (pos[el] == 0).astype(float))
    keep = np.isfinite(s); s, p, n = s[keep], p[keep], n[keep]
    c = weighted_curves(s, p, n)
    out = dict(AUC=c['auc'], AP=c['ap'], AP_lift=c['ap_lift'], prevalence=c['prevalence'])
    for b in budgets:
        out[f'lift@{b}'] = precision_at_budget(s, p, n, b)['lift']
    return out


MODULE_RULES = [
    ('Registers A/X/Y/S', r'^(a|x|y|s|nota|notx|noty|nots)\d$'),
    ('Program counter', r'^(pcl|pch|pclp|pchp|notpcl|notpch|#?pcl|#?pch)\w*$'),
    ('Status flags', r'^(p\d|.*flag.*|#?C\w*out|D1x1|nmig|irqp\w*)$'),
    ('ALU', r'^(alu\w*|#?alu\w*|dasb\w*|daa\w*|dsa\w*|notalu\w*|aluvout|alucout)$'),
    ('Internal buses', r'^(idb|adl|adh|sb|notidl|idl)\d$'),
    ('External buses and latches', r'^(ab|db|abl|abh|dor|dl|notdor)\d$'),
    ('Instruction register and decode', r'^(ir|notir|pd|op-.*|#op-.*|clearir|fetch|.*-op.*)\d*$'),
    ('Timing and control', r'^(t\d|clk\w*|cp1|cclk|sync|rw|res\w*|irq|nmi|rdy|so|phi\w*|#?T\d\w*)$'),
]


def module_of_name(name):
    import re as _re
    for module, pattern in MODULE_RULES:
        if _re.match(pattern, name, _re.I):
            return module
    return None


def assign_modules(n_units, unit_nodes, node_names, A_units, propagate_rounds=6):
    """Module label for each unit from named nodes (unit_nodes[u] = candidate
    node ids in priority order), then majority label propagation over the
    undirected structural graph A_units for unnamed units. Returns array of str."""
    name_of = {}
    for nm, nid in node_names.items():
        name_of.setdefault(nid, nm)
    labels = np.array([None] * n_units, dtype=object); source = np.array(['unassigned'] * n_units, dtype=object)
    for u in range(n_units):
        for nid in unit_nodes[u]:
            m = module_of_name(name_of.get(int(nid), '')) if int(nid) in name_of else None
            if m:
                labels[u] = m; source[u] = 'named'; break
    S = sparse.csr_matrix(A_units); S = ((S + S.T) > 0).tocsr()
    for _ in range(propagate_rounds):
        changed = 0; new = labels.copy()
        for u in np.flatnonzero(labels == None):  # noqa: E711
            neigh = [labels[v] for v in S.indices[S.indptr[u]:S.indptr[u + 1]] if labels[v] is not None]
            if neigh:
                vals, cnt = np.unique(np.asarray(neigh, dtype=str), return_counts=True)
                new[u] = vals[np.argmax(cnt)]; source[u] = 'propagated'; changed += 1
        labels = new
        if not changed:
            break
    labels = np.where(labels == None, 'Unassigned', labels).astype(str)  # noqa: E711
    return labels, source


def within_between_ratio(score, module, perms=1000, seed=0):
    """Mean score within modules / mean between modules for a [G,G] score,
    with a label-permutation null (sizes preserved). 'Unassigned' excluded."""
    rng = np.random.default_rng(seed); S = np.asarray(score, float)
    keep = module != 'Unassigned'; S = S[np.ix_(keep, keep)]; mod = module[keep]
    off = ~np.eye(len(mod), dtype=bool); ok = off & np.isfinite(S)
    def ratio(m):
        same = (m[:, None] == m[None, :]) & ok
        return S[same].mean() / S[ok & ~same].mean()
    obs = ratio(mod); null = np.asarray([ratio(rng.permutation(mod)) for _ in range(perms)])
    return dict(ratio=float(obs), null_mean=float(null.mean()), null_q975=float(np.quantile(null, .975)),
                p_perm=float((1 + np.sum(null >= obs)) / (perms + 1)), units=int(keep.sum()))


def community_agreement(W, labels, seed=0, resolution=1.0):
    """Louvain communities of the symmetrised weighted graph W vs module labels:
    NMI and ARI (sklearn). Returns dict and community assignment."""
    from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
    try:
        import networkx as nx
        nx.community.louvain_communities
    except (ImportError, AttributeError):
        return dict(communities=np.nan, NMI=np.nan, ARI=np.nan, units=0), None
    W = np.nan_to_num(np.asarray(W, float)); W = np.maximum(W, W.T); np.fill_diagonal(W, 0)
    keep = labels != 'Unassigned'; W = W[np.ix_(keep, keep)]; lab = labels[keep]
    g = nx.from_numpy_array(W)
    comms = nx.community.louvain_communities(g, weight='weight', resolution=resolution, seed=seed)
    assign = np.zeros(len(lab), int)
    for c_, members in enumerate(comms):
        assign[list(members)] = c_
    return dict(communities=len(comms), NMI=float(normalized_mutual_info_score(lab, assign)),
                ARI=float(adjusted_rand_score(lab, assign)), units=int(keep.sum())), assign


def node_reference_from_transistors(gate, c1, c2, nodes, supply=()):
    """Directed node-level signal-flow reference: gate node -> each channel node
    of every transistor (supply nodes excluded), restricted to recorded nodes."""
    idx = {int(n): i for i, n in enumerate(nodes)}; rows, cols = [], []
    for g, a, b in zip(gate, c1, c2):
        if int(g) not in idx:
            continue
        for c in (a, b):
            if int(c) in supply or int(c) not in idx or int(c) == int(g):
                continue
            rows.append(idx[int(g)]); cols.append(idx[int(c)])
    A = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(nodes), len(nodes)))
    A.data[:] = 1.0
    return A


def nmos_task(args):
    """Synthetic NMOS benchmark replicate (switch-level simulation, known wiring).
    Methods: pairwise TE at lag 0 and lag 1 (BH, bit-correct; and oracle budget),
    zero-lag MI, and optionally IDTxl multivariate / link-wise bivariate TE (lag 1)."""
    import time, sim6502
    a = dict(args)
    defs, meta = sim6502.make_nmos_circuit(n_gates=a['n_gates'], n_inputs=a['n_inputs'], n_latches=a['n_latches'], seed=a['seed'])
    S, chip = sim6502.simulate_nmos(defs, meta, a['T'] + 200, seed=a['seed'] + 13)
    S = S[200:].T
    if a['representation'] == 'rising':
        R = np.zeros_like(S); R[:, 1:] = (S[:, 1:] == 1) & (S[:, :-1] == 0); S = R
    nodes = np.asarray(chip.used_nodes)
    g_, c1_, c2_ = chip.transistor_matrix()
    A_units = node_reference_from_transistors(g_, c1_, c2_, nodes, supply={chip.gnd, chip.pwr})
    reps, ug = group_identical(S); G = len(reps); X = S[reps]
    pos, neg = group_reference(A_units, ug, G)
    base = dict(seed=a['seed'], T=a['T'], representation=a['representation'], nodes=len(nodes), groups=G,
                wired_pairs=float(pos.sum()))
    rows = []
    if G < 3:
        return dict(rows=rows)
    ks = np.asarray([ais_select_k(x, a['k_max']) for x in X])
    off = ~np.eye(G, dtype=bool)
    mi, _ = te_matrix(X, np.zeros(G, int), delay=0)
    r = rank_metrics(mi, pos, neg, budgets=()); rows.append(dict(base, method='Zero-lag MI (ranking)', **r))
    for d in (0, 1):
        te, nobs = te_matrix(X, ks, delay=d); te = np.where(np.isfinite(te), te, 0)
        r = rank_metrics(te, pos, neg, budgets=()); rows.append(dict(base, method=f'Pairwise TE lag {d} (ranking)', **r))
        logp = np.full((G, G), np.nan)
        for t in range(G):
            logp[:, t] = log_p_from_bits(te[:, t], nobs[t], df_te(ks[t]))
        pred = np.zeros((G, G), bool); pred[off] = bh_adjust_logp(logp[off]) <= np.log(a['alpha'])
        m = binary_metrics(pred[off], pos[off], neg[off]); rows.append(dict(base, method=f'Pairwise TE lag {d} (BH)', **m))
    if a.get('run_idtxl') and G <= a.get('idtxl_max_groups', 80):
        ensure_jvm(max_heap=a.get('max_heap', '1g')); patch_idtxl()
        from idtxl.data import Data
        from idtxl.multivariate_te import MultivariateTE
        data = _quiet(Data, X.astype(np.int32), dim_order='ps', normalise=False)
        for method in a['idtxl_methods']:
            Analysis = MultivariateTE if method.startswith('multivariate') else make_linkwise_bivariate()
            settings = idtxl_settings(max_lag_target=a['k_max'], max_lag_sources=1,
                                      min_lag_sources=0 if method.endswith('lag0') else 1, alpha=a['alpha'])
            t0 = time.perf_counter(); summaries = []
            for t in range(G):
                res = _quiet(Analysis().analyse_single_target, settings=settings, data=data, target=t,
                             sources=[u for u in range(G) if u != t])
                summaries.append(_single_summary(res, t, method, 0.0, n_candidates=G - 1, n_obs=a['T'] - a['k_max']))
            pred = assemble_network(summaries, 'bivariate' if method == 'bivariate' else 'multivariate', G, a['alpha'])
            m = binary_metrics(pred[off], pos[off], neg[off])
            label = {'multivariate': 'IDTxl multivariate TE (lags 1)', 'multivariate_lag0': 'IDTxl multivariate (source lags 0-1)',
                     'bivariate': 'IDTxl bivariate TE (lag 1)'}.get(method, method)
            rows.append(dict(base, method=label, seconds=time.perf_counter() - t0, **m))
    return dict(rows=rows)

In [ ]:
# 5 Shared setup: load the run, the recording and the netlist; build structural references.
import sys, importlib, json, math, time, hashlib, platform, ast, traceback
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display, Markdown
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
import te6502_toolkit as tk
importlib.reload(tk)

RQ_DIR = OUT / 'thesis_v5'; RQ_FIG = RQ_DIR / 'figures'; RQ_TAB = RQ_DIR / 'tables'
for _d in (RQ_DIR, RQ_FIG, RQ_TAB):
    _d.mkdir(parents=True, exist_ok=True)
(RQ_DIR / 'configuration.json').write_text(json.dumps(RQ, indent=2))
plt.rcParams.update({'savefig.dpi': 300, 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 9, 'legend.fontsize': 8, 'figure.dpi': 110})

def rq_json(value):
    if isinstance(value, dict): return {str(k): rq_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [rq_json(v) for v in value]
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating, float)): return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_): return bool(value)
    if isinstance(value, np.ndarray): return rq_json(value.tolist())
    return value

RESULTS = json.loads((RQ_DIR / 'results_ledger.json').read_text()) if (RQ_DIR / 'results_ledger.json').exists() else {}
FIGURES, TABLES = {}, {}
def rq_record(key, value):
    RESULTS[key] = rq_json(value)
    (RQ_DIR / 'results_ledger.json').write_text(json.dumps(RESULTS, indent=2))

def rq_fig(fig, name, caption):
    for ext in ('pdf', 'png'):
        fig.savefig(RQ_FIG / f'{name}.{ext}', bbox_inches='tight')
    FIGURES[name] = caption; plt.show(); plt.close(fig)

def rq_table(df, name, caption, show=True, fmt='{:.4g}'):
    df.to_csv(RQ_TAB / f'{name}.csv', index=False)
    tk.latex_table(df, RQ_TAB / f'{name}.tex', caption, f'tab:{name}', fmt)
    TABLES[name] = caption
    if show:
        display(df)

def rq_ci(point, lo, hi, digits=3):
    f = f'{{:.{digits}f}}'
    return '--' if not np.isfinite(point) else f'{f.format(point)} [{f.format(lo)}, {f.format(hi)}]'

# Run outputs (Sections 1-2).
with np.load(OUT / 'groups.npz') as z:
    rq_reps = z['representatives'].astype(int); rq_p2g = z['physical_to_group'].astype(int)
with np.load(OUT / 'reduction_audit.npz') as z:
    rq_phase = int(z['phase']); rq_stride = int(z['stride']); rq_stop = int(z['original_active_rows'])
with np.load(OUT / 'te.npz') as z:
    RQ_TE = z['te_bits'].copy(); RQ_LOGP = z['log_p'].copy(); RQ_JIDT_P = z['jidt_p'].copy()
    # chi2.logsf can return -inf for astronomically small p; keep such tests (as most significant) in all rankings and FDR steps.
    RQ_LOGP = np.where(np.isneginf(RQ_LOGP), np.nanmin(np.where(np.isfinite(RQ_LOGP), RQ_LOGP, 0)) - 1e3, RQ_LOGP)
    RQ_K = z['optimal_k'].astype(int); RQ_NOBS = z['observations'].astype(int)
with np.load(OUT / 'mi.npz') as z:
    RQ_MI = z['mi_bits'].copy()
rq_raw = np.load(DATA_PATH, mmap_mode='r')

def rq_series(phase=None, stride=None, block=40000):
    """[group, time] uint8 series for any phase/stride using this run's representatives."""
    phase = rq_phase if phase is None else int(phase); stride = rq_stride if stride is None else int(stride)
    rows = np.arange(phase, rq_stop, stride)
    out = np.empty((len(rq_reps), len(rows)), dtype=np.uint8)
    for a in range(0, len(rows), block):
        r = rows[a:a + block]
        out[:, a:a + len(r)] = np.asarray(rq_raw[r[0]:r[-1] + 1:stride])[:, rq_reps].T
    return out

RQ_X = rq_series()
RQ_G, RQ_N = RQ_X.shape
RQ_RATE = RQ_X.mean(axis=1)
assert RQ_TE.shape == (RQ_G, RQ_G) and len(RQ_K) == RQ_G

# Netlist: directed terminal-to-gate references (supply nodes excluded), split by pull-down vs pass.
VSS, VCC = 558, 657
rq_defs = []
for line in (ROOT / 'data/visual6502/transdefs.js').read_text().splitlines():
    if line.strip().startswith("['t"):
        rq_defs.append(ast.literal_eval(line.strip().rstrip(',')))
rq_defs.sort(key=lambda d: int(d[0][1:]))
RQ_M = len(rq_defs)
assert RQ_M == len(rq_p2g), 'Netlist and recording disagree on transistor count.'
rq_gate, rq_c1, rq_c2 = (np.asarray([d[i] for d in rq_defs]) for i in (1, 2, 3))
rq_box = np.asarray([d[4] for d in rq_defs], float)
RQ_CENTRE = np.column_stack(((rq_box[:, 0] + rq_box[:, 1]) / 2, (rq_box[:, 2] + rq_box[:, 3]) / 2))
_gate_lookup = {}
for t_, g_ in enumerate(rq_gate):
    _gate_lookup.setdefault(int(g_), []).append(t_)
_src, _dst, _pd = [], [], []
for s_ in range(RQ_M):
    dests = set()
    for term in (rq_c1[s_], rq_c2[s_]):
        if term not in (VSS, VCC):
            dests.update(_gate_lookup.get(int(term), []))
    dests.discard(s_)
    for t_ in dests:
        _src.append(s_); _dst.append(t_); _pd.append(VSS in (rq_c1[s_], rq_c2[s_]))
_src, _dst, _pd = map(np.asarray, (_src, _dst, _pd))
def _csr(mask=None):
    s, d = (_src, _dst) if mask is None else (_src[mask], _dst[mask])
    return sparse.csr_matrix((np.ones(len(s)), (s, d)), shape=(RQ_M, RQ_M))
RQ_A = _csr(); RQ_A_PD = _csr(_pd); RQ_A_PASS = _csr(~_pd)
_active = np.flatnonzero(rq_p2g >= 0)
RQ_P = sparse.csr_matrix((np.ones(len(_active)), (_active, rq_p2g[_active])), shape=(RQ_M, RQ_G))
def rq_agg(B):
    C = np.asarray((RQ_P.T @ sparse.csr_matrix(B) @ RQ_P).todense()); np.fill_diagonal(C, 0); return C

RQ_POS = rq_agg(RQ_A); RQ_POS_PD = rq_agg(RQ_A_PD); RQ_POS_PASS = rq_agg(RQ_A_PASS)
RQ_SIZE = np.bincount(rq_p2g[_active], minlength=RQ_G).astype(float)
RQ_PAIRS = np.outer(RQ_SIZE, RQ_SIZE); np.fill_diagonal(RQ_PAIRS, 0)
RQ_NEG = RQ_PAIRS - RQ_POS
assert (RQ_NEG >= 0).all()
# Indirect relations among physical non-edges (both directions of direct wiring excluded).
_Ad = (RQ_A.toarray() > 0); _I = np.eye(RQ_M, dtype=bool)
_none = ~_Ad & ~_Ad.T & ~_I
_rel = {'reverse_only': _Ad.T & ~_Ad & ~_I,
        'two_step': ((RQ_A @ RQ_A).toarray() > 0) & ~_Ad & ~_I,
        'shared_parent': ((RQ_A.T @ RQ_A).toarray() > 0) & _none,
        'shared_child': ((RQ_A @ RQ_A.T).toarray() > 0) & _none}
RQ_REL = {k: rq_agg(v.astype(float)) for k, v in _rel.items()}
# Mutually exclusive relation classes (precedence: reverse, two-step, shared parent, shared child, unrelated).
_taken = _Ad | _I; RQ_REL_X = {}
for _name in ['reverse_only', 'two_step', 'shared_parent', 'shared_child']:
    _m = _rel[_name] & ~_taken; RQ_REL_X[_name] = rq_agg(_m.astype(float)); _taken |= _m
RQ_REL_X['unrelated'] = np.maximum(RQ_NEG - sum(RQ_REL_X.values()), 0)
del _Ad, _I, _none, _rel, _taken
legacy_counts = OUT / 'analysis_pairwise/structural_counts.npz'
if legacy_counts.exists():
    with np.load(legacy_counts) as z:
        assert np.array_equal(z['positive_count'], RQ_POS), 'Structural reference differs from Section 3.'

RQ_EL = ~np.eye(RQ_G, dtype=bool)
RQ_COL = np.broadcast_to(np.arange(RQ_G)[None, :], (RQ_G, RQ_G))[RQ_EL]   # target of each item
def rq_weights(level):
    if level == 'physical':
        return RQ_POS[RQ_EL].astype(float), RQ_NEG[RQ_EL].astype(float)
    return (RQ_POS[RQ_EL] > 0).astype(float), (RQ_POS[RQ_EL] == 0).astype(float)

class Ranked:
    """Pre-sorted score for fast cluster-bootstrap metrics (weights change, order does not)."""
    def __init__(self, score_matrix, level, mask=None):
        s = np.asarray(score_matrix, float)[RQ_EL]; p, n = rq_weights(level); c = RQ_COL
        keep = np.isfinite(s) & ((p + n) > 0) & (True if mask is None else mask[RQ_EL])
        order = np.argsort(-s[keep], kind='stable')
        self.s, self.p, self.n, self.c = s[keep][order], p[keep][order], n[keep][order], c[keep][order]
        self.ends = np.r_[np.flatnonzero(np.diff(self.s) != 0), len(self.s) - 1]
    def metrics(self, mult=None, budgets=()):
        w = np.ones(len(self.c)) if mult is None else mult[self.c]
        p, n = self.p * w, self.n * w
        tp = np.cumsum(p)[self.ends]; fp = np.cumsum(n)[self.ends]; items = np.cumsum(w)[self.ends]
        P, Nn = tp[-1], fp[-1]
        if P <= 0 or Nn <= 0: raise ValueError('degenerate draw')
        tpr = np.r_[0, tp / P]; fpr = np.r_[0, fp / Nn]
        prec = tp / np.maximum(tp + fp, 1e-300)
        out = dict(AUC=tk._trapz(tpr, fpr), AP=float(np.sum(np.diff(np.r_[0, tp / P]) * prec)), prevalence=P / (P + Nn))
        out['AP_lift'] = out['AP'] / out['prevalence']
        for b in budgets:
            i = int(np.searchsorted(items, b))
            if i >= len(items): i = len(items) - 1
            prev_items = items[i - 1] if i > 0 else 0.0; prev_tp = tp[i - 1] if i > 0 else 0.0; prev_fp = fp[i - 1] if i > 0 else 0.0
            frac = (b - prev_items) / max(items[i] - prev_items, 1e-300); frac = min(max(frac, 0), 1)
            t_ = prev_tp + frac * (tp[i] - prev_tp); f_ = prev_fp + frac * (fp[i] - prev_fp)
            out[f'P@{b}'] = t_ / (t_ + f_) if t_ + f_ > 0 else np.nan
            out[f'lift@{b}'] = out[f'P@{b}'] / out['prevalence']
        return out
    def curves(self):
        tp = np.cumsum(self.p)[self.ends]; fp = np.cumsum(self.n)[self.ends]
        return dict(fpr=np.r_[0, fp / fp[-1]], tpr=np.r_[0, tp / tp[-1]], recall=tp / tp[-1], precision=tp / (tp + fp))

coverage = dict(transistors=RQ_M, active_transistors=int(len(_active)), activity_groups=RQ_G,
                retained_samples=RQ_N, structural_edges_total=int(len(_src)),
                structural_edges_evaluable=int(RQ_POS.sum()),
                evaluable_fraction=float(RQ_POS.sum() / len(_src)),
                pulldown_edges_evaluable=int(RQ_POS_PD.sum()), pass_edges_evaluable=int(RQ_POS_PASS.sum()),
                positive_group_pairs=int((RQ_POS > 0).sum()), group_pairs=int(RQ_EL.sum()),
                physical_pairs=float(RQ_PAIRS.sum()))
_gs, _gd = rq_p2g[_src], rq_p2g[_dst]
coverage['structural_edges_inactive_endpoint'] = int(((_gs < 0) | (_gd < 0)).sum())
coverage['structural_edges_within_identical_group'] = int(((_gs >= 0) & (_gs == _gd)).sum())
rq_record('coverage', coverage)
display(pd.DataFrame([coverage]).T.rename(columns={0: 'value'}))
fig, ax = plt.subplots(figsize=(6.5, 2.4), constrained_layout=True)
_parts = [('Evaluable (between activity groups)', coverage['structural_edges_evaluable']),
          ('Inside identical-activity groups', coverage['structural_edges_within_identical_group']),
          ('Inactive endpoint', coverage['structural_edges_inactive_endpoint'])]
_left = 0
for (_lab, _v), _c in zip(_parts, ['C0', 'C1', 'C7']):
    ax.barh([0], [_v], left=_left, color=_c, label=f'{_lab}: {_v:,} ({_v / coverage["structural_edges_total"]:.1%})'); _left += _v
ax.set(yticks=[], xlabel='Directed terminal-to-gate connections', title='What the recording can observe'); ax.legend(fontsize=7, loc='upper center', bbox_to_anchor=(.5, -.45), ncol=1)
rq_fig(fig, 'fig05_coverage', 'Observability ceiling: share of the 6502 terminal-to-gate connections that can be evaluated from the recording.')

## 6. Estimator validation: JIDT units audit and empirical calibration of the analytical null

**Why this matters.** Plug-in TE on discrete data has a known finite-sample bias and, under independence, $2N\ln 2\,\widehat{TE}_{\text{bits}}\sim\chi^2_{df}$ asymptotically. If an implementation forms the statistic from a bit-valued estimate as though it were in nats, the statistic shrinks by $\ln 2$, the null mean is understated by the same factor and every p-value is anti-conservative.

**What this section does.** (i) Documents the convention of the loaded JIDT jar against the bit-correct null. (ii) Calibrates the bit-correct analytical null on the real recording: for stratified targets, the target series is circularly shifted relative to all sources (destroying source–target coupling while preserving every autocorrelation) and the false-positive rate at α = 0.05 is measured for the bit-correct and JIDT conventions. Residual excess over 0.05 under the bit-correct null quantifies the effect of temporal dependence and shared periodic drive, which the asymptotic null ignores.

In [ ]:
audit = tk.jidt_units_audit(jpype)
audit_df = pd.DataFrame(audit['rows'])
rq_table(audit_df[['measure', 'k', 'n', 'df', 'value_bits', 'jidt_p', 'p_bit_correct', 'p_if_bits_treated_as_nats',
                   'jidt_null_mean', 'null_mean_bits', 'null_mean_nats']],
         'jidt_units_audit', 'JIDT discrete analytic null versus the bit-correct chi-square null (synthetic weakly coupled pair).')
rq_record('jidt_units_audit', {k: v for k, v in audit.items() if k != 'rows'})
print('JIDT convention detected:', audit['convention'])

# Empirical null calibration on the real recording.
rng = np.random.default_rng(RQ['seed'])
eligible_t = np.flatnonzero((RQ_RATE > 0.001) & (RQ_RATE < 0.999))
strata = pd.qcut(RQ_K[eligible_t] + 1e-6 * rng.random(len(eligible_t)), 4, labels=False, duplicates='drop')
chosen = []
for s_ in np.unique(strata):
    pool = eligible_t[strata == s_]
    chosen += list(rng.choice(pool, size=min(len(pool), max(1, RQ['null_targets'] // 4)), replace=False))
chosen = np.asarray(sorted(chosen))
Xsp0 = tk.as_sparse_sources(RQ_X)
null_rows = []
started = time.perf_counter()
for i, t in enumerate(chosen):
    k = int(RQ_K[t]); src = np.arange(RQ_G) != t
    shifts = rng.integers(RQ['null_min_shift'], RQ_N - RQ['null_min_shift'], RQ['null_shifts'])
    for sh in shifts:
        te_null, n_ = tk.te_all_sources(Xsp0, np.roll(RQ_X[t], int(sh)), k)
        te_null = te_null[src]; df = tk.df_te(k)
        p_bits = tk.p_from_bits(te_null, n_, df)
        p_jidt = tk.chi2.sf(2 * n_ * te_null, df)
        null_rows.append(dict(target=int(t), k=k, shift=int(sh), n=n_, mean_null_te=float(te_null.mean()),
                              analytic_mean_bits=tk.null_mean_bits(df, n_), analytic_mean_nats=df / (2 * n_),
                              fpr_bits=float(np.mean(p_bits <= RQ['alpha'])), fpr_jidt=float(np.mean(p_jidt <= RQ['alpha']))))
    print(f'  null calibration target {i + 1}/{len(chosen)} ({time.perf_counter() - started:.0f}s)', flush=True)
null_df = pd.DataFrame(null_rows)
null_df.to_csv(RQ_TAB / 'null_calibration_draws.csv', index=False)
per_target = null_df.groupby(['target', 'k']).agg(mean_null_te=('mean_null_te', 'mean'), analytic_bits=('analytic_mean_bits', 'mean'),
                                                  analytic_nats=('analytic_mean_nats', 'mean'), fpr_bits=('fpr_bits', 'mean'),
                                                  fpr_jidt=('fpr_jidt', 'mean')).reset_index()
per_target['ratio_empirical_to_bits'] = per_target.mean_null_te / per_target.analytic_bits
rq_table(per_target, 'null_calibration_by_target',
         'Circular-shift null on the recording: empirical mean null TE, bit-correct and nats-treated analytical means, and false-positive rates at alpha=0.05.')
summary = dict(targets=len(chosen), shifts_per_target=RQ['null_shifts'],
               median_ratio_empirical_to_bit_null=float(per_target.ratio_empirical_to_bits.median()),
               mean_fpr_bit_correct=float(null_df.fpr_bits.mean()), mean_fpr_jidt_convention=float(null_df.fpr_jidt.mean()))
rq_record('null_calibration', summary); print(summary)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
lim = [0, per_target[['analytic_bits', 'mean_null_te']].to_numpy().max() * 1.1]
ax[0].scatter(per_target.analytic_bits, per_target.mean_null_te, s=14, label='bit-correct null mean')
ax[0].scatter(per_target.analytic_nats, per_target.mean_null_te, s=14, marker='x', label='nats-treated (JIDT)')
ax[0].plot(lim, lim, '--', c='grey', lw=1); ax[0].set(xlim=lim, ylim=lim, xlabel='Analytical null mean (bits)',
           ylabel='Empirical circular-shift null mean (bits)', title='(a) Null mean'); ax[0].legend()
ax[1].boxplot([null_df.fpr_bits, null_df.fpr_jidt]); ax[1].set_xticks([1, 2], ['bit-correct', 'JIDT convention'])
ax[1].axhline(RQ['alpha'], ls='--', c='grey', lw=1); ax[1].set(ylabel='False-positive rate at α = 0.05', title='(b) Null calibration')
rq_fig(fig, 'fig06_null_calibration', 'Calibration of the analytical TE null on the 6502 recording using circular target shifts.')

## 7. Synthetic ground-truth validation: clocked noisy logic circuits

Following the validation-first design of network-inference studies (Novelli et al. 2019; Novelli & Lizier 2021), each replicate is a random logic circuit (inputs with persistence, a period-2 clock, BUF/NOT/AND/OR/NAND/NOR/XOR/MAJ3 gates with one-step delay, sparse feedback and 2% bit-flip noise) whose parents are known exactly. Each circuit is recorded in two representations:

* **state** — the node value at each step (what TE theory assumes);
* **rising** — 0→1 event indicators, the representation of the 6502 recording.

Methods compared on identical data: pairwise TE (exact plug-in, AIS-selected history, bit-correct χ² null, BH over all pairs); pairwise TE at an oracle budget equal to the true edge count; IDTxl bivariate TE (link-wise, link-level FDR; see 9.4) and IDTxl multivariate TE (network-level FDR over omnibus tests), both with the bit-correct analytic null; and, on one replicate per condition, IDTxl multivariate TE with the uncorrected analytic null. Metrics: precision, recall, MCC, false-positive anatomy (reverse, two-step, shared parent, shared child, unrelated), recall by edge class (inverting gate, synergistic XOR, clock source), and the sign of the source effect.

In [ ]:
import multiprocessing as mp, warnings
warnings.filterwarnings('ignore', message='invalid value encountered in cast')
from concurrent.futures import ProcessPoolExecutor, as_completed
SYN_DIR = RQ_DIR / 'synthetic'; SYN_DIR.mkdir(exist_ok=True)
try:
    import idtxl  # noqa: F401
    IDTXL_OK = True
except ImportError:
    IDTXL_OK = False
    print('IDTxl not importable: synthetic IDTxl arms and Section 9 are skipped.\n'
          'Install with: pip install "git+https://github.com/pwollstadt/IDTxl.git" statsmodels h5py networkx')

syn_tasks = []
for rep in range(RQ['syn_reps']):
    for T in RQ['syn_T']:
        for representation in ['state', 'rising']:
            run_idtxl = IDTXL_OK and rep < RQ['syn_idtxl_reps'] and T <= RQ['syn_idtxl_T_max']
            methods = [('multivariate', True), ('bivariate', True)] + ([('multivariate', False)] if rep == 0 else [])
            task = dict(seed=1000 + rep, T=int(T), representation=representation, n_nodes=RQ['syn_n_nodes'],
                        n_inputs=RQ['syn_inputs'], noise=RQ['syn_noise'], k_max=RQ['syn_k_max'], alpha=RQ['alpha'],
                        run_idtxl=run_idtxl, idtxl_methods=methods if run_idtxl else [],
                        idtxl_max_lag_target=RQ['syn_idtxl_max_lag_target'], max_heap='1g')
            key = hashlib.sha256(json.dumps(task, sort_keys=True).encode()).hexdigest()[:16]
            syn_tasks.append((key, task))

def syn_run(tasks):
    pending = [(k, t) for k, t in tasks if not (SYN_DIR / f'{k}.json').exists()]
    print(f'Synthetic tasks: {len(tasks)} total, {len(pending)} to run', flush=True)
    def save(k, result):
        tmp = SYN_DIR / f'{k}.tmp'; tmp.write_text(json.dumps(rq_json(result))); tmp.replace(SYN_DIR / f'{k}.json')
    if RQ['workers'] <= 1:
        for i, (k, t) in enumerate(pending, 1):
            save(k, tk.synthetic_task(t)); print(f'  {i}/{len(pending)}', flush=True)
        return
    with ProcessPoolExecutor(max_workers=RQ['workers'], mp_context=mp.get_context('spawn')) as pool:
        futures = {pool.submit(tk.synthetic_task, t): k for k, t in pending}
        for i, f in enumerate(as_completed(futures), 1):
            try:
                save(futures[f], f.result())
            except Exception as error:
                print('  task failed:', futures[f], repr(error))
            if i % 5 == 0 or i == len(pending):
                print(f'  {i}/{len(pending)}', flush=True)

syn_started = time.perf_counter(); syn_run(syn_tasks)
syn_rows, syn_edges = [], []
for k, _ in syn_tasks:
    path = SYN_DIR / f'{k}.json'
    if path.exists():
        r = json.loads(path.read_text()); syn_rows += r['rows']; syn_edges += r['edges']
SYN = pd.DataFrame(syn_rows); SYN_E = pd.DataFrame(syn_edges)
SYN.to_csv(RQ_TAB / 'synthetic_all_runs.csv', index=False); SYN_E.to_csv(RQ_TAB / 'synthetic_true_edges.csv', index=False)
print(f'Synthetic benchmark: {len(SYN)} method-runs in {time.perf_counter() - syn_started:.0f}s')

In [ ]:
def mean_ci(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) == 0: return (np.nan, np.nan, np.nan)
    if len(v) == 1: return (v[0], np.nan, np.nan)
    boots = np.random.default_rng(1).choice(v, (2000, len(v))).mean(axis=1)
    return (v.mean(), np.quantile(boots, .025), np.quantile(boots, .975))

agg_rows = []
for (method, rep_, T), g in SYN.groupby(['method', 'representation', 'T']):
    row = dict(method=method, representation=rep_, T=T, replicates=len(g))
    for metric in ['precision', 'recall', 'MCC', 'FPR']:
        m_, lo, hi = mean_ci(g[metric]); row[metric] = rq_ci(m_, lo, hi)
        row[f'_{metric}'] = m_; row[f'_{metric}_lo'] = lo; row[f'_{metric}_hi'] = hi
    fp = g[['fp_reverse_only', 'fp_two_step', 'fp_shared_parent', 'fp_shared_child', 'fp_unrelated']].sum()
    tot = max(fp.sum(), 1)
    row['FP share indirect (two-step + shared parent)'] = f"{(fp['fp_two_step'] + fp['fp_shared_parent']) / tot:.2f}"
    agg_rows.append(row)
SYN_AGG = pd.DataFrame(agg_rows)
rq_table(SYN_AGG[[c for c in SYN_AGG.columns if not c.startswith('_')]], 'synthetic_summary',
         'Synthetic logic circuits: mean [95% bootstrap CI] across replicates, by method, representation and sample length.')

methods_order = [m for m in ['Pairwise TE (bit-correct, BH)', 'Pairwise TE (top-|E| budget)', 'IDTxl bivariate TE',
                             'IDTxl multivariate TE', 'IDTxl multivariate TE (uncorrected analytic null)'] if m in set(SYN.method)]
fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharex=True, constrained_layout=True)
for r_, rep_ in enumerate(['state', 'rising']):
    for c_, metric in enumerate(['precision', 'recall', 'MCC']):
        ax = axes[r_, c_]
        for m in methods_order:
            d = SYN_AGG[(SYN_AGG.method == m) & (SYN_AGG.representation == rep_)].sort_values('T')
            if d.empty: continue
            ax.plot(d['T'], d[f'_{metric}'], marker='o', ms=3, label=m)
            ax.fill_between(d['T'], d[f'_{metric}_lo'], d[f'_{metric}_hi'], alpha=.15)
        ax.set(xscale='log', ylim=(-0.05, 1.05), title=f'{metric} — {rep_} representation')
        if r_ == 1: ax.set_xlabel('Samples T')
axes[0, 0].legend(fontsize=6)
rq_fig(fig, 'fig07_synthetic_convergence', 'Synthetic validation: precision, recall and MCC versus sample length for each method and representation (mean and 95% CI).')

if not SYN_E.empty:
    SYN_E['edge_class'] = np.select([SYN_E.clock_source, SYN_E.gate.eq('XOR'), SYN_E.inverting], ['clock source', 'XOR (synergistic)', 'inverting gate'], 'non-inverting gate')
    cols = [c for c in ['pairwise_detected', 'idtxl_bivariate_detected', 'idtxl_multivariate_detected'] if c in SYN_E]
    T_ref = SYN_E[[c for c in cols if c != 'pairwise_detected']].dropna().index if len(cols) > 1 else SYN_E.index
    E = SYN_E.loc[T_ref] if len(cols) > 1 else SYN_E
    cls = E.groupby(['representation', 'edge_class'])[cols].mean().reset_index()
    sign = SYN_E.groupby(['representation', 'inverting']).agg(edges=('delta', 'size'), fraction_negative_effect=('delta', lambda d: float((d < 0).mean()))).reset_index()
    rq_table(cls, 'synthetic_recall_by_edge_class', 'Synthetic recall by edge class (edges from runs where all listed methods were applied).')
    rq_table(sign, 'synthetic_effect_sign', 'Sign of the source effect (delta) on true edges: inverting versus non-inverting gates, by representation.')
    rq_record('synthetic_effect_sign', sign.to_dict('records'))
best_T = max(RQ['syn_T']); rq_record('synthetic_summary_at_largest_T', SYN_AGG[SYN_AGG['T'] == SYN_AGG['T'].max()][['method', 'representation', '_precision', '_recall', '_MCC']].to_dict('records'))
fix_cmp = SYN[SYN.method.str.startswith('IDTxl multivariate')].groupby(['method', 'representation', 'T'])[['TP', 'FP']].mean().reset_index()
if not fix_cmp.empty:
    rq_table(fix_cmp, 'synthetic_idtxl_units_effect', 'IDTxl multivariate TE with bit-correct versus uncorrected analytic surrogates (mean TP and FP per circuit).')

# Part III — RQ1: pairwise transfer entropy versus the known circuit

Primary analysis (H1, H2): Section 8.1. Pre-specified secondary analyses: 8.3 (H3) and 8.4 (H4). Sections 8.2 and 8.5–8.9 are robustness and exploratory analyses.

## 8.1 RQ1 — Pairwise TE versus the netlist, with uncertainty

Scores are evaluated against the terminal-to-gate reference at two levels: **physical** (each group pair weighted by the physical connected/unconnected transistor pairs it represents) and **group** (one label per group pair; positive if any physical edge exists). Uncertainty is a **target-cluster bootstrap**: whole target groups are resampled with replacement, keeping every incoming pair together, with identical draws for all scores so that differences are paired. Intervals therefore describe sensitivity to which targets the chip contains; they are not intervals for repeated recordings, which this single workload cannot provide.

First, the exact estimator is checked against the saved JIDT matrix (it must agree to numerical precision), and the target conditional entropy and signed effects used later are computed.

In [ ]:
t0 = time.perf_counter()
RQ_TE_EXACT, _nobs, RQ_DELTA, RQ_HCOND = tk.te_matrix(RQ_X, RQ_K, details=True, progress=max(1, RQ_G // 5))
diff = np.nanmax(np.abs(RQ_TE_EXACT - RQ_TE))
rq_record('estimator_agreement_with_jidt', dict(max_abs_difference_bits=float(diff), seconds=time.perf_counter() - t0))
print(f'Exact estimator vs JIDT: max |difference| = {diff:.2e} bits ({time.perf_counter() - t0:.0f}s)')
assert diff < 1e-9, 'Fast estimator disagrees with JIDT; do not proceed.'
assert np.array_equal(_nobs, RQ_NOBS)
np.savez_compressed(RQ_DIR / 'signed_effect.npz', delta=RQ_DELTA, target_conditional_entropy=RQ_HCOND, representatives=rq_reps)
# Contemporaneous conditional MI ('lag-0 TE'): I(y_t; x_t | y_{t-k..t-1}).
RQ_TE0, RQ_NOBS0 = tk.te_matrix(RQ_X, RQ_K, delay=0)
RQ_LOGP0 = np.full_like(RQ_TE0, np.nan)
for _t in range(RQ_G):
    RQ_LOGP0[:, _t] = tk.log_p_from_bits(RQ_TE0[:, _t], RQ_NOBS0[_t], tk.df_te(RQ_K[_t]))
RQ_LOGP0[~RQ_EL] = np.nan
np.savez_compressed(RQ_DIR / 'te_lag0.npz', te_bits=RQ_TE0, log_p=RQ_LOGP0, representatives=rq_reps)

null_cols = np.asarray([tk.null_mean_bits(tk.df_te(k), n) for k, n in zip(RQ_K, RQ_NOBS)])
SCORES = {'Pairwise TE': RQ_TE,
          'Bias-corrected TE': RQ_TE - null_cols[None, :],
          '−log10 p (bit-correct)': -RQ_LOGP / np.log(10),
          'Pairwise TE, lag 0': RQ_TE0,
          'Zero-lag MI': RQ_MI}
budgets = RQ['budgets_items']
point_rows, boot = [], {}
for level in ['physical', 'group']:
    ranked = {name: Ranked(S, level) for name, S in SCORES.items()}
    for name, r in ranked.items():
        point_rows.append(dict(level=level, score=name, **r.metrics(budgets=budgets)))
    def stat(mult, ranked=ranked):
        out = {}
        for name, r in ranked.items():
            m = r.metrics(mult, budgets=budgets)
            for key in ['AUC', 'AP_lift'] + [f'lift@{b}' for b in budgets]:
                out[f'{name}|{key}'] = m[key]
        for key in ['AUC', 'AP_lift']:
            out[f'TE−MI|{key}'] = out[f'Pairwise TE|{key}'] - out[f'Zero-lag MI|{key}']
            out[f'p−TE|{key}'] = out[f'−log10 p (bit-correct)|{key}'] - out[f'Pairwise TE|{key}']
            out[f'lag0−lag1|{key}'] = out[f'Pairwise TE, lag 0|{key}'] - out[f'Pairwise TE|{key}']
        return out
    boot[level] = tk.bootstrap_targets(stat, RQ_G, RQ['bootstrap_draws'], RQ['seed'])
POINT = pd.DataFrame(point_rows)

table_rows = []
for _, r in POINT.iterrows():
    row = dict(level=r.level, score=r.score)
    for key in ['AUC', 'AP_lift'] + [f'lift@{b}' for b in budgets]:
        lo, hi, _ = tk.interval(boot[r.level], f'{r.score}|{key}')
        row[key] = rq_ci(r[key], lo, hi, 3 if key == 'AUC' else 2)
    table_rows.append(row)
rq_table(pd.DataFrame(table_rows), 'rq1_discrimination',
         'Pairwise TE (lag 1), bias-corrected TE, bit-correct significance, contemporaneous (lag-0) TE and zero-lag MI versus the netlist: ROC-AUC, average-precision lift and precision lift at fixed group-pair budgets, with 95% target-cluster bootstrap intervals.')
diff_rows = []
for level in ['physical', 'group']:
    for name in ['TE−MI', 'p−TE', 'lag0−lag1']:
        for key in ['AUC', 'AP_lift']:
            v = [d[f'{name}|{key}'] for d in boot[level] if np.isfinite(d[f'{name}|{key}'])]
            diff_rows.append(dict(level=level, contrast=name, metric=key, median=np.median(v),
                                  ci_low=np.quantile(v, .025), ci_high=np.quantile(v, .975),
                                  fraction_draws_positive=np.mean(np.asarray(v) > 0)))
rq_table(pd.DataFrame(diff_rows), 'rq1_paired_contrasts', 'Paired target-bootstrap contrasts between scores.')
rq_record('rq1_point_metrics', POINT.to_dict('records')); rq_record('rq1_paired_contrasts', diff_rows)

fig, axes = plt.subplots(2, 2, figsize=(9, 7), constrained_layout=True)
for i, level in enumerate(['physical', 'group']):
    for name, S in SCORES.items():
        c = Ranked(S, level).curves()
        axes[i, 0].plot(c['fpr'], c['tpr'], lw=1, label=name)
        axes[i, 1].plot(c['recall'], c['precision'], lw=1, label=name)
    prev = POINT[(POINT.level == level)].prevalence.iloc[0]
    axes[i, 0].plot([0, 1], [0, 1], ':', c='grey'); axes[i, 1].axhline(prev, ls=':', c='grey', label='prevalence')
    axes[i, 0].set(xlabel='False-positive rate', ylabel='True-positive rate', title=f'ROC — {level} level')
    axes[i, 1].set(xlabel='Recall', ylabel='Precision', yscale='log', title=f'Precision–recall — {level} level')
axes[0, 1].legend(fontsize=7)
rq_fig(fig, 'fig08_1_roc_pr', 'ROC and precision–recall curves for pairwise TE, bias-corrected TE, bit-correct significance and zero-lag MI against the netlist.')

In [ ]:
# Score distributions for wired and unwired pairs, and TE by structural relation.
el = RQ_EL
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
for ax, (lab, S) in zip(axes, [('Pairwise TE, lag 1', RQ_TE), ('Pairwise TE, lag 0', RQ_TE0)]):
    for name, W, c in [('wired', RQ_POS, 'C3'), ('unwired', RQ_NEG, 'C0')]:
        v = S[el]; w = W[el].astype(float); ok = (w > 0) & np.isfinite(v)
        o = np.argsort(v[ok]); x = v[ok][o]; cw = np.cumsum(w[ok][o]) / w[ok].sum()
        zero = cw[np.searchsorted(x, 0, side='right') - 1] if (x <= 0).any() else 0
        ax.step(np.maximum(x, 1e-7), cw, where='post', c=c, label=f'{name} (exactly zero: {zero:.1%})')
    ax.set(xscale='log', xlabel=f'{lab} (bits)', ylabel='Cumulative fraction of physical pairs', title=lab); ax.legend(fontsize=7)
rq_fig(fig, 'fig08_1_score_distributions', 'Physical-pair-weighted cumulative distributions of lag-1 and lag-0 pairwise TE for wired and unwired transistor pairs.')
bh1 = np.zeros_like(el); bh1[el] = tk.bh_adjust_logp(RQ_LOGP[el]) <= np.log(RQ['alpha'])
rows = []
for name, W in [('direct (wired)', RQ_POS)] + [(k.replace('_', ' '), v) for k, v in RQ_REL_X.items()]:
    w = W[el].astype(float)
    if w.sum() == 0: continue
    rows.append(dict(relation=name, physical_pairs=w.sum(), mean_TE_lag1=float((w * RQ_TE[el]).sum() / w.sum()),
                     mean_TE_lag0=float((w * RQ_TE0[el]).sum() / w.sum()), mean_MI=float((w * RQ_MI[el]).sum() / w.sum()),
                     fraction_BH_significant=float((w * bh1[el]).sum() / w.sum())))
REL = pd.DataFrame(rows)
rq_table(REL, 'rq1_te_by_relation', 'Mean information by structural relation of the transistor pair (physical-pair weighted; mutually exclusive classes).')
rq_record('rq1_te_by_relation', REL.to_dict('records'))

## 8.2 RQ1 — Decision rules and the practical impact of the units correction

Binary networks from significance rules are compared using the bit-correct p-values and, for contrast, the p-values JIDT produced with its nats-treated null. BH and BY are applied to log-p values, so no test underflows. A significant pairwise TE is evidence against the estimator null, not evidence of a direct wire; the comparison shows how much of the "significant" network is structural.

In [ ]:
el = RQ_EL
logp = RQ_LOGP[el]; m_tests = el.sum()
jidt_logp = np.log(np.clip(RQ_JIDT_P[el], 1e-300, 1))
rules = {
    'Raw p ≤ α (bit-correct)': logp <= np.log(RQ['alpha']),
    'BH q ≤ α (bit-correct)': tk.bh_adjust_logp(logp) <= np.log(RQ['alpha']),
    'BY q ≤ α (bit-correct)': tk.bh_adjust_logp(logp, 'BY') <= np.log(RQ['alpha']),
    'Bonferroni (bit-correct)': logp <= np.log(RQ['alpha'] / m_tests),
    'BH q ≤ α (JIDT nats-treated p)': tk.bh_adjust_logp(jidt_logp) <= np.log(RQ['alpha']),
}
pos_p, neg_p = rq_weights('physical'); pos_g, neg_g = rq_weights('group')
rows = []
for name, pred in rules.items():
    for level, (p, n) in [('physical', (pos_p, neg_p)), ('group', (pos_g, neg_g))]:
        m = tk.binary_metrics(pred, p, n)
        rows.append(dict(rule=name, level=level, significant_group_pairs=int(pred.sum()),
                         fraction_significant=pred.mean(), precision=m['precision'], recall=m['recall'],
                         precision_lift=m['precision_lift'], MCC=m['MCC']))
DEC = pd.DataFrame(rows)
rq_table(DEC, 'rq1_decision_rules', 'Pairwise-TE significance networks against the netlist (bit-correct versus JIDT nats-treated p-values).')
rq_record('rq1_decision_rules', DEC.to_dict('records'))
fig, ax = plt.subplots(figsize=(5.5, 3.4), constrained_layout=True)
bins = np.linspace(0, max(1.0, min(320, np.nanpercentile(np.r_[-logp, -jidt_logp] / np.log(10), 99.9))), 80)
ax.hist(-logp / np.log(10), bins=bins, alpha=.6, label='bit-correct', log=True)
ax.hist(-jidt_logp / np.log(10), bins=bins, alpha=.6, label='JIDT nats-treated', log=True)
ax.axvline(-np.log10(RQ['alpha'] / m_tests), c='k', ls='--', lw=1, label='Bonferroni')
ax.set(xlabel='−log10 p', ylabel='Group pairs'); ax.legend()
rq_fig(fig, 'fig08_2_pvalue_distributions', 'Distribution of pairwise-TE p-values under the bit-correct and JIDT nats-treated nulls.')

## 8.3 RQ1 — Does TE identify the direction of signal flow? Matched histories and nuisance baselines

Restricted to group pairs wired in one direction only (weighted by physical edges), a pair is oriented correctly when $TE(i\to j) > TE(j\to i)$. Saved TE uses target-specific histories, so asymmetry can arise from different $k$ alone; therefore TE is recomputed with a **common history $k$ in both directions**. Direction is also predicted from nuisance properties alone (event rate; conditional entropy of the target) and a grouped cross-validated logistic model tests whether TE asymmetry adds information beyond them.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

uni = (RQ_POS > 0) & (RQ_POS.T == 0) & RQ_EL
I_, J_ = np.nonzero(uni); W_ = RQ_POS[I_, J_].astype(float)
dir_scores = {'saved target-specific k': RQ_TE_EXACT}
for k in RQ['direction_common_k']:
    t0 = time.perf_counter(); dir_scores[f'common k = {k}'] = tk.te_matrix(RQ_X, np.full(RQ_G, k))[0]
    print(f'  common-k TE k={k} in {time.perf_counter() - t0:.0f}s', flush=True)
hcond_common = {k: np.asarray([tk.conditional_entropy_bits(RQ_X[g], k) for g in range(RQ_G)]) for k in [1, 4]}
nuisance = {'higher event rate is target': RQ_RATE, 'higher H(Y|Y^(4)) is target': hcond_common[4],
            'higher H(Y|Y^(k)) is target (saved k)': RQ_HCOND}

def oriented_accuracy(fwd, rev, w, mult=None):
    ww = w if mult is None else w * mult
    correct = (fwd > rev); tie = np.isclose(fwd, rev, rtol=0, atol=1e-15)
    return float((ww * (correct & ~tie)).sum() / max((ww * ~tie).sum(), 1e-300)), float((ww * tie).sum() / ww.sum())

rows = []
for name, S in dir_scores.items():
    acc, ties = oriented_accuracy(S[I_, J_], S[J_, I_], W_)
    draws = tk.bootstrap_targets(lambda m: dict(a=oriented_accuracy(S[I_, J_], S[J_, I_], W_, m[J_])[0]), RQ_G, RQ['bootstrap_draws'], RQ['seed'])
    lo, hi, _ = tk.interval(draws, 'a')
    rows.append(dict(predictor=f'TE asymmetry, {name}', accuracy=rq_ci(acc, lo, hi), tie_fraction=ties, _acc=acc))
for name, v in nuisance.items():
    acc, ties = oriented_accuracy(v[J_], v[I_], W_)
    draws = tk.bootstrap_targets(lambda m: dict(a=oriented_accuracy(v[J_], v[I_], W_, m[J_])[0]), RQ_G, RQ['bootstrap_draws'], RQ['seed'])
    lo, hi, _ = tk.interval(draws, 'a')
    rows.append(dict(predictor=f'Nuisance: {name}', accuracy=rq_ci(acc, lo, hi), tie_fraction=ties, _acc=acc))
DIR = pd.DataFrame(rows)
rq_table(DIR.drop(columns='_acc'), 'rq1_direction',
         'Direction of signal flow on unidirectionally wired group pairs (physical-edge weighted): TE asymmetry under saved and common histories versus nuisance predictors, with 95% target-cluster bootstrap intervals.')

# Incremental value of TE asymmetry over nuisance asymmetries (grouped 5-fold CV; random orientation).
rng = np.random.default_rng(RQ['seed']); flip = rng.random(len(I_)) < 0.5
a_, b_ = np.where(flip, J_, I_), np.where(flip, I_, J_); y_ = (~flip).astype(int)   # y=1 when a->b is the wired direction
feat_base = np.column_stack([RQ_RATE[b_] - RQ_RATE[a_], hcond_common[4][b_] - hcond_common[4][a_], RQ_HCOND[b_] - RQ_HCOND[a_]])
cv_rows = []
for name, S in dir_scores.items():
    feat_te = np.column_stack([feat_base, S[a_, b_] - S[b_, a_]])
    for label, F in [('nuisance only', feat_base), (f'nuisance + TE asymmetry ({name})', feat_te)]:
        pred = np.zeros(len(y_))
        for tr, te_ in GroupKFold(5).split(F, y_, groups=np.minimum(a_, b_)):
            mdl = LogisticRegression(max_iter=1000).fit(F[tr], y_[tr], sample_weight=W_[tr]); pred[te_] = mdl.predict_proba(F[te_])[:, 1]
        cv_rows.append(dict(model=label, cv_AUC=roc_auc_score(y_, pred, sample_weight=W_),
                            cv_accuracy=float(np.average((pred > .5) == y_, weights=W_))))
CV = pd.DataFrame(cv_rows).drop_duplicates('model')
rq_table(CV, 'rq1_direction_incremental', 'Grouped cross-validated direction classification: nuisance asymmetries alone versus with TE asymmetry.')
rq_record('rq1_direction', dict(table=DIR.drop(columns='_acc').to_dict('records'), cv=CV.to_dict('records'),
                                unidirectional_group_pairs=int(len(I_)), physical_edges=float(W_.sum())))
fig, ax = plt.subplots(figsize=(6, 3.6), constrained_layout=True)
ax.barh(DIR.predictor, DIR._acc, color=['C0' if p.startswith('TE') else 'C7' for p in DIR.predictor])
ax.axvline(.5, c='k', ls='--', lw=1); ax.set(xlabel='Physical-edge-weighted orientation accuracy', xlim=(0, 1)); ax.invert_yaxis()
rq_fig(fig, 'fig08_3_direction', 'Orientation accuracy of TE asymmetry (saved and matched histories) versus nuisance predictors.')

## 8.4 RQ1 — Signed influence: are wired edges inhibitory in the rising-event representation?

In NMOS logic a transistor whose channel connects a node to ground (**pull-down**) discharges that node when it turns on, which turns *off* every transistor gated by the node. The recording keeps only off-to-on events, so the direct consequence of a pull-down edge (the gated transistor switching off) is never recorded; what can be observed is a *suppression* of the target's rising events. TE is unsigned, so this hypothesis is tested with the state-conditioned effect
$\delta = \sum_s w_s\,[P(y_{t}=1\mid s, x_{t-1}=1) - P(y_{t}=1\mid s, x_{t-1}=0)]$, where $s$ is the target history state. Negative $\delta$ indicates suppression. The synthetic benchmark (7) provides the matching check on circuits with known inverting gates.

In [ ]:
cls = {'pull-down edges only': (RQ_POS_PD > 0) & (RQ_POS_PASS == 0) & RQ_EL,
       'pass edges only': (RQ_POS_PASS > 0) & (RQ_POS_PD == 0) & RQ_EL,
       'no structural edge': (RQ_POS == 0) & RQ_EL}
weights = {'pull-down edges only': RQ_POS_PD, 'pass edges only': RQ_POS_PASS, 'no structural edge': RQ_NEG}
rows = []
for name, mask in cls.items():
    d = RQ_DELTA[mask]; w = weights[name][mask].astype(float); ok = np.isfinite(d) & (w > 0)
    k_neg = int((d[ok] < 0).sum()); n_ = int(ok.sum()); lo, hi = tk.wilson(k_neg, n_)
    rows.append(dict(category=name, group_pairs=n_, physical_pairs=float(w[ok].sum()),
                     fraction_negative_group_pairs=f'{k_neg / max(n_, 1):.3f} [{lo:.3f}, {hi:.3f}]',
                     fraction_negative_physical_weighted=float((w[ok] * (d[ok] < 0)).sum() / w[ok].sum()) if n_ else np.nan,
                     median_delta=float(np.median(d[ok])) if n_ else np.nan,
                     median_TE_bits=float(np.median(RQ_TE[mask][ok])) if n_ else np.nan))
SIGN = pd.DataFrame(rows)
rq_table(SIGN, 'rq1_signed_effect', 'Sign of the state-conditioned source effect by structural class (Wilson 95% intervals over group pairs).')
signed_scores = {'TE × [δ<0] (suppressive TE)': np.where(RQ_DELTA < 0, RQ_TE, 0.0),
                 'TE × [δ>0] (facilitating TE)': np.where(RQ_DELTA > 0, RQ_TE, 0.0),
                 '−δ': -RQ_DELTA, '|δ|': np.abs(RQ_DELTA), 'Pairwise TE (unsigned)': RQ_TE}
rows = []
for level in ['physical', 'group']:
    for name, S in signed_scores.items():
        m = Ranked(np.where(np.isfinite(S), S, 0.0), level).metrics(budgets=[1000])
        rows.append(dict(level=level, score=name, AUC=m['AUC'], AP_lift=m['AP_lift'], lift_at_1000=m['lift@1000']))
SIGNED = pd.DataFrame(rows)
rq_table(SIGNED, 'rq1_signed_scores', 'Discrimination of signed information scores (a mechanistically motivated hypothesis, stated before inspection).')
rq_record('rq1_signed', dict(classes=SIGN.to_dict('records'), scores=SIGNED.to_dict('records')))
fig, ax = plt.subplots(figsize=(6, 3.4), constrained_layout=True)
for name, mask in cls.items():
    d = RQ_DELTA[mask]; w = weights[name][mask].astype(float); ok = np.isfinite(d) & (w > 0)
    ax.hist(d[ok], bins=np.linspace(-1, 1, 81), weights=w[ok] / w[ok].sum(), histtype='step', lw=1.3, label=name)
ax.axvline(0, c='k', lw=.8); ax.set(xlabel='State-conditioned effect δ', ylabel='Physical-pair-weighted fraction'); ax.legend()
rq_fig(fig, 'fig08_4_signed_effect', 'Distribution of the signed source effect δ for pull-down, pass and unwired pairs.')

## 8.5 RQ1 — Sampling phase and interaction delay, full network

The stride-3 reduction keeps one phase. Here pairwise TE is recomputed for **every phase** and several **source delays** over all group pairs (not a panel), and optionally at the original resolution (stride 1). Group identities are those of the analysed phase; in other phases some members of a group can differ, which is noted as a limitation. Delays are reported in original row intervals. Delay 0 is the contemporaneous conditional MI $I(y_t;x_t\mid y_{t-k..t-1})$: in switch-level simulation, combinational logic settles within one step, so direct wiring can act at lag 0 (Part V tests this on a chip with known wiring).

In [ ]:
sweep_rows = []
def sweep_eval(S, label):
    out = dict(label)
    for level in ['physical', 'group']:
        m = Ranked(np.where(np.isfinite(S), S, 0.0), level).metrics(budgets=[1000])
        out[f'{level}_AUC'] = m['AUC']; out[f'{level}_AP_lift'] = m['AP_lift']; out[f'{level}_lift@1000'] = m['lift@1000']
    return out
for phase in RQ['phase_sweep']:
    Xp = RQ_X if phase == rq_phase else rq_series(phase=phase)
    const = float(np.mean(Xp.min(1) == Xp.max(1)))
    for d in RQ['delay_sweep']:
        t0 = time.perf_counter()
        if phase == rq_phase and d in (0, 1):
            S = RQ_TE0 if d == 0 else RQ_TE_EXACT   # already computed in 8.1; not recomputed
        else:
            S, _ = tk.te_matrix(Xp, RQ_K, delay=d)
        sweep_rows.append(sweep_eval(S, dict(stride=rq_stride, phase=phase, delay_samples=d, delay_rows=d * rq_stride,
                                             constant_group_fraction=const, seconds=time.perf_counter() - t0)))
        print(f'  stride {rq_stride} phase {phase} delay {d}: {time.perf_counter() - t0:.0f}s', flush=True)
    del Xp
if RQ['run_stride1']:
    X1 = rq_series(phase=0, stride=1)
    for d in RQ['stride1_delays']:
        t0 = time.perf_counter(); S, _ = tk.te_matrix(X1, np.full(RQ_G, RQ['stride1_k']), delay=d)
        sweep_rows.append(sweep_eval(S, dict(stride=1, phase=0, delay_samples=d, delay_rows=d, constant_group_fraction=float(np.mean(X1.min(1) == X1.max(1))),
                                             seconds=time.perf_counter() - t0)))
        print(f'  stride 1 delay {d}: {time.perf_counter() - t0:.0f}s', flush=True)
    del X1
SWEEP = pd.DataFrame(sweep_rows)
rq_table(SWEEP, 'rq1_phase_delay_sweep', 'Full-network pairwise TE by sampling phase and source delay.')
rq_record('rq1_phase_delay', SWEEP.to_dict('records'))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
for (stride, phase), g in SWEEP.groupby(['stride', 'phase']):
    axes[0].plot(g.delay_rows, g.physical_AUC, marker='o', label=f'stride {stride}, phase {phase}')
    axes[1].plot(g.delay_rows, g.group_AP_lift, marker='o', label=f'stride {stride}, phase {phase}')
axes[0].axhline(.5, c='grey', ls=':'); axes[0].set(xlabel='Source delay (original rows)', ylabel='Physical ROC-AUC')
axes[1].axhline(1, c='grey', ls=':'); axes[1].set(xlabel='Source delay (original rows)', ylabel='Group AP lift'); axes[1].legend(fontsize=7)
rq_fig(fig, 'fig08_5_phase_delay', 'Structural discrimination of pairwise TE across sampling phases and source delays.')

## 8.6 RQ1 — Shared-driver conditioning

A synchronous chip has globally shared drivers. The groups with the largest mean outgoing TE (excluding near-constant groups) are treated as candidate shared drivers and each target's TE is recomputed **conditioned on their past**. Pairs involving the conditioning groups are excluded from both conditions so the comparison uses identical items. If shared drive inflates TE on unwired pairs, conditioning should raise discrimination and remove proportionally more TE from non-edges than from edges.

In [ ]:
out_strength = np.nanmean(RQ_TE, axis=1)
candidates = [g for g in np.argsort(-out_strength) if 0.01 < RQ_RATE[g] < 0.99]
rows = []
for h in RQ['hub_counts']:
    hubs = list(candidates[:h]); mask = RQ_EL.copy(); mask[hubs, :] = False; mask[:, hubs] = False
    t0 = time.perf_counter()
    S_c, _ = tk.te_matrix(RQ_X, RQ_K, cond=lambda t: RQ_X[[u for u in hubs if u != t]] if any(u != t for u in hubs) else None)
    for level in ['physical', 'group']:
        base_m = Ranked(RQ_TE, level, mask).metrics(budgets=[1000]); cond_m = Ranked(S_c, level, mask).metrics(budgets=[1000])
        rows.append(dict(hubs=str([int(u) for u in hubs]), level=level, AUC_unconditioned=base_m['AUC'], AUC_conditioned=cond_m['AUC'],
                         AP_lift_unconditioned=base_m['AP_lift'], AP_lift_conditioned=cond_m['AP_lift'],
                         lift1000_unconditioned=base_m['lift@1000'], lift1000_conditioned=cond_m['lift@1000']))
    keep = mask & (RQ_TE > 0)
    for lab, sel in [('edges', keep & (RQ_POS > 0)), ('non-edges', keep & (RQ_POS == 0))]:
        value = float(np.median(1 - S_c[sel] / RQ_TE[sel])) if sel.any() else np.nan
        for row in rows[-2:]:
            row[f'median_fraction_TE_removed_{lab}'] = value  # negative values: conditioning increased TE (synergy)
    print(f'  conditioned on {h} hub(s) in {time.perf_counter() - t0:.0f}s', flush=True)
HUB = pd.DataFrame(rows)
rq_table(HUB, 'rq1_shared_driver', 'Pairwise TE conditioned on the strongest shared drivers (identical evaluation sets).')
rq_record('rq1_shared_driver', HUB.to_dict('records'))

## 8.7 RQ1 — Convergence with recording length (real data)

Following the sample-size analyses of the exemplar theses, pairwise TE is recomputed on contiguous segments covering 5–100% of the retained recording (several non-overlapping segments for short lengths) with the saved histories. If structural recovery improves steadily with N, weak recovery is a data-limitation; if it plateaus near chance, the limitation is representational or conceptual.

In [ ]:
fractions = [0.1, 0.5, 1.0] if TE6502_QUICK else [0.05, 0.1, 0.25, 0.5, 1.0]
conv_rows = []
for f in fractions:
    L = int(round(f * RQ_N)); n_seg = 1 if f == 1.0 else min(4, int(1 / f))
    for s_ in range(n_seg):
        a = s_ * L; seg = RQ_X[:, a:a + L]
        if seg.shape[1] < 2 * RQ_K.max() + 10: continue
        t0 = time.perf_counter()
        S, nobs = (RQ_TE_EXACT, RQ_NOBS) if L == RQ_N else tk.te_matrix(seg, RQ_K)   # full length = 8.1 matrix
        S = np.where(np.isfinite(S), S, 0)
        null_med = float(np.median([tk.null_mean_bits(tk.df_te(k), n) for k, n in zip(RQ_K, nobs)]))
        row = dict(fraction=f, samples=L, segment=s_, median_null_mean_bits=null_med, seconds=time.perf_counter() - t0)
        for level in ['physical', 'group']:
            m = Ranked(S, level).metrics(budgets=[1000])
            row[f'{level}_AUC'] = m['AUC']; row[f'{level}_AP_lift'] = m['AP_lift']; row[f'{level}_lift@1000'] = m['lift@1000']
        conv_rows.append(row)
    print(f'  fraction {f}: {n_seg} segment(s)', flush=True)
CONV = pd.DataFrame(conv_rows)
rq_table(CONV, 'rq1_convergence', 'Pairwise TE structural discrimination versus recording length (contiguous segments).')
CONV_S = CONV.groupby(['fraction', 'samples']).agg(**{c: (c, 'mean') for c in ['physical_AUC', 'physical_AP_lift', 'group_AUC', 'group_AP_lift']},
                                                    physical_AUC_min=('physical_AUC', 'min'), physical_AUC_max=('physical_AUC', 'max')).reset_index()
rq_record('rq1_convergence', CONV_S.to_dict('records'))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
for level, c in [('physical', 'C0'), ('group', 'C1')]:
    ax[0].plot(CONV_S.samples, CONV_S[f'{level}_AUC'], 'o-', c=c, label=level)
    ax[1].plot(CONV_S.samples, CONV_S[f'{level}_AP_lift'], 'o-', c=c, label=level)
    pts = CONV[['samples', f'{level}_AUC']].to_numpy(); ax[0].scatter(pts[:, 0], pts[:, 1], s=8, c=c, alpha=.4)
ax[0].axhline(.5, ls=':', c='grey'); ax[1].axhline(1, ls=':', c='grey')
ax[0].set(xscale='log', xlabel='Retained samples', ylabel='ROC-AUC'); ax[1].set(xscale='log', xlabel='Retained samples', ylabel='AP lift'); ax[0].legend()
rq_fig(fig, 'fig08_7_convergence', 'Convergence of pairwise-TE structural discrimination with recording length (points: segments; lines: means).')

## 8.8 RQ1 — Does TE add information beyond distance and activity?

Wiring on a chip is spatially local, and active transistors are easier to score. A transistor-pair classifier is therefore fitted with **distance and event rates only** and then with TE (lag 1, lag 0) or MI added. Positives are all evaluable physical edges; negatives are a random sample of unwired physical pairs, reweighted to the full population. Folds are grouped by target activity group (no target appears in both training and test), so the comparison measures generalisation to unseen targets. If TE adds no out-of-fold information beyond distance and activity, its apparent structural signal is explained by those nuisance variables.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
rng = np.random.default_rng(RQ['seed'] + 8)
gs_, gd_ = rq_p2g[_src], rq_p2g[_dst]; okp = (gs_ >= 0) & (gd_ >= 0) & (gs_ != gd_)
P_s, P_t = _src[okp], _dst[okp]
act_ = np.flatnonzero(rq_p2g >= 0); n_neg = 20000 if TE6502_QUICK else 60000
wired_set = set(zip(_src.tolist(), _dst.tolist()))
neg_s, neg_t = [], []
while len(neg_s) < n_neg:
    a_, b_ = rng.choice(act_, 2 * n_neg), rng.choice(act_, 2 * n_neg)
    for u, v in zip(a_, b_):
        if rq_p2g[u] != rq_p2g[v] and (u, v) not in wired_set:
            neg_s.append(u); neg_t.append(v)
            if len(neg_s) >= n_neg: break
neg_s, neg_t = np.asarray(neg_s), np.asarray(neg_t)
S_all = np.r_[P_s, neg_s]; T_all = np.r_[P_t, neg_t]; y_all = np.r_[np.ones(len(P_s)), np.zeros(len(neg_s))]
w_all = np.r_[np.ones(len(P_s)), np.full(len(neg_s), RQ_NEG[RQ_EL].sum() / len(neg_s))]
gS, gT = rq_p2g[S_all], rq_p2g[T_all]
lg = lambda v: np.log10(np.maximum(v, 0) + 1e-6)
F = dict(distance=np.log1p(np.linalg.norm(RQ_CENTRE[S_all] - RQ_CENTRE[T_all], axis=1)),
         source_rate=np.log10(RQ_RATE[gS] + 1e-6), target_rate=np.log10(RQ_RATE[gT] + 1e-6),
         TE_lag1=lg(RQ_TE[gS, gT]), TE_lag0=lg(RQ_TE0[gS, gT]), MI=lg(RQ_MI[gS, gT]))
models = {'Distance + rates': ['distance', 'source_rate', 'target_rate'],
          '+ TE lag 1': ['distance', 'source_rate', 'target_rate', 'TE_lag1'],
          '+ TE lag 0': ['distance', 'source_rate', 'target_rate', 'TE_lag0'],
          '+ TE lag 0 and lag 1': ['distance', 'source_rate', 'target_rate', 'TE_lag0', 'TE_lag1'],
          '+ MI': ['distance', 'source_rate', 'target_rate', 'MI'],
          'TE lag 1 only': ['TE_lag1'], 'TE lag 0 only': ['TE_lag0']}
rows = []
for fold, (tr, te_) in enumerate(GroupKFold(5).split(y_all, y_all, groups=gT)):
    for name, cols in models.items():
        X_ = np.column_stack([F[c] for c in cols])
        mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_[tr], y_all[tr], logisticregression__sample_weight=w_all[tr])
        p = mdl.predict_proba(X_[te_])[:, 1]
        rows.append(dict(fold=fold, model=name, AUC=roc_auc_score(y_all[te_], p, sample_weight=w_all[te_]),
                         AP=average_precision_score(y_all[te_], p, sample_weight=w_all[te_]),
                         log_loss=log_loss(y_all[te_], np.clip(p, 1e-9, 1 - 1e-9), sample_weight=w_all[te_])))
PRED = pd.DataFrame(rows)
base_ = PRED[PRED.model == 'Distance + rates'].set_index('fold')
summ = []
for name, g in PRED.groupby('model', sort=False):
    g = g.set_index('fold'); d_auc = g.AUC - base_.AUC; d_ap = g.AP - base_.AP
    summ.append(dict(model=name, AUC_mean=g.AUC.mean(), AUC_sd=g.AUC.std(), AP_mean=g.AP.mean(), AP_lift_mean=(g.AP / (len(P_s) / w_all.sum())).mean(),
                     delta_AUC_vs_base=d_auc.mean(), folds_AUC_improved=int((d_auc > 0).sum()), delta_AP_vs_base=d_ap.mean(), folds_AP_improved=int((d_ap > 0).sum())))
PREDS = pd.DataFrame(summ)
rq_table(PREDS, 'rq1_incremental_prediction', 'Out-of-fold (grouped by target) prediction of wiring from distance and activity, with and without TE or MI (5 folds).')
rq_record('rq1_incremental_prediction', PREDS.to_dict('records'))

## 8.9 RQ1 — Spatial organisation of information on the die

Each active transistor is drawn at its layout position and coloured by its activity group's mean incoming and outgoing pairwise TE (lag 1 and lag 0), with one shared colour scale per lag. These are descriptive maps of information hubs, not inferred wires.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)
act_t = np.flatnonzero(rq_p2g >= 0)
for r_, (lab, S) in enumerate([('lag 1', RQ_TE), ('lag 0', RQ_TE0)]):
    inc = np.nanmean(S, axis=0); out = np.nanmean(S, axis=1)
    vmax = np.nanpercentile(np.r_[inc, out], 99.5); vmin = max(np.nanpercentile(np.r_[inc, out], 1), 1e-6)
    for c_, (dlab, v) in enumerate([('incoming', inc), ('outgoing', out)]):
        ax = axes[r_, c_]
        ax.scatter(RQ_CENTRE[:, 0], RQ_CENTRE[:, 1], s=1, c='0.85')
        sc = ax.scatter(RQ_CENTRE[act_t, 0], RQ_CENTRE[act_t, 1], s=2, c=np.clip(v[rq_p2g[act_t]], vmin, vmax), cmap='magma',
                        norm=plt.matplotlib.colors.LogNorm(vmin, vmax))
        ax.set(aspect='equal', xticks=[], yticks=[], title=f'Mean {dlab} TE, {lab}')
    fig.colorbar(sc, ax=axes[r_, :], shrink=.8, label='bits')
rq_fig(fig, 'fig08_9_layout_maps', 'Mean incoming and outgoing pairwise TE (lag 1, lag 0) of the activity group of each transistor on the 6502 layout; grey = inactive.')

# Part IV — RQ2: multivariate transfer entropy (IDTxl)

IDTxl multivariate TE (Novelli et al. 2019) is run on **every active target**. It uses non-uniform target embedding, iterative source selection with maximum and minimum statistics, an omnibus test and network-level FDR. IDTxl bivariate TE, in the same framework, is the matched pairwise comparator. Section 9.4 compares multivariate TE with bivariate TE, with JIDT pairwise TE under the bit-correct BH rule, and with pairwise TE at an equal edge budget, all on identical targets with paired bootstrap intervals.

## 9.1 RQ2 — IDTxl multivariate TE: setup, target set and data export

IDTxl's `MultivariateTE` performs greedy, iterative conditioning: candidate past variables of the target are selected first (non-uniform embedding up to `max_lag_target`), then source variables (lag `min_lag_sources`–`max_lag_sources`) are added while they provide significant conditional TE given everything selected so far, followed by pruning (minimum statistic), an omnibus test and sequential maximum statistics (Lizier & Rubinov 2012; Novelli et al. 2019; Wollstadt et al. 2019). `BivariateTE` applies the same machinery to each source separately and provides the matched pairwise comparison.

Settings: `JidtDiscreteCMI` on binary data, `max_lag_sources = 1` (the delay used for pairwise TE), `max_lag_target = 12` (the AIS range), α = 0.05 at every stage, network-level FDR over omnibus p-values. With `permute_in_time=True` IDTxl draws analytic surrogates from JIDT; the estimator class from 5 rescales them to bits (validated in 7). Two run-time patches are applied that do not change results: NumPy-2 compatibility and a vectorised replacement for `combine_discrete_dimensions` (output-identical; ~13× faster).

All active targets are analysed (a stratified handful in smoke-test mode). Every comparison in 9.4 uses exactly the completed targets.

In [ ]:
IDX_DIR = RQ_DIR / 'idtxl'; IDX_DIR.mkdir(exist_ok=True)
if IDTXL_OK:
    tk.patch_idtxl()
    data_path = IDX_DIR / 'data_int8.npy'
    if not data_path.exists():
        np.save(data_path, RQ_X.astype(np.int8))
    active_t = np.flatnonzero((RQ_RATE > 0) & (RQ_RATE < 1))
    if not TE6502_QUICK:
        IDX_TARGETS = active_t                       # thesis run: every active target
    else:                                            # smoke test: a stratified handful
        rng = np.random.default_rng(RQ['seed'] + 37)
        kq = pd.qcut(RQ_K[active_t] + 1e-6 * rng.random(len(active_t)), 3, labels=False, duplicates='drop')
        picks = [int(rng.choice(active_t[kq == a])) for a in np.unique(kq)]
        rest = np.setdiff1d(active_t, picks)
        picks += list(rng.choice(rest, size=max(0, min(RQ['idtxl_quick_targets'] - len(picks), len(rest))), replace=False))
        IDX_TARGETS = np.asarray(sorted(int(t) for t in picks))
    np.save(IDX_DIR / 'targets.npy', IDX_TARGETS)
    IDX_SETTINGS = dict(max_lag_target=RQ['idtxl_max_lag_target'], max_lag_sources=RQ['idtxl_max_lag_sources'],
                        alpha=RQ['alpha'], n_perm=RQ['idtxl_n_perm'], analytic_bits_fix=True)
    (IDX_DIR / 'settings.json').write_text(json.dumps(dict(IDX_SETTINGS, scope='smoke test' if TE6502_QUICK else 'all active targets', targets=IDX_TARGETS.tolist(),
                                                          idtxl_version=getattr(idtxl, '__version__', 'unknown')), indent=2))
    print(f"IDTxl targets: {len(IDX_TARGETS)} ({'smoke test' if TE6502_QUICK else 'all active'}); with structural inputs: {int((RQ_POS[:, IDX_TARGETS] > 0).any(axis=0).sum())}")

## 9.2 RQ2 — Run IDTxl (checkpointed, resumable, parallel)

Each (method, target) writes one JSON file. Re-running this cell resumes. A small subset is also run with the **uncorrected** analytic null to quantify the effect of the units error on real data.

In [ ]:
if IDTXL_OK:
    jobs = []
    for method in RQ['idtxl_methods']:
        d = IDX_DIR / method; d.mkdir(exist_ok=True)
        for t in IDX_TARGETS:
            out = d / f'target_{int(t):04d}.json'
            if not out.exists():
                jobs.append(dict(data_path=str(data_path), target=int(t), method=method, settings=IDX_SETTINGS,
                                 sources=None, out_path=str(out), max_heap=RQ['idtxl_max_heap']))
    d = IDX_DIR / 'multivariate_uncorrected'; d.mkdir(exist_ok=True)
    for t in IDX_TARGETS[:RQ['idtxl_nofix_targets']]:
        out = d / f'target_{int(t):04d}.json'
        if not out.exists():
            jobs.append(dict(data_path=str(data_path), target=int(t), method='multivariate',
                             settings=dict(IDX_SETTINGS, analytic_bits_fix=False), sources=None, out_path=str(out),
                             max_heap=RQ['idtxl_max_heap']))
    print(f'IDTxl jobs to run: {len(jobs)}', flush=True)
    failures = []; started = time.perf_counter()
    if jobs and RQ['workers'] <= 1:
        for i, job in enumerate(jobs, 1):
            try: tk.idtxl_target_worker(job)
            except Exception: failures.append(dict(job=job, error=traceback.format_exc()))
            print(f'  {i}/{len(jobs)} ({time.perf_counter() - started:.0f}s)', flush=True)
    elif jobs:
        with ProcessPoolExecutor(max_workers=RQ['workers'], mp_context=mp.get_context('spawn')) as pool:
            futures = {pool.submit(tk.idtxl_target_worker, job): job for job in jobs}
            for i, f in enumerate(as_completed(futures), 1):
                try: f.result()
                except Exception: failures.append(dict(job=futures[f], error=traceback.format_exc()))
                if i % max(1, len(jobs) // 20) == 0 or i == len(jobs):
                    el_ = time.perf_counter() - started
                    print(f'  {i}/{len(jobs)} done; {el_ / 60:.1f} min elapsed; ~{el_ / i * (len(jobs) - i) / 60:.1f} min remaining', flush=True)
    (IDX_DIR / 'failures.json').write_text(json.dumps(failures, indent=2, default=str))
    if failures:
        print(f'{len(failures)} jobs failed; first error:\n', failures[0]['error'][-1500:])

## 9.3 RQ2 — Assemble networks with network-level FDR

In [ ]:
def idx_load(method):
    out = []
    for t in IDX_TARGETS:
        p = IDX_DIR / method / f'target_{int(t):04d}.json'
        if p.exists(): out.append(json.loads(p.read_text()))
    return out

IDX_NET, IDX_W, IDX_SUM = {}, {}, {}
if IDTXL_OK:
    for method in RQ['idtxl_methods'] + ['multivariate_uncorrected']:
        summaries = idx_load(method)
        if not summaries: continue
        base_method = 'bivariate' if method == 'bivariate' else 'multivariate'
        A_ = tk.assemble_network(summaries, base_method, RQ_G, RQ['alpha'])
        W_m = np.zeros((RQ_G, RQ_G))
        for s in summaries:
            for (src, lag), te_v in zip(s['selected_vars_sources'], s['selected_sources_te'] or []):
                if A_[src, s['target']]:
                    W_m[src, s['target']] = max(W_m[src, s['target']], te_v)
        IDX_NET[method] = A_; IDX_W[method] = W_m; IDX_SUM[method] = summaries
        np.savez_compressed(IDX_DIR / f'network_{method}.npz', adjacency=A_, selected_te=W_m, targets=IDX_TARGETS, representatives=rq_reps)
        print(f"{method}: {len(summaries)} targets analysed, {int(A_.any(axis=0).sum())} targets with selected sources, {int(A_.sum())} edges")
    IDX_DONE = np.asarray(sorted(set.intersection(*[set(s['target'] for s in IDX_SUM[m]) for m in RQ['idtxl_methods'] if m in IDX_SUM]))) if IDX_SUM else np.array([], int)
    rq_record('rq2_targets_completed', len(IDX_DONE))

## 9.4 RQ2 — Does multivariate TE improve on pairwise TE? Paired comparison on identical targets

All methods are evaluated on the same completed target columns. Multivariate TE uses IDTxl's target-level network FDR (BH over omnibus p-values). For bivariate TE the omnibus gate is disabled and FDR is controlled at link level: IDTxl's joint omnibus over all individually selected sources has an analytic degrees-of-freedom count far beyond the sample size and cannot reject, which silently empties bivariate networks (demonstrated in 7). Link-level BH uses exact bit-correct analytic p-values for each selected source, with all tested candidates counted. Compared networks: IDTxl multivariate TE; IDTxl bivariate TE (same framework, no conditioning on other sources); JIDT pairwise TE with the bit-correct BH rule (Section 8.2); pairwise TE ranked to **the same number of edges as multivariate TE** (equal budget, isolating the effect of selection from the effect of network size). Differences use the paired target-cluster bootstrap.

In [ ]:
if IDTXL_OK and len(IDX_DONE):
    colmask = np.zeros((RQ_G, RQ_G), bool); colmask[:, IDX_DONE] = True; colmask &= RQ_EL
    bh_full = np.zeros((RQ_G, RQ_G), bool); bh_full[RQ_EL] = tk.bh_adjust_logp(RQ_LOGP[RQ_EL]) <= np.log(RQ['alpha'])
    nets = {}
    if 'multivariate' in IDX_NET: nets['IDTxl multivariate TE'] = IDX_NET['multivariate']
    if 'bivariate' in IDX_NET: nets['IDTxl bivariate TE'] = IDX_NET['bivariate']
    nets['Pairwise TE, BH (bit-correct)'] = bh_full
    if 'multivariate' in IDX_NET:
        B = int((IDX_NET['multivariate'] & colmask).sum())
        sc = np.where(colmask, RQ_TE, -np.inf); thr = np.sort(sc[colmask])[::-1][max(B - 1, 0)] if B else np.inf
        nets[f'Pairwise TE, top-{B} (equal budget)'] = colmask & (sc >= thr)
    if 'multivariate_uncorrected' in IDX_NET:
        nets['IDTxl multivariate TE, uncorrected null (subset)'] = IDX_NET['multivariate_uncorrected']
    cm = colmask[RQ_EL]; tcol = RQ_COL[cm]
    def eval_net(A_, level, mult=None):
        p, n = rq_weights(level); p, n = p[cm], n[cm]
        if mult is not None: p = p * mult[tcol]; n = n * mult[tcol]
        return tk.binary_metrics(A_[RQ_EL][cm], p, n)
    rows = []
    for name, A_ in nets.items():
        if 'subset' in name: continue
        for level in ['physical', 'group']:
            m = eval_net(A_, level); rows.append(dict(method=name, level=level, edges=int(A_[colmask].sum()), **{k: m[k] for k in ['TP', 'FP', 'precision', 'recall', 'F1', 'MCC', 'precision_lift']}))
    CMP = pd.DataFrame(rows)
    rq_table(CMP, 'rq2_network_comparison', f'Networks on the {len(IDX_DONE)} analysed targets against the netlist (physical and group levels).')
    ref = 'IDTxl multivariate TE'
    drows = []
    if ref in nets:
        for other in [n for n in nets if n != ref and 'subset' not in n]:
            for level in ['physical', 'group']:
                def stat(mult, other=other, level=level):
                    a = eval_net(nets[ref], level, mult); b = eval_net(nets[other], level, mult)
                    return {k: a[k] - b[k] for k in ['precision', 'recall', 'F1', 'MCC']}
                draws = []  # resample only the analysed targets
                rng = np.random.default_rng(RQ['seed'] + 5)
                for d_ in range(RQ['bootstrap_draws']):
                    mult = np.zeros(RQ_G); np.add.at(mult, rng.choice(IDX_DONE, len(IDX_DONE)), 1)
                    try: draws.append(stat(mult))
                    except Exception: pass
                for k in ['precision', 'recall', 'F1', 'MCC']:
                    v = np.asarray([d[k] for d in draws if np.isfinite(d[k])])
                    point = eval_net(nets[ref], level)[k] - eval_net(nets[other], level)[k]
                    drows.append(dict(contrast=f'{ref} − {other}', level=level, metric=k, difference=point,
                                      ci_low=np.quantile(v, .025) if len(v) else np.nan, ci_high=np.quantile(v, .975) if len(v) else np.nan,
                                      fraction_draws_positive=float(np.mean(v > 0)) if len(v) else np.nan))
    DCMP = pd.DataFrame(drows)
    if not DCMP.empty:
        rq_table(DCMP, 'rq2_paired_differences', 'Paired target-bootstrap differences: IDTxl multivariate TE minus each comparator.')
    if 'multivariate_uncorrected' in IDX_NET:
        sub = np.zeros((RQ_G, RQ_G), bool); sub[:, [s['target'] for s in IDX_SUM['multivariate_uncorrected']]] = True; sub &= RQ_EL
        units = dict(targets=int(sub.any(0).sum()), edges_corrected=int((IDX_NET['multivariate'] & sub).sum()),
                     edges_uncorrected=int((IDX_NET['multivariate_uncorrected'] & sub).sum()))
        rq_record('rq2_units_effect_real_data', units); print('Units effect on real targets:', units)
    rq_record('rq2_comparison', CMP.to_dict('records')); rq_record('rq2_paired_differences', DCMP.to_dict('records'))

## 9.5 RQ2 — Anatomy of false positives and recall by edge class

Multivariate conditioning is expected to remove false positives explained by indirect paths and shared parents, while it cannot remove false positives that reflect genuine dependencies outside the reference (and may miss synergistic or redundant inputs). Each predicted non-edge group pair is assigned to the first matching category: reverse edge only, two-step path, shared parent, shared child, or unrelated. Recall is reported separately for pull-down and pass edges.

In [ ]:
if IDTXL_OK and len(IDX_DONE):
    cats = [('reverse only', RQ_REL['reverse_only'] > 0), ('two-step path', RQ_REL['two_step'] > 0),
            ('shared parent', RQ_REL['shared_parent'] > 0), ('shared child', RQ_REL['shared_child'] > 0)]
    rows = []
    for name, A_ in nets.items():
        if 'subset' in name: continue
        fp = A_ & colmask & (RQ_POS == 0); done = np.zeros_like(fp); row = dict(method=name, false_positive_group_pairs=int(fp.sum()))
        for cname, cmask in cats:
            sel = fp & cmask & ~done; row[cname] = int(sel.sum()); done |= sel
        row['unrelated'] = int((fp & ~done).sum())
        tot = max(row['false_positive_group_pairs'], 1)
        row['indirect share (two-step + shared parent)'] = (row['two-step path'] + row['shared parent']) / tot
        for lab, Wc in [('recall pull-down', RQ_POS_PD), ('recall pass', RQ_POS_PASS)]:
            w = Wc * colmask; row[lab] = float((w * A_).sum() / max(w.sum(), 1e-300))
        rows.append(row)
    FPA = pd.DataFrame(rows)
    rq_table(FPA, 'rq2_false_positive_anatomy', 'False-positive anatomy (group pairs) and physical-edge recall by edge class on the analysed targets.')
    rq_record('rq2_false_positive_anatomy', FPA.to_dict('records'))
    plot = FPA.set_index('method')[['reverse only', 'two-step path', 'shared parent', 'shared child', 'unrelated']]
    frac = plot.div(plot.sum(axis=1).replace(0, 1), axis=0)
    fig, ax = plt.subplots(figsize=(7, 3.4), constrained_layout=True)
    frac.plot.barh(stacked=True, ax=ax, width=.7); ax.set(xlabel='Share of false-positive group pairs', ylabel='')
    ax.legend(fontsize=7, bbox_to_anchor=(1, 1)); ax.invert_yaxis()
    rq_fig(fig, 'fig09_5_fp_anatomy', 'Composition of false positives for multivariate, bivariate and pairwise networks on identical targets.')

## 9.6 RQ2 — Target embeddings, source counts and computational cost

In [ ]:
if IDTXL_OK and 'multivariate' in IDX_SUM:
    s_ = IDX_SUM['multivariate']
    emb = pd.DataFrame([dict(target=s['target'], AIS_k=int(RQ_K[s['target']]), selected_target_lags=len(s['selected_vars_target']),
                             max_target_lag=max([l for _, l in s['selected_vars_target']] or [0]), sources=len(s['selected_vars_sources']),
                             omnibus_te=s['omnibus_te'], seconds=s['seconds'], structural_inputs=int((RQ_POS[:, s['target']] > 0).sum()))
                        for s in s_])
    emb.to_csv(RQ_TAB / 'rq2_target_summaries.csv', index=False)
    from scipy.stats import spearmanr
    rho = spearmanr(emb.AIS_k, emb.max_target_lag)
    runtime = {m: float(sum(x['seconds'] for x in IDX_SUM[m]) / 3600) for m in IDX_SUM}
    rq_record('rq2_embedding', dict(spearman_AIS_k_vs_max_target_lag=float(rho.statistic), p=float(rho.pvalue),
                                    median_sources=float(emb.sources.median()), cpu_hours=runtime))
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2), constrained_layout=True)
    ax[0].scatter(emb.AIS_k + np.random.default_rng(0).normal(0, .1, len(emb)), emb.max_target_lag, s=12)
    ax[0].set(xlabel='AIS-selected k (pairwise)', ylabel='Largest IDTxl target lag', title=f'(a) ρ = {rho.statistic:.2f}')
    ax[1].hist(emb.sources, bins=np.arange(emb.sources.max() + 2) - .5); ax[1].set(xlabel='Selected sources per target', ylabel='Targets', title='(b)')
    ax[2].hist(emb.seconds / 60, bins=20); ax[2].set(xlabel='Run time per target (min)', ylabel='Targets', title='(c)')
    rq_fig(fig, 'fig09_6_embedding_runtime', 'IDTxl multivariate TE: target embedding versus AIS history, parent-set sizes and per-target run time.')

## 9.7 RQ2 — Source-lag sensitivity for IDTxl multivariate TE

The main IDTxl run uses one source lag (matching pairwise TE). Because the reduction keeps one sample per three rows and the switch-level simulator settles combinational logic within a step (Part V), relevant influence can be contemporaneous (lag 0) or span several samples. A subset of the analysed targets (the first `idtxl_lag_targets`) is rerun with source lags 0–`idtxl_lag_max` and compared on identical targets: edges, precision, recall and the distribution of selected lags. Lag-0 selections are conditional contemporaneous dependencies; their direction is constrained by conditioning on the target's past, not by temporal precedence, which is stated as a limitation.

In [ ]:
if IDTXL_OK and len(IDX_DONE):
    LAG_T = IDX_DONE[:RQ.get('idtxl_lag_targets', 20)]
    lag_settings = dict(IDX_SETTINGS, max_lag_sources=RQ.get('idtxl_lag_max', 3), min_lag_sources=0)
    d = IDX_DIR / 'multivariate_lags'; d.mkdir(exist_ok=True)
    jobs = [dict(data_path=str(data_path), target=int(t), method='multivariate', settings=lag_settings, sources=None,
                 out_path=str(d / f'target_{int(t):04d}.json'), max_heap=RQ['idtxl_max_heap'])
            for t in LAG_T if not (d / f'target_{int(t):04d}.json').exists()]
    print(f'Lag-sensitivity jobs: {len(jobs)}', flush=True)
    if jobs:
        with ProcessPoolExecutor(max_workers=2, mp_context=mp.get_context('spawn')) as pool:
            for i, f in enumerate(as_completed([pool.submit(tk.idtxl_target_worker, j) for j in jobs]), 1):
                try: f.result()
                except Exception: print(traceback.format_exc()[-800:])
    lag_sum = [json.loads((d / f'target_{int(t):04d}.json').read_text()) for t in LAG_T if (d / f'target_{int(t):04d}.json').exists()]
    if lag_sum:
        A_lag = tk.assemble_network(lag_sum, 'multivariate', RQ_G, RQ['alpha'])
        m_lag = np.zeros((RQ_G, RQ_G), bool); m_lag[:, [s['target'] for s in lag_sum]] = True; m_lag &= RQ_EL
        cm2 = m_lag[RQ_EL]
        rows = []
        for name, A_ in [('Source lag 1 (main run)', IDX_NET['multivariate']), (f"Source lags 0–{lag_settings['max_lag_sources']}", A_lag)]:
            for level in ['physical', 'group']:
                p, n = rq_weights(level)
                m = tk.binary_metrics(A_[RQ_EL][cm2], p[cm2], n[cm2])
                rows.append(dict(setting=name, level=level, edges=int((A_ & m_lag).sum()), precision=m['precision'], recall=m['recall'], MCC=m['MCC']))
        LAGS = pd.DataFrame(rows)
        rq_table(LAGS, 'rq2_source_lag_sensitivity', 'IDTxl multivariate TE with one source lag versus source lags including 0 (identical targets).')
        lag_hist = pd.Series([l for s in lag_sum for _, l in s['selected_vars_sources']]).value_counts().sort_index()
        rq_record('rq2_source_lags', dict(table=LAGS.to_dict('records'), selected_lag_counts=lag_hist.to_dict()))
        print('Selected source lags:', lag_hist.to_dict())

# Part V — A ground-truth 6502: switch-level simulation, interventions, modules, synergy and instruction-aligned dynamics

The recording comes from a switch-level simulation of the 6502 netlist. Part V rebuilds that setting under full control, using a Python port of the published Visual6502 algorithm (James, Silverman & Silverman 2010). This makes possible what a fixed recording cannot:

| Section | Item | Question |
|---|---|---|
| 10 | — | Is the simulator faithful? (the simulated CPU must execute a test program exactly) |
| 11 | Interventions | Which nodes *causally* change which others? Clamping each node at sampled moments gives an interventional ground truth to set against wiring and against TE. Whole-run lesions test whether information measures identify functionally critical nodes (cf. Jonas & Kording 2017). |
| 12 | State vs rising edges; RQ2 on a known chip | Does the representation used by the recording hide structure? Pairwise TE (lags 0–2), zero-lag MI and IDTxl multivariate TE against wiring *and* intervention references. |
| 13 | Functional modules | Even if edges are not recovered, do information networks respect the chip's functional organisation (registers, ALU, decode, timing …)? |
| 14 | Synergy | Partial information decomposition of NOR gates: how much of a gate's output information is synergistic, and does synergy predict which inputs pairwise TE misses? |
| 15 | Realistic synthetic circuits | Random NMOS circuits (pull-ups, NOR/NAND pull-down networks, two-phase clock, dynamic latches) simulated with the same engine: lag 0 vs lag 1, pairwise vs IDTxl (including lag-0 sources), state vs rising. |
| 16 | Instruction-aligned dynamics | When, within each instruction, is information stored and transferred? |

All Part V data are generated here; nothing depends on the recording's unknown semantics. The netlist files (`segdefs.js`, `transdefs.js`, `nodenames.js`) are © Greg James, Brian Silverman and Barry Silverman (CC BY-NC-SA 3.0); cite them in the thesis.

In [ ]:
# 10.0 Configuration for Part V.
SIM = dict(
    half_steps=4000 if TE6502_QUICK else 60000,     # recorded half-steps of the simulated chip
    warmup=200, k_max=6, delays=[0, 1, 2],
    intervention_times=6 if TE6502_QUICK else 40,  # snapshot moments for clamping experiments
    intervention_node_cap=120 if TE6502_QUICK else None,  # None = every non-constant node
    lesions=8 if TE6502_QUICK else 80, lesion_half_steps=1200 if TE6502_QUICK else 4000,
    idtxl_targets=3 if TE6502_QUICK else 30, idtxl_methods=['multivariate', 'multivariate_lag0'],
    module_perms=100 if TE6502_QUICK else 1000,
    nmos_gates=30 if TE6502_QUICK else 80, nmos_inputs=6, nmos_latches=8 if TE6502_QUICK else 16,
    nmos_T=[2000] if TE6502_QUICK else [2000, 6000, 20000, 60000], nmos_reps=2 if TE6502_QUICK else 10,
    nmos_idtxl_reps=1 if TE6502_QUICK else 4, nmos_idtxl_T_max=2000 if TE6502_QUICK else 6000,
    local_edges=60 if TE6502_QUICK else 400, max_cycle_position=14,
)
SIM_DIR = RQ_DIR / 'simulated_chip'; SIM_DIR.mkdir(exist_ok=True)
(SIM_DIR / 'configuration.json').write_text(json.dumps(SIM, indent=2))

In [ ]:
%%writefile sim6502.py
"""sim6502: a Python port of the Visual6502 switch-level simulator (chipsim.js,
Greg James, Brian Silverman & Barry Silverman, 2010; netlist CC BY-NC-SA 3.0).

The algorithm is the published Visual6502 one: transistors are switches; when
a node changes, the connected group of nodes (through 'on' transistors) is
resolved to a single value (ground > power > pull-up/pull-down > any high
member), and transistors gated by changed nodes are switched, repeating until
quiescent. Additions for the thesis:
  * per-half-step recording of every node state;
  * node clamping (in-silico lesions / interventions);
  * fast state snapshots (save/restore) for short-horizon perturbations;
  * instruction annotation (SYNC, address bus, opcode) for alignment.
"""
from __future__ import annotations
import re
import numpy as np

PROGRAM_FIB_LFSR = bytes([   # variables live at $80-$82, outside the code
    0xA2, 0x00, 0xA9, 0x01, 0x85, 0x80, 0xA9, 0x00, 0x85, 0x81,       # LDX #0; LDA #1; STA $80; LDA #0; STA $81
    0xA5, 0x80, 0x9D, 0x00, 0x02, 0x18, 0x65, 0x81, 0xA4, 0x80,       # loop: LDA $80; STA $0200,X; CLC; ADC $81; LDY $80
    0x84, 0x81, 0x85, 0x80, 0xE8, 0xE0, 0x20, 0xD0, 0xED,             # STY $81; STA $80; INX; CPX #$20; BNE loop
    0xA9, 0x01, 0x85, 0x82, 0xA0, 0x00,                               # LDA #1; STA $82; LDY #0
    0xA5, 0x82, 0x0A, 0x90, 0x02, 0x49, 0x1D, 0x85, 0x82,             # lf: LDA $82; ASL A; BCC +2; EOR #$1D; STA $82
    0x99, 0x00, 0x03, 0xC8, 0x4C, 0x23, 0x00,                         # STA $0300,Y; INY; JMP lf
])


def reference_fib_lfsr(n_lfsr=64):
    """Expected memory contents written by PROGRAM_FIB_LFSR (Python reference)."""
    a, b, fib = 1, 0, []
    for _ in range(32):
        fib.append(a); a, b = (a + b) & 0xFF, a
    x, lfsr = 1, []
    for _ in range(n_lfsr):
        carry = x & 0x80; x = (x << 1) & 0xFF
        if carry: x ^= 0x1D
        lfsr.append(x)
    return fib, lfsr


def load_netlist(folder):
    folder = str(folder)
    seg = open(folder + '/segdefs.js').read()
    trans = open(folder + '/transdefs.js').read()
    names = open(folder + '/nodenames.js').read()
    pullup = {}
    for m in re.finditer(r"\[\s*(\d+)\s*,\s*'([+-])'", seg):
        n = int(m.group(1)); pullup[n] = pullup.get(n, False) or m.group(2) == '+'
    tdefs = [(int(a), int(b), int(c), int(d)) for a, b, c, d in
             re.findall(r"\['t(\d+)',\s*(\d+),\s*(\d+),\s*(\d+)", trans)]
    tdefs.sort()
    nodenames = {k: int(v) for k, v in re.findall(r"^\s*([A-Za-z_][\w#]*)\s*:\s*(\d+)", names, re.M)}
    return pullup, tdefs, nodenames


class Chip:
    def __init__(self, netlist_folder=None, definitions=None):
        """Build from a Visual6502 netlist folder, or from definitions =
        (pullup dict, [(transistor_id, gate, c1, c2)], names dict with 'vss'/'vcc')."""
        pullup, tdefs, names = load_netlist(netlist_folder) if definitions is None else definitions
        self.names = dict(names)
        self.gnd, self.pwr = self.names['vss'], self.names['vcc']
        n_nodes = max(max(pullup), max(max(t[1:]) for t in tdefs)) + 1
        self.n_nodes = n_nodes
        self.pullup = [bool(pullup.get(i, False)) for i in range(n_nodes)]
        self.pulldown = [False] * n_nodes
        self.state = [False] * n_nodes
        self.clamp = {}                      # node -> forced value (intervention)
        self.t_gate, self.t_c1, self.t_c2 = [], [], []
        self.t_on = []
        self.gates = [[] for _ in range(n_nodes)]   # transistors gated by node
        self.c1c2s = [[] for _ in range(n_nodes)]   # transistors with a channel terminal at node
        for idx, g, c1, c2 in tdefs:
            if c1 == self.gnd: c1, c2 = c2, self.gnd
            if c1 == self.pwr: c1, c2 = c2, self.pwr
            t = len(self.t_gate)
            self.t_gate.append(g); self.t_c1.append(c1); self.t_c2.append(c2); self.t_on.append(False)
            self.gates[g].append(t); self.c1c2s[c1].append(t); self.c1c2s[c2].append(t)
        self.transistor_ids = [d[0] for d in tdefs]
        self.used_nodes = sorted({n for t in tdefs for n in t[1:]} - {self.gnd, self.pwr})
        self.memory = bytearray(65536)
        self.half_steps = 0

    # ---------------- core switch-level algorithm (chipsim.js) ----------------
    def _group(self, start):
        group = {start}; stack = [start]
        gnd, pwr = self.gnd, self.pwr
        t_on, t_c1, t_c2, c1c2s = self.t_on, self.t_c1, self.t_c2, self.c1c2s
        while stack:
            i = stack.pop()
            if i == gnd or i == pwr:
                continue
            for t in c1c2s[i]:
                if not t_on[t]:
                    continue
                other = t_c2[t] if t_c1[t] == i else t_c1[t]
                if other not in group:
                    group.add(other); stack.append(other)
        return group

    def _value(self, group):
        if self.clamp:
            for n in group:
                if n in self.clamp:
                    return self.clamp[n]
        if self.gnd in group: return False
        if self.pwr in group: return True
        pullup, pulldown, state = self.pullup, self.pulldown, self.state
        for n in group:
            if pullup[n]: return True
            if pulldown[n]: return False
        for n in group:
            if state[n]: return True
        return False

    def recalc(self, nodes):
        gnd, pwr = self.gnd, self.pwr
        work = [n for n in dict.fromkeys(nodes) if n != gnd and n != pwr]
        for _ in range(100):
            if not work:
                return
            pending = {}
            for node in work:
                group = self._group(node)
                value = self._value(group)
                for n in group:
                    if self.state[n] == value:
                        continue
                    self.state[n] = value
                    for t in self.gates[n]:
                        if value and not self.t_on[t]:
                            self.t_on[t] = True; pending[self.t_c1[t]] = 1
                        elif not value and self.t_on[t]:
                            self.t_on[t] = False; pending[self.t_c1[t]] = 1; pending[self.t_c2[t]] = 1
            work = [n for n in pending if n != gnd and n != pwr]

    # ---------------- pins, buses, clocking ----------------
    def set_pin(self, name, high):
        n = self.names[name] if isinstance(name, str) else name
        self.pullup[n] = bool(high); self.pulldown[n] = not high
        self.recalc([n])

    def high(self, name):
        return self.state[self.names[name]]

    def read_bits(self, prefix, n):
        return sum(int(self.state[self.names[f'{prefix}{i}']]) << i for i in range(n))

    def write_data_bus(self, x):
        nodes = []
        for i in range(8):
            n = self.names[f'db{i}']; bit = (x >> i) & 1
            self.pullup[n] = bool(bit); self.pulldown[n] = not bit; nodes.append(n)
        self.recalc(nodes)

    def half_step(self):
        if self.high('clk0'):
            self.set_pin('clk0', False)
            if self.high('rw'):
                self.write_data_bus(self.memory[self.read_bits('ab', 16)])
        else:
            self.set_pin('clk0', True)
            if not self.high('rw'):
                self.memory[self.read_bits('ab', 16)] = self.read_bits('db', 8)
        self.half_steps += 1

    def reset(self, program=PROGRAM_FIB_LFSR, origin=0x0000):
        self.memory[:] = bytes(65536)
        self.memory[origin:origin + len(program)] = program
        self.memory[0xFFFC] = origin & 0xFF; self.memory[0xFFFD] = origin >> 8
        self.state = [False] * self.n_nodes; self.state[self.pwr] = True
        self.t_on = [False] * len(self.t_on)
        for name, v in [('res', False), ('clk0', False), ('rdy', True), ('so', False), ('irq', True), ('nmi', True)]:
            n = self.names[name]; self.pullup[n] = v; self.pulldown[n] = not v
        self.recalc(range(self.n_nodes))
        for _ in range(8):
            self.set_pin('clk0', True); self.set_pin('clk0', False)
        self.set_pin('res', True)
        self.half_steps = 0

    # ---------------- snapshots and interventions ----------------
    def snapshot(self):
        return (self.state.copy(), self.t_on.copy(), self.pullup.copy(), self.pulldown.copy(),
                bytes(self.memory), self.half_steps)

    def restore(self, snap):
        s, t, pu, pd, mem, hs = snap
        self.state = s.copy(); self.t_on = t.copy(); self.pullup = pu.copy(); self.pulldown = pd.copy()
        self.memory[:] = mem; self.half_steps = hs

    def set_clamp(self, node, value):
        """Force a node (and its connected group) to a value: an in-silico lesion."""
        self.clamp[int(node)] = bool(value)
        self.recalc([int(node)])

    def clear_clamps(self):
        nodes = list(self.clamp); self.clamp = {}
        self.recalc(nodes)

    # ---------------- recording ----------------
    def run(self, half_steps, record_nodes=None, progress=0):
        """Run and record node states after every half-step.
        Returns (states [T, len(record_nodes)] uint8, annotation dict)."""
        rec = np.asarray(self.used_nodes if record_nodes is None else record_nodes, dtype=np.int64)
        out = np.zeros((half_steps, len(rec)), dtype=np.uint8)
        clk = np.zeros(half_steps, np.uint8); sync = np.zeros(half_steps, np.uint8)
        ab = np.zeros(half_steps, np.int32); db = np.zeros(half_steps, np.int16); rw = np.zeros(half_steps, np.uint8)
        state = self.state
        for t in range(half_steps):
            self.half_step(); state = self.state
            out[t] = [state[n] for n in rec]
            clk[t] = self.high('clk0'); sync[t] = self.high('sync'); rw[t] = self.high('rw')
            ab[t] = self.read_bits('ab', 16); db[t] = self.read_bits('db', 8)
            if progress and (t + 1) % progress == 0:
                print(f'    simulated {t + 1}/{half_steps} half-steps', flush=True)
        return out, dict(nodes=rec, clk0=clk, sync=sync, rw=rw, address=ab, data=db)

    def transistor_matrix(self):
        """[transistor] -> (gate, c1, c2) in original transdefs order (supply-swapped)."""
        return np.asarray(self.t_gate), np.asarray(self.t_c1), np.asarray(self.t_c2)


# ---------------------------------------------------------------------------
# Synthetic NMOS circuits simulated with the same switch-level engine
# ---------------------------------------------------------------------------

def make_nmos_circuit(n_gates=60, n_inputs=6, n_latches=12, seed=0, gate_mix=(('INV', .3), ('NOR2', .3), ('NOR3', .15), ('NAND2', .25))):
    """Random NMOS logic with depletion pull-ups, two-phase clock and dynamic
    latches (pass transistors gated by phi1/phi2), built like the 6502:
    gate outputs have pull-ups; NOR = parallel pull-downs; NAND = series
    pull-downs through an internal node; latches store charge on unpulled nodes.
    Returns definitions for Chip(definitions=...) plus metadata."""
    rng = np.random.default_rng(seed)
    VSS, VCC, PHI1, PHI2 = 0, 1, 2, 3
    nxt = [4]
    def new():
        n = nxt[0]; nxt[0] += 1; return n
    inputs = [new() for _ in range(n_inputs)]
    latches = [new() for _ in range(n_latches)]      # storage nodes (no pull-up)
    pullup = {VSS: False, VCC: False}; tdefs = []; gates = []
    def tr(g, c1, c2):
        tdefs.append((len(tdefs), g, c1, c2))
    pool = list(inputs) + list(latches)
    kinds, probs = zip(*gate_mix); probs = np.asarray(probs) / sum(probs)
    for _ in range(n_gates):
        kind = str(rng.choice(kinds, p=probs)); out = new(); pullup[out] = True
        k = {'INV': 1, 'NOR2': 2, 'NOR3': 3, 'NAND2': 2}[kind]
        ins = [int(v) for v in rng.choice(pool, size=min(k, len(pool)), replace=False)]
        if kind == 'NAND2':
            mid = new(); pullup[mid] = False; tr(ins[0], out, mid); tr(ins[1], mid, VSS)
        else:
            for g in ins:
                tr(g, out, VSS)
        gates.append(dict(kind=kind, output=out, inputs=ins)); pool.append(out)
    outputs = [g['output'] for g in gates]
    latch_src = []
    for i, s_ in enumerate(latches):   # latch input: a gate output (feedback allowed), clocked by phi1 or phi2
        src = int(rng.choice(outputs)); clk = PHI1 if i % 2 == 0 else PHI2
        tr(clk, s_, src); latch_src.append((s_, src, clk)); pullup.setdefault(s_, False)
    for n in range(4, nxt[0]):
        pullup.setdefault(n, False)
    names = {'vss': VSS, 'vcc': VCC, 'phi1': PHI1, 'phi2': PHI2}
    for i, n in enumerate(inputs):
        names[f'in{i}'] = n
    return (pullup, tdefs, names), dict(inputs=inputs, latches=latch_src, gates=gates, n_nodes=nxt[0])


def simulate_nmos(definitions, meta, half_steps, seed=0, input_persistence=0.9):
    """Two-phase non-overlapping clock; inputs change once per cycle with persistence."""
    rng = np.random.default_rng(seed)
    chip = Chip(definitions=definitions)
    chip.state = [False] * chip.n_nodes; chip.state[chip.pwr] = True
    chip.recalc(range(chip.n_nodes))
    vals = rng.integers(0, 2, len(meta['inputs']))
    out = np.zeros((half_steps, len(chip.used_nodes)), np.uint8)
    for t in range(half_steps):
        phase = t % 2
        if phase == 0:
            keep = rng.random(len(vals)) < input_persistence
            vals = np.where(keep, vals, rng.integers(0, 2, len(vals)))
            for i, n in enumerate(meta['inputs']):
                chip.pullup[n] = bool(vals[i]); chip.pulldown[n] = not vals[i]
            chip.recalc(meta['inputs'])
        chip.set_pin('phi1', phase == 0)
        chip.set_pin('phi2', phase == 1)
        out[t] = [chip.state[n] for n in chip.used_nodes]
    return out, chip

## 10. The simulated 6502 and its validation

The test program computes the first 32 Fibonacci numbers (mod 256) into `$0200`, then runs an endless 8-bit LFSR writing to `$0300`. It exercises loads, stores, indexed addressing, ADC with carry, shifts, EOR, compare, branches and jumps. The simulation is accepted only if every value written matches a Python reference. Node states are recorded after every half-step together with the clock, SYNC (opcode fetch), R/W and address bus.

In [ ]:
import importlib, urllib.request
NET_DIR = ROOT / 'data/visual6502'
for fname in ['segdefs.js', 'transdefs.js', 'nodenames.js']:
    if not (NET_DIR / fname).exists():
        url = f'https://raw.githubusercontent.com/trebonian/visual6502/master/{fname}'
        try:
            print('Downloading', url); urllib.request.urlretrieve(url, NET_DIR / fname)
        except Exception as error:
            raise FileNotFoundError(f'{fname} is required for Part V and could not be downloaded ({error}). '
                                    f'Download it manually from {url} into {NET_DIR}.') from None
import sim6502; importlib.reload(sim6502)
SIM_STATES = SIM_DIR / 'states.npz'
CHIP = sim6502.Chip(NET_DIR)
if SIM_STATES.exists():
    with np.load(SIM_STATES) as z:
        S_STATE = z['states']; S_ANN = {k: z[k] for k in ['clk0', 'sync', 'rw', 'address', 'data', 'nodes']}
        S_MEM = bytes(z['memory'])
else:
    CHIP.reset(); t0 = time.perf_counter()
    CHIP.run(SIM['warmup'])
    S_STATE, S_ANN = CHIP.run(SIM['half_steps'], progress=max(1, SIM['half_steps'] // 5))
    S_MEM = bytes(CHIP.memory)
    print(f"Simulated {SIM['half_steps']} half-steps in {time.perf_counter() - t0:.0f}s")
    np.savez_compressed(SIM_STATES, states=S_STATE, memory=np.frombuffer(S_MEM, np.uint8), **S_ANN)
fib_ref, lfsr_ref = sim6502.reference_fib_lfsr(256)
fib_ok = list(S_MEM[0x200:0x220]) == fib_ref
n_lfsr = next((i for i in range(256) if S_MEM[0x300 + i] == 0), 256)
if n_lfsr < 256:
    lfsr_ok = list(S_MEM[0x300:0x300 + n_lfsr]) == lfsr_ref[:n_lfsr]
else:  # the LFSR loop ran past 256 writes and Y wrapped: replay the writes with wraparound
    _, _seq = sim6502.reference_fib_lfsr(8192); _buf = bytearray(256); lfsr_ok = False
    for _i, _v in enumerate(_seq):
        _buf[_i & 0xFF] = _v
        if _i >= 255 and bytes(_buf) == bytes(S_MEM[0x300:0x400]) and S_MEM[0x82] == _v:
            lfsr_ok = True; break
validation = dict(fibonacci_values_correct=fib_ok, lfsr_values_checked=n_lfsr, lfsr_correct=lfsr_ok,
                  half_steps=int(len(S_STATE)), recorded_nodes=int(S_STATE.shape[1]),
                  instructions=int(np.sum(np.diff(S_ANN['sync'].astype(int)) == 1)))
rq_record('sim_validation', validation); print(validation)
assert fib_ok and lfsr_ok, 'Simulated CPU did not execute the program correctly; do not use Part V.'

S_NODES = np.asarray(S_ANN['nodes'])
g_, c1_, c2_ = CHIP.transistor_matrix()
S_A = tk.node_reference_from_transistors(g_, c1_, c2_, S_NODES, supply={CHIP.gnd, CHIP.pwr})
S_REPR = {'state': S_STATE.T.astype(np.uint8)}
_r = np.zeros_like(S_REPR['state']); _r[:, 1:] = (S_REPR['state'][:, 1:] == 1) & (S_REPR['state'][:, :-1] == 0)
S_REPR['rising'] = _r
S_GROUPS = {}
for rep_, X_ in S_REPR.items():
    reps_, ug_ = tk.group_identical(X_); pos_, neg_ = tk.group_reference(S_A, ug_, len(reps_))
    S_GROUPS[rep_] = dict(reps=reps_, ug=ug_, X=X_[reps_], pos=pos_, neg=neg_, G=len(reps_))
    print(f"{rep_}: {len(reps_)} distinct non-constant node series; wired node pairs evaluable: {int(pos_.sum())} of {int(S_A.sum())}")

## 11. Interventional ground truth

At each of `intervention_times` moments the full chip state is saved. For every non-constant node *i*, the chip is restored, node *i* is clamped to the opposite of its current value, and the change in every other node is recorded **within the same half-step** (after the network re-settles; the instantaneous causal effect) and **after the next half-step** (one-step causal effect). $E_0[i,j]$ and $E_1[i,j]$ are the fractions of moments at which clamping *i* changed *j*. A directed pair is a *causal edge* if $E>0$. This is the counterfactual notion of influence that TE approximates observationally; comparing it with wiring separates "wired" from "effective" connections.

**Whole-run lesions.** A sample of nodes is clamped low for the entire program; a lesion is *critical* if the program's output differs from the reference. The question, after Jonas & Kording (2017), is whether information measures (TE out-strength, AIS, activity) identify critical nodes better than chance.

In [ ]:
INT_PATH = SIM_DIR / 'interventions.npz'
varying = np.flatnonzero(S_STATE.min(0) != S_STATE.max(0))
if SIM['intervention_node_cap']:
    varying = np.sort(np.random.default_rng(1).choice(varying, min(len(varying), SIM['intervention_node_cap']), replace=False))
if INT_PATH.exists():
    with np.load(INT_PATH) as z: E0, E1, int_nodes, n_times = z['E0'], z['E1'], z['nodes'], int(z['n_times'])
else:
    times = np.sort(np.random.default_rng(2).choice(np.arange(500, SIM['half_steps'] - 2), SIM['intervention_times'], replace=False))
    CHIP.reset(); CHIP.run(SIM['warmup'])
    snaps, t_prev = [], 0
    for t in times:
        for _ in range(t - t_prev): CHIP.half_step()
        t_prev = t; snaps.append(CHIP.snapshot())
    rec = S_NODES; idx_state = lambda: np.fromiter((CHIP.state[n] for n in rec), dtype=bool, count=len(rec))
    E0 = np.zeros((len(varying), len(rec)), np.float32); E1 = np.zeros_like(E0)
    t0 = time.perf_counter()
    for k_, snap in enumerate(snaps):
        CHIP.restore(snap); base0 = idx_state(); CHIP.half_step(); base1 = idx_state()
        for a_, i in enumerate(varying):
            CHIP.restore(snap); CHIP.clamp = {}
            node = int(rec[i]); CHIP.set_clamp(node, not CHIP.state[node])
            E0[a_] += idx_state() != base0
            CHIP.half_step(); E1[a_] += idx_state() != base1
            CHIP.clamp = {}
        print(f'  moment {k_ + 1}/{len(snaps)} ({time.perf_counter() - t0:.0f}s)', flush=True)
    n_times = len(snaps); E0 /= n_times; E1 /= n_times; int_nodes = varying
    np.savez_compressed(INT_PATH, E0=E0, E1=E1, nodes=int_nodes, n_times=n_times)
# Unit-level causal references [node, node] (rows: intervened node).
nU = len(S_NODES)
def full_matrix(E):
    M = np.zeros((nU, nU), np.float32); M[int_nodes] = E; np.fill_diagonal(M, 0); return M
S_C0, S_C1 = full_matrix(E0), full_matrix(E1)
A_dense = S_A.toarray() > 0
sub = np.zeros(nU, bool); sub[int_nodes] = True
rows_ = []
for name, C in [('instantaneous (same half-step)', S_C0), ('next half-step', S_C1)]:
    Cb = C[sub] > 0; Ab = A_dense[sub]
    rows_.append(dict(effect=name, causal_pairs=int(Cb.sum()), wired_pairs=int(Ab.sum()),
                      wired_pairs_with_causal_effect=float((Cb & Ab).sum() / max(Ab.sum(), 1)),
                      causal_pairs_that_are_wired=float((Cb & Ab).sum() / max(Cb.sum(), 1)),
                      median_effect_on_wired=float(np.median(C[sub][Ab])) if Ab.any() else np.nan))
INT_SUM = pd.DataFrame(rows_)
rq_table(INT_SUM, 'sim_wiring_vs_intervention', 'Simulated 6502: overlap between wiring and interventional (clamping) effects among intervened nodes.')
rq_record('sim_interventions', INT_SUM.to_dict('records'))

In [ ]:
# Whole-run lesions and information measures.
LES_PATH = SIM_DIR / 'lesions.json'
fib_ref, lfsr_ref = sim6502.reference_fib_lfsr(256)
if LES_PATH.exists():
    LES = pd.DataFrame(json.loads(LES_PATH.read_text()))
else:
    rng = np.random.default_rng(3); cand = np.flatnonzero(S_STATE.min(0) != S_STATE.max(0))
    picks = rng.choice(cand, min(SIM['lesions'], len(cand)), replace=False); out = []
    CHIP.reset(); CHIP.run(SIM['lesion_half_steps']); ref_mem = bytes(CHIP.memory)
    for i, u in enumerate(picks):
        CHIP.reset(); CHIP.clamp = {}; CHIP.set_clamp(int(S_NODES[u]), False)
        try:
            CHIP.run(SIM['lesion_half_steps'])
            mem = bytes(CHIP.memory); broken = mem[0x200:0x400] != ref_mem[0x200:0x400]
        except Exception:
            broken = True
        CHIP.clamp = {}
        out.append(dict(node_index=int(u), node=int(S_NODES[u]), critical=bool(broken)))
    LES = pd.DataFrame(out); LES_PATH.write_text(LES.to_json(orient='records'))
G_ = S_GROUPS['state']; Xs = G_['X']; ks_state = np.asarray([tk.ais_select_k(x, SIM['k_max']) for x in Xs])
TE_s1, _ = tk.te_matrix(Xs, ks_state, delay=1); TE_s0, _ = tk.te_matrix(Xs, ks_state, delay=0)
grp = G_['ug'][LES.node_index]
LES['TE_out_lag1'] = [np.nanmean(TE_s1[g]) if g >= 0 else np.nan for g in grp]
LES['TE_out_lag0'] = [np.nanmean(TE_s0[g]) if g >= 0 else np.nan for g in grp]
LES['activity'] = S_STATE[:, LES.node_index].mean(0)
LES['causal_out_degree'] = [float((S_C0[u] > 0).sum()) if sub[u] else np.nan for u in LES.node_index]
LES['wired_out_degree'] = A_dense[LES.node_index].sum(1)
from sklearn.metrics import roc_auc_score
rows_ = []
for feat in ['TE_out_lag1', 'TE_out_lag0', 'activity', 'wired_out_degree', 'causal_out_degree']:
    ok = LES[feat].notna()
    if LES.critical[ok].nunique() == 2:
        rows_.append(dict(predictor=feat, lesions=int(ok.sum()), critical=int(LES.critical[ok].sum()),
                          AUC_predicting_criticality=roc_auc_score(LES.critical[ok], LES[feat][ok])))
LESR = pd.DataFrame(rows_)
if not LESR.empty: rq_table(LESR, 'sim_lesion_criticality', 'Whole-run lesions: do information, activity or degree measures predict which nodes are critical to the program?')
rq_record('sim_lesions', dict(n=len(LES), critical_fraction=float(LES.critical.mean()), predictors=LESR.to_dict('records')))
print(f'Lesions: {len(LES)}; critical fraction {LES.critical.mean():.2f}')

## 12. Does the representation hide structure? Pairwise and multivariate TE on a chip with known wiring and causal effects (item 2; RQ1/RQ2 on ground truth)

Every score is evaluated against three references on the simulated chip: **wiring** (gate node → channel nodes), **instantaneous causal effect** ($E_0>0$) and **one-step causal effect** ($E_1>0$), for both the full **state** representation and the **rising-edge** representation used by the recording. Scores: zero-lag MI; pairwise TE at source lags 0, 1, 2 (AIS-selected target history); and IDTxl multivariate TE (source lag 1, and source lags 0–1) on a subset of targets. Causal references are restricted to intervened source nodes.

In [ ]:
def group_causal(C, ug, G):
    Cs = sparse.csr_matrix((C > 0).astype(float)); return tk.group_reference(Cs, ug, G)
REFS = {}
for rep_, d in S_GROUPS.items():
    gsub = np.zeros(d['G'], bool); gsub[np.unique(d['ug'][int_nodes][d['ug'][int_nodes] >= 0])] = True
    REFS[rep_] = {'wiring': (d['pos'], d['neg'], None)}
    for nm, C in [('causal, same half-step', S_C0), ('causal, next half-step', S_C1)]:
        p_, n_ = group_causal(C, d['ug'], d['G']); REFS[rep_][nm] = (p_, n_, gsub)
rows_ = []; S_SCORES = {}
for rep_, d in S_GROUPS.items():
    X_ = d['X']
    ks_ = ks_state if rep_ == 'state' else np.asarray([tk.ais_select_k(x, SIM['k_max']) for x in X_]); d['k'] = ks_
    cached = {0: TE_s0, 1: TE_s1} if rep_ == 'state' else {}   # state lag 0/1 already computed in Section 11
    scores = {'Zero-lag MI': tk.te_matrix(X_, np.zeros(d['G'], int), delay=0)[0]}
    for lag in SIM['delays']:
        scores[f'Pairwise TE, lag {lag}'] = cached[lag] if lag in cached else tk.te_matrix(X_, ks_, delay=lag)[0]
    S_SCORES[rep_] = scores
    for ref_name, (p_, n_, gsub) in REFS[rep_].items():
        for sname, S in scores.items():
            S2 = np.where(np.isfinite(S), S, 0).copy()
            if gsub is not None:
                S2[~gsub, :] = np.nan
            m = tk.rank_metrics(S2, p_, n_, budgets=(100,))
            rows_.append(dict(representation=rep_, reference=ref_name, score=sname, AUC=m['AUC'], AP_lift=m['AP_lift'], lift_at_100=m['lift@100']))
    print(f'  {rep_}: scores computed', flush=True)
SIMR = pd.DataFrame(rows_)
rq_table(SIMR, 'sim_representation_scores', 'Simulated 6502: discrimination of MI and pairwise TE (lags 0–2) against wiring and interventional references, state versus rising-edge representation.')
rq_record('sim_representation_scores', SIMR.to_dict('records'))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True, constrained_layout=True)
for ax, ref_name in zip(axes, ['wiring', 'causal, same half-step', 'causal, next half-step']):
    d = SIMR[SIMR.reference == ref_name].pivot(index='score', columns='representation', values='AUC')
    d.plot.barh(ax=ax); ax.axvline(.5, ls=':', c='k'); ax.set(xlabel='ROC-AUC', title=f'Reference: {ref_name}')
rq_fig(fig, 'fig12_representation', 'Simulated 6502: ROC-AUC of MI and pairwise TE at lags 0–2 against wiring and causal references, for state and rising-edge representations.')

In [ ]:
# IDTxl on the simulated chip (subset of targets), both representations.
SIM_IDX = {}
if IDTXL_OK:
    for rep_, d in S_GROUPS.items():
        path_ = SIM_DIR / f'idtxl_data_{rep_}.npy'
        if not path_.exists(): np.save(path_, d['X'].astype(np.int8))
        rng = np.random.default_rng(9); has_in = np.flatnonzero(d['pos'].sum(0) > 0)
        targets = np.sort(rng.choice(has_in, min(SIM['idtxl_targets'], len(has_in)), replace=False))
        jobs = []
        for method in SIM['idtxl_methods']:
            out_dir = SIM_DIR / f'idtxl_{rep_}_{method}'; out_dir.mkdir(exist_ok=True)
            st = dict(max_lag_target=SIM['k_max'], max_lag_sources=1, min_lag_sources=0 if method.endswith('lag0') else 1,
                      alpha=RQ['alpha'], n_perm=RQ['idtxl_n_perm'], analytic_bits_fix=True)
            for t in targets:
                out = out_dir / f'target_{int(t):04d}.json'
                if not out.exists():
                    jobs.append(dict(data_path=str(path_), target=int(t), method='multivariate', settings=st, sources=None,
                                     out_path=str(out), max_heap=RQ['idtxl_max_heap']))
        print(f'{rep_}: IDTxl jobs {len(jobs)}', flush=True)
        if jobs:
            with ProcessPoolExecutor(max_workers=RQ['workers'], mp_context=mp.get_context('spawn')) as pool:
                for f in as_completed([pool.submit(tk.idtxl_target_worker, j) for j in jobs]):
                    try: f.result()
                    except Exception: print(traceback.format_exc()[-800:])
        for method in SIM['idtxl_methods']:
            out_dir = SIM_DIR / f'idtxl_{rep_}_{method}'
            summ = [json.loads(p.read_text()) for p in sorted(out_dir.glob('target_*.json'))]
            if summ: SIM_IDX[(rep_, method)] = (tk.assemble_network(summ, 'multivariate', d['G'], RQ['alpha']), [s['target'] for s in summ])
    rows_ = []
    for (rep_, method), (A_, tg) in SIM_IDX.items():
        d = S_GROUPS[rep_]; G = d['G']; cm = np.zeros((G, G), bool); cm[:, tg] = True; np.fill_diagonal(cm, False)
        B = int(A_[cm].sum()); te1 = np.where(np.isfinite(S_SCORES[rep_]['Pairwise TE, lag 1']), S_SCORES[rep_]['Pairwise TE, lag 1'], -np.inf)
        thr = np.sort(te1[cm])[::-1][max(B - 1, 0)] if B else np.inf
        for ref_name, (p_, n_, gsub) in REFS[rep_].items():
            mask = cm if gsub is None else cm & gsub[:, None]
            for label, P_ in [(f'IDTxl {method}', A_), (f'Pairwise TE lag 1, top-{B}', cm & (te1 >= thr))]:
                m = tk.binary_metrics(P_[mask], p_[mask], n_[mask])
                rows_.append(dict(representation=rep_, reference=ref_name, method=label, targets=len(tg), edges=int(P_[mask].sum()),
                                  precision=m['precision'], recall=m['recall'], MCC=m['MCC']))
    if rows_:
        SIMI = pd.DataFrame(rows_)
        rq_table(SIMI, 'sim_idtxl', 'Simulated 6502: IDTxl multivariate TE (source lag 1; source lags 0–1) versus pairwise TE at equal budget, against wiring and causal references.')
        rq_record('sim_idtxl', SIMI.to_dict('records'))

## 13. Functional modules

Transistors and nodes are assigned to functional units from Visual6502 node names (registers, program counter, status flags, ALU, internal buses, external buses and latches, instruction register and decode, timing and control); unnamed units inherit the majority label of their structural neighbours (label propagation, reported separately). Two tests per information network: (i) the ratio of mean within-module to between-module score, against a label-permutation null that preserves module sizes; (ii) Louvain communities of the information network compared with module labels (NMI, ARI). The structural network is analysed identically as a reference point.

In [ ]:
names_ = CHIP.names
# Real recording: transistor -> module via gate/channel node names, propagate over the transistor graph.
mod_t, src_t = tk.assign_modules(RQ_M, [(rq_gate[t], rq_c1[t], rq_c2[t]) for t in range(RQ_M)], names_, RQ_A)
grp_mod = np.array(['Unassigned'] * RQ_G, dtype=object)
for g in range(RQ_G):
    m_ = mod_t[rq_p2g == g]; m_ = m_[m_ != 'Unassigned']
    if len(m_): v, c = np.unique(m_, return_counts=True); grp_mod[g] = v[np.argmax(c)]
grp_mod = grp_mod.astype(str)
# Simulated chip: node -> module.
mod_n, src_n = tk.assign_modules(len(S_NODES), [(n,) for n in S_NODES], names_, S_A)
sim_grp_mod = np.array(['Unassigned'] * S_GROUPS['state']['G'], dtype=object)
for g in range(S_GROUPS['state']['G']):
    m_ = mod_n[S_GROUPS['state']['ug'] == g]; m_ = m_[m_ != 'Unassigned']
    if len(m_): v, c = np.unique(m_, return_counts=True); sim_grp_mod[g] = v[np.argmax(c)]
sim_grp_mod = sim_grp_mod.astype(str)
cov = pd.DataFrame([dict(data='recording (transistors)', named=int((src_t == 'named').sum()), propagated=int((src_t == 'propagated').sum()), unassigned=int((src_t == 'unassigned').sum())),
                    dict(data='simulated chip (nodes)', named=int((src_n == 'named').sum()), propagated=int((src_n == 'propagated').sum()), unassigned=int((src_n == 'unassigned').sum()))])
rq_table(cov, 'module_assignment', 'Functional-module assignment coverage (named versus propagated labels).')
networks = {('recording', 'Pairwise TE'): (RQ_TE, grp_mod), ('recording', 'Zero-lag MI'): (RQ_MI, grp_mod),
            ('recording', 'Structural (group edges)'): ((RQ_POS > 0).astype(float), grp_mod),
            ('simulated, state', 'Pairwise TE lag 1'): (S_SCORES['state']['Pairwise TE, lag 1'], sim_grp_mod),
            ('simulated, state', 'Pairwise TE lag 0'): (S_SCORES['state']['Pairwise TE, lag 0'], sim_grp_mod),
            ('simulated, state', 'Zero-lag MI'): (S_SCORES['state']['Zero-lag MI'], sim_grp_mod),
            ('simulated, state', 'Structural (group edges)'): ((S_GROUPS['state']['pos'] > 0).astype(float), sim_grp_mod)}
if 'multivariate' in IDX_W:
    networks[('recording', 'IDTxl multivariate TE (analysed targets)')] = (IDX_W['multivariate'], grp_mod)
rows_ = []
for (data_, name), (W, lab) in networks.items():
    r1 = tk.within_between_ratio(W, lab, perms=SIM['module_perms'], seed=4)
    r2, _ = tk.community_agreement(W, lab, seed=4)
    rows_.append(dict(data=data_, network=name, within_between_ratio=r1['ratio'], null_mean=r1['null_mean'],
                      null_97_5=r1['null_q975'], p_perm=r1['p_perm'], louvain_communities=r2['communities'], NMI=r2['NMI'], ARI=r2['ARI']))
MOD = pd.DataFrame(rows_)
rq_table(MOD, 'modules', 'Functional-module organisation of information and structural networks: within/between ratio with permutation null; Louvain agreement with module labels.')
rq_record('modules', MOD.to_dict('records'))
mods = [m for m in np.unique(grp_mod) if m != 'Unassigned']
H = np.full((len(mods), len(mods)), np.nan)
for i, a in enumerate(mods):
    for j, b in enumerate(mods):
        blk = RQ_TE[np.ix_(grp_mod == a, grp_mod == b)]; H[i, j] = np.nanmean(blk) if blk.size else np.nan
fig, ax = plt.subplots(figsize=(6.5, 5.2), constrained_layout=True)
im = ax.imshow(H, cmap='viridis'); ax.set_xticks(range(len(mods)), mods, rotation=60, ha='right'); ax.set_yticks(range(len(mods)), mods)
ax.set(xlabel='Target module', ylabel='Source module', title='Mean pairwise TE between functional modules (recording)'); fig.colorbar(im, ax=ax, label='bits')
rq_fig(fig, 'fig13_module_te', 'Mean pairwise TE between functional modules of the 6502 (recording).')

## 14. Synergy in logic gates: partial information decomposition

Two-input NOR gates are identified in the netlist (a pulled-up output node whose only channel connections are two parallel pull-down transistors). For each gate the information that the inputs carry about the output is decomposed into redundant, unique and synergistic parts (Williams & Beer 2010 I_min; minimum-MI redundancy as a check). Lag 0 measures the logic itself; lag 1 is what lag-1 TE can see. The decomposition is computed on the simulated chip in both representations, and on the recording where both inputs and the output are observable through transistors they gate. Finally, the synergy fraction of each input edge is related to whether pairwise TE ranks that edge highly.

In [ ]:
gate_of = {}
for t, (g, a, b) in enumerate(zip(g_, c1_, c2_)):
    gate_of.setdefault(int(a), []).append((t, int(g), int(b)))
nor2 = []
for o in range(CHIP.n_nodes):
    if not CHIP.pullup[o] or o in (CHIP.gnd, CHIP.pwr): continue
    ch = [CHIP.t_c2[t] if CHIP.t_c1[t] == o else CHIP.t_c1[t] for t in CHIP.c1c2s[o]]
    if len(CHIP.c1c2s[o]) == 2 and all(c == CHIP.gnd for c in ch):
        ins = sorted({CHIP.t_gate[t] for t in CHIP.c1c2s[o]})
        if len(ins) == 2: nor2.append((o, ins[0], ins[1]))
print('Two-input NOR gates found in the netlist:', len(nor2))
col = {int(n): i for i, n in enumerate(S_NODES)}
rows_ = []
for rep_, Xfull in S_REPR.items():
    for o, a, b in nor2:
        if not all(n in col for n in (o, a, b)): continue
        y, x1, x2 = Xfull[col[o]], Xfull[col[a]], Xfull[col[b]]
        if min(y.std(), x1.std(), x2.std()) == 0: continue
        for lag in (0, 1):
            r = tk.pid_two_sources(x1[:len(y) - lag], x2[:len(y) - lag], y[lag:])
            rows_.append(dict(data='simulated', representation=rep_, lag=lag, output=o, input1=a, input2=b, **r))
# Recording: node -> active group of a transistor gated by that node.
node_group = {}
for t in range(RQ_M):
    if rq_p2g[t] >= 0: node_group.setdefault(int(rq_gate[t]), int(rq_p2g[t]))
for o, a, b in nor2:
    if all(n in node_group for n in (o, a, b)) and len({node_group[n] for n in (o, a, b)}) == 3:
        y, x1, x2 = RQ_X[node_group[o]], RQ_X[node_group[a]], RQ_X[node_group[b]]
        r = tk.pid_two_sources(x1[:-1], x2[:-1], y[1:])
        rows_.append(dict(data='recording', representation='rising', lag=1, output=o, input1=a, input2=b, **r))
PID = pd.DataFrame(rows_)
if not PID.empty:
    for k_ in ['Imin', 'MMI']:
        PID[f'{k_}_synergy_fraction'] = PID[f'{k_}_synergy'] / PID.I_joint.where(PID.I_joint > 0)
        PID[f'{k_}_redundancy_fraction'] = PID[f'{k_}_redundancy'] / PID.I_joint.where(PID.I_joint > 0)
    PID.to_csv(RQ_TAB / 'pid_nor_gates.csv', index=False)
    PS = PID.groupby(['data', 'representation', 'lag']).agg(gates=('I_joint', 'size'), joint_MI=('I_joint', 'median'),
          redundancy_frac=('Imin_redundancy_fraction', 'median'), synergy_frac=('Imin_synergy_fraction', 'median'),
          synergy_frac_MMI=('MMI_synergy_fraction', 'median')).reset_index()
    rq_table(PS, 'pid_summary', 'Partial information decomposition of two-input NOR gates (medians across gates).')
    rq_record('pid_summary', PS.to_dict('records'))
    # Does synergy predict missed input edges (pairwise TE lag 1, simulated state)?
    d = S_GROUPS['state']; S1 = S_SCORES['state']['Pairwise TE, lag 1']; rank_rows = []
    pct = lambda v: np.mean(S1[np.isfinite(S1)] <= v)
    sub_ = PID[(PID.data == 'simulated') & (PID.representation == 'state') & (PID.lag == 1)]
    for _, r in sub_.iterrows():
        go, ga, gb = (d['ug'][col[int(n)]] for n in (r.output, r.input1, r.input2))
        if min(go, ga, gb) < 0: continue
        for gi in (ga, gb):
            rank_rows.append(dict(synergy_fraction=r.Imin_synergy_fraction, te_percentile=pct(S1[gi, go])))
    RR = pd.DataFrame(rank_rows).dropna()
    if len(RR) > 5:
        from scipy.stats import spearmanr
        rho = spearmanr(RR.synergy_fraction, RR.te_percentile)
        rq_record('pid_synergy_vs_te_rank', dict(edges=len(RR), spearman=float(rho.statistic), p=float(rho.pvalue)))
        fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
        for (rep_, lag), g in PID[PID.data == 'simulated'].groupby(['representation', 'lag']):
            ax[0].hist(g.Imin_synergy_fraction.dropna(), bins=20, histtype='step', label=f'{rep_}, lag {lag}')
        ax[0].set(xlabel='Synergy fraction of joint MI (I_min)', ylabel='NOR gates'); ax[0].legend(fontsize=7)
        ax[1].scatter(RR.synergy_fraction, RR.te_percentile, s=10); ax[1].set(xlabel='Synergy fraction', ylabel='Percentile of input-edge TE (lag 1)', title=f'Spearman ρ = {rho.statistic:.2f}')
        rq_fig(fig, 'fig14_pid', 'Synergy in NOR gates and its relation to pairwise-TE ranks of their input edges.')

## 15. Realistic synthetic circuits: random NMOS logic on the switch-level engine

Random circuits are built from the 6502's own device vocabulary — depletion pull-ups, inverters, NOR (parallel pull-downs) and NAND (series pull-downs through an internal node), dynamic latches clocked by two non-overlapping phases with feedback through latches — and simulated with the same engine. Ground truth is the same node-level signal-flow reference used for the chip. Methods: zero-lag MI; pairwise TE at lags 0 and 1 (ranking; BH); IDTxl multivariate TE with source lag 1 and with source lags 0–1; link-wise bivariate TE. A calibration table compares basic statistics of these circuits, the simulated 6502 and the recording.

In [ ]:
NM_DIR = SIM_DIR / 'nmos'; NM_DIR.mkdir(exist_ok=True)
tasks = []
for rep in range(SIM['nmos_reps']):
    for T in SIM['nmos_T']:
        for representation in ['state', 'rising']:
            run_id = IDTXL_OK and rep < SIM['nmos_idtxl_reps'] and T <= SIM['nmos_idtxl_T_max']
            task = dict(seed=500 + rep, T=int(T), representation=representation, n_gates=SIM['nmos_gates'], n_inputs=SIM['nmos_inputs'],
                        n_latches=SIM['nmos_latches'], k_max=4, alpha=RQ['alpha'], run_idtxl=run_id,
                        idtxl_methods=['multivariate', 'multivariate_lag0', 'bivariate'], idtxl_max_groups=120, max_heap='1g')
            tasks.append((hashlib.sha256(json.dumps(task, sort_keys=True).encode()).hexdigest()[:16], task))
pending = [(k, t) for k, t in tasks if not (NM_DIR / f'{k}.json').exists()]
print(f'NMOS tasks: {len(tasks)} ({len(pending)} to run)', flush=True)
def _save_nm(k, r): (NM_DIR / f'{k}.json').write_text(json.dumps(rq_json(r)))
if pending and RQ['workers'] > 1:
    with ProcessPoolExecutor(max_workers=RQ['workers'], mp_context=mp.get_context('spawn')) as pool:
        futs = {pool.submit(tk.nmos_task, t): k for k, t in pending}
        for i, f in enumerate(as_completed(futs), 1):
            try: _save_nm(futs[f], f.result())
            except Exception: print(traceback.format_exc()[-800:])
            if i % 5 == 0 or i == len(pending): print(f'  {i}/{len(pending)}', flush=True)
else:
    for i, (k, t) in enumerate(pending, 1):
        _save_nm(k, tk.nmos_task(t)); print(f'  {i}/{len(pending)}', flush=True)
NM = pd.DataFrame([r for k, _ in tasks if (NM_DIR / f'{k}.json').exists() for r in json.loads((NM_DIR / f'{k}.json').read_text())['rows']])
NM.to_csv(RQ_TAB / 'nmos_all_runs.csv', index=False)
metric_cols = [c for c in ['AUC', 'AP_lift', 'precision', 'recall', 'MCC'] if c in NM]
NMS = NM.groupby(['method', 'representation', 'T'])[metric_cols].agg(['mean', 'std']).round(3)
NMS.columns = [f'{a}_{b}' for a, b in NMS.columns]; NMS = NMS.reset_index()
rq_table(NMS, 'nmos_summary', 'Synthetic NMOS circuits (switch-level): mean and SD across replicates by method, representation and length.')
rq_record('nmos_summary', NMS.to_dict('records'))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True, constrained_layout=True)
for ax, rep_ in zip(axes, ['state', 'rising']):
    for m in sorted(NM.method.unique()):
        d = NM[(NM.method == m) & (NM.representation == rep_)]
        col_ = 'AUC' if 'ranking' in m else 'MCC'
        g = d.groupby('T')[col_].mean()
        if len(g): ax.plot(g.index, g.values, 'o-', ms=3, label=f'{m} ({col_})')
    ax.set(xscale='log', xlabel='Half-steps', title=f'{rep_} representation', ylim=(-0.05, 1.05))
axes[0].set_ylabel('AUC (rankings) or MCC (networks)'); axes[1].legend(fontsize=6, bbox_to_anchor=(1, 1))
rq_fig(fig, 'fig15_nmos', 'Synthetic NMOS circuits: ranking AUC and network MCC versus length for lag-0/lag-1 pairwise TE, MI and IDTxl variants.')
# Calibration of synthetic circuits against the chip and the recording.
import sim6502 as _s
defs, meta = _s.make_nmos_circuit(n_gates=SIM['nmos_gates'], n_inputs=SIM['nmos_inputs'], n_latches=SIM['nmos_latches'], seed=500)
Xn, _c = _s.simulate_nmos(defs, meta, 6000, seed=513); Xn = Xn[200:].T
def stats_of(X, label):
    v = X[X.min(1) != X.max(1)]; reps_, _ = tk.group_identical(X)
    return dict(data=label, units=len(X), constant_fraction=1 - len(v) / len(X), distinct_fraction=len(reps_) / max(len(v), 1),
                median_event_rate=float(np.median(v.mean(1))), median_AIS_k=float(np.median([tk.ais_select_k(x, 6) for x in v[:200]])))
CAL = pd.DataFrame([stats_of(Xn, 'NMOS synthetic (state)'), stats_of(S_REPR['state'][:, :20000], 'simulated 6502 (state)'),
                    stats_of(S_REPR['rising'][:, :20000], 'simulated 6502 (rising)'), stats_of(RQ_X[:, :20000], 'recording (groups, rising)')])
rq_table(CAL, 'synthetic_calibration', 'Calibration: basic statistics of synthetic NMOS circuits, the simulated 6502 and the recording.')

## 16. Instruction-aligned information dynamics

Local (pointwise) TE and AIS are averaged across wired node pairs (and, for comparison, random unwired pairs) and aligned to the instruction cycle: the position since the last SYNC (opcode fetch) and the clock phase. This shows *when* within an instruction information is stored and transferred, and which instructions move most information. On the recording, the same alignment is made where the SYNC node is observable through a transistor it gates.

In [ ]:
d = S_GROUPS['state']; Xst = S_REPR['state']; T_ = Xst.shape[1]
sync = S_ANN['sync'].astype(int); clk = S_ANN['clk0'].astype(int)
starts = np.flatnonzero(np.diff(np.r_[0, sync]) == 1)
pos_ = np.full(T_, -1); op_ = np.full(T_, -1)
mem_ = np.frombuffer(S_MEM, np.uint8)
for i, s0 in enumerate(starts):
    e = starts[i + 1] if i + 1 < len(starts) else T_
    pos_[s0:e] = np.arange(e - s0); op_[s0:e] = mem_[S_ANN['address'][s0]]
MNEM = {0xA2: 'LDX #', 0xA9: 'LDA #', 0x85: 'STA zp', 0xA5: 'LDA zp', 0x9D: 'STA abs,X', 0x18: 'CLC', 0x65: 'ADC zp', 0xA4: 'LDY zp',
        0x84: 'STY zp', 0xE8: 'INX', 0xE0: 'CPX #', 0xD0: 'BNE', 0xA0: 'LDY #', 0x0A: 'ASL A', 0x90: 'BCC', 0x49: 'EOR #',
        0x99: 'STA abs,Y', 0xC8: 'INY', 0x4C: 'JMP abs'}
rng = np.random.default_rng(6); Asim = S_A.toarray() > 0
var_ = Xst.min(1) != Xst.max(1)
wired = np.argwhere(Asim & var_[:, None] & var_[None, :]); wired = wired[rng.choice(len(wired), min(SIM['local_edges'], len(wired)), replace=False)]
rand_ = np.argwhere(~Asim & ~np.eye(len(Asim), dtype=bool) & var_[:, None] & var_[None, :]); rand_ = rand_[rng.choice(len(rand_), len(wired), replace=False)]
def mean_local(pairs, lag):
    acc = np.zeros(T_); cnt = np.zeros(T_)
    for s_, t_ in pairs:
        k = tk.ais_select_k(Xst[t_], SIM['k_max']); l, st = tk.local_te(Xst[s_], Xst[t_], k, delay=lag)
        acc[st:] += l; cnt[st:] += 1
    return acc / np.maximum(cnt, 1)
series = {f'local TE lag {lag}, wired': mean_local(wired, lag) for lag in (0, 1)}
series['local TE lag 1, unwired'] = mean_local(rand_, 1)
ais_acc = np.zeros(T_); cnt = np.zeros(T_)
for u in np.flatnonzero(var_)[:300]:
    k = tk.ais_select_k(Xst[u], SIM['k_max']); l, st = tk.local_ais(Xst[u], k); ais_acc[st:] += l; cnt[st:] += 1
series['local AIS (nodes)'] = ais_acc / np.maximum(cnt, 1)
mask = (pos_ >= 0) & (pos_ < SIM['max_cycle_position'])
prof = pd.DataFrame({'cycle_position': pos_[mask], 'clock_high': clk[mask], **{k: v[mask] for k, v in series.items()}})
PROF = prof.groupby(['cycle_position', 'clock_high']).mean().reset_index()
rq_table(PROF, 'instruction_aligned_profile', 'Mean local TE and AIS by half-step position within the instruction and clock phase (simulated 6502).')
byop = pd.DataFrame({'opcode': [MNEM.get(int(o), hex(int(o))) for o in op_[op_ >= 0]], **{k: v[op_ >= 0] for k, v in series.items()}})
OPS = byop.groupby('opcode').mean().sort_values('local TE lag 1, wired', ascending=False).reset_index()
rq_table(OPS, 'instruction_profile_by_opcode', 'Mean local TE and AIS by instruction (simulated 6502).')
rq_record('instruction_aligned', dict(profile=PROF.to_dict('records'), by_opcode=OPS.to_dict('records')))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
for k in series:
    g = prof.groupby('cycle_position')[k].mean(); axes[0].plot(g.index, g.values, 'o-', ms=3, label=k)
axes[0].set(xlabel='Half-steps since opcode fetch (SYNC)', ylabel='Mean local value (bits)', title='(a) Information dynamics within instructions'); axes[0].legend(fontsize=7)
axes[1].barh(OPS.opcode, OPS['local TE lag 1, wired']); axes[1].invert_yaxis(); axes[1].set(xlabel='Mean local TE, lag 1, wired pairs (bits)', title='(b) By instruction')
rq_fig(fig, 'fig16_instruction_aligned', 'Instruction-aligned local transfer entropy and active information storage on the simulated 6502.')
# Recording: SYNC observable through a transistor gated by the sync node?
sync_node = CHIP.names['sync']; sync_groups = sorted({int(rq_p2g[t]) for t in range(RQ_M) if rq_gate[t] == sync_node and rq_p2g[t] >= 0})
if sync_groups:
    ev = RQ_X[sync_groups[0]].astype(int); st_ = np.flatnonzero(ev == 1); pos_r = np.full(RQ_N, -1)
    for i, s0 in enumerate(st_):
        e = st_[i + 1] if i + 1 < len(st_) else RQ_N; pos_r[s0:e] = np.arange(e - s0)
    pairs = np.argwhere((RQ_POS > 0) & RQ_EL); pairs = pairs[np.random.default_rng(7).choice(len(pairs), min(SIM['local_edges'], len(pairs)), replace=False)]
    acc = np.zeros(RQ_N); cnt = np.zeros(RQ_N)
    for s_, t_ in pairs:
        l, st = tk.local_te(RQ_X[s_], RQ_X[t_], int(RQ_K[t_])); acc[st:] += l; cnt[st:] += 1
    loc = acc / np.maximum(cnt, 1); m_ = (pos_r >= 0) & (pos_r < SIM['max_cycle_position'])
    RP = pd.DataFrame({'samples_since_sync': pos_r[m_], 'local_TE_wired': loc[m_]}).groupby('samples_since_sync').mean().reset_index()
    rq_table(RP, 'recording_sync_aligned', 'Recording: mean local TE over wired group pairs by samples since the SYNC-gated transistor fired.')
    rq_record('recording_sync_aligned', RP.to_dict('records'))
else:
    print('SYNC is not observable in the recording (no active transistor gated by the sync node); recording alignment skipped.')

# Part VI — Run integrity and outputs

## 17. Run health and results at a glance

Before any number is reported, the run is checked. The checks confirm:
* the run used the thesis configuration;
* the estimator agrees with JIDT;
* the simulated CPU executed correctly;
* every planned synthetic, NMOS and IDTxl task completed without failure;
* every required figure and table exists.

Any FAIL means the affected results must not be reported until it is resolved. A results summary is then generated from the saved tables, so the text quotes exactly what was computed.

In [ ]:
REQUIRED_FIGURES = ['fig05_coverage', 'fig06_null_calibration', 'fig07_synthetic_convergence', 'fig15_nmos', 'fig08_1_roc_pr', 'fig08_1_score_distributions', 'fig08_2_pvalue_distributions', 'fig08_3_direction', 'fig08_4_signed_effect', 'fig08_5_phase_delay', 'fig08_7_convergence', 'fig08_9_layout_maps', 'fig09_5_fp_anatomy', 'fig09_6_embedding_runtime', 'fig12_representation', 'fig13_module_te', 'fig14_pid', 'fig16_instruction_aligned']
REQUIRED_TABLES = ['jidt_units_audit', 'null_calibration_by_target', 'synthetic_summary', 'synthetic_recall_by_edge_class', 'synthetic_effect_sign', 'synthetic_idtxl_units_effect', 'nmos_summary', 'synthetic_calibration', 'rq1_discrimination', 'rq1_paired_contrasts', 'rq1_te_by_relation', 'rq1_decision_rules', 'rq1_direction', 'rq1_direction_incremental', 'rq1_signed_effect', 'rq1_signed_scores', 'rq1_phase_delay_sweep', 'rq1_shared_driver', 'rq1_convergence', 'rq1_incremental_prediction', 'rq2_network_comparison', 'rq2_paired_differences', 'rq2_false_positive_anatomy', 'rq2_source_lag_sensitivity', 'sim_wiring_vs_intervention', 'sim_lesion_criticality', 'sim_representation_scores', 'sim_idtxl', 'modules', 'pid_summary', 'instruction_aligned_profile', 'instruction_profile_by_opcode']

In [ ]:
health = []
def chk(name, ok, detail='', level='FAIL'):
    health.append(dict(check=name, status='PASS' if ok else level, detail=str(detail)[:200]))
chk('Thesis configuration (not quick smoke-test mode)', not TE6502_QUICK, 'TE6502_QUICK=1: numbers are NOT thesis results' if TE6502_QUICK else 'thesis settings')
au = RESULTS.get('jidt_units_audit', {})
chk('JIDT analytic convention identified', au.get('convention') in ('values_in_bits_null_in_nats', 'bit_consistent'), au.get('convention'))
ea = RESULTS.get('estimator_agreement_with_jidt', {})
chk('Exact TE estimator equals JIDT (< 1e-9 bits)', (ea.get('max_abs_difference_bits') if ea.get('max_abs_difference_bits') is not None else 1) < 1e-9, ea.get('max_abs_difference_bits'))
cov = RESULTS.get('coverage', {})
chk('Structural reference built', cov.get('structural_edges_evaluable', 0) > 0, cov.get('structural_edges_evaluable'))
sv = RESULTS.get('sim_validation', {})
chk('Simulated 6502 executes the test program exactly', bool(sv.get('fibonacci_values_correct')) and bool(sv.get('lfsr_correct')), sv)
done = sum((SYN_DIR / f'{k}.json').exists() for k, _ in syn_tasks)
chk('Synthetic logic-circuit tasks complete', done == len(syn_tasks), f'{done}/{len(syn_tasks)}')
nm_done = sum((NM_DIR / f'{k}.json').exists() for k, _ in tasks)
chk('NMOS synthetic tasks complete', nm_done == len(tasks), f'{nm_done}/{len(tasks)}')
chk('IDTxl installed', IDTXL_OK, 'install IDTxl to run Section 9' if not IDTXL_OK else 'ok')
if IDTXL_OK:
    fails = json.loads((IDX_DIR / 'failures.json').read_text()) if (IDX_DIR / 'failures.json').exists() else []
    chk('IDTxl jobs ran without failures', len(fails) == 0, f'{len(fails)} failures (see thesis_v5/idtxl/failures.json)')
    chk('IDTxl: every active target completed (both methods)', len(IDX_DONE) == len(IDX_TARGETS), f'{len(IDX_DONE)}/{len(IDX_TARGETS)} targets')
missing_f = [f for f in REQUIRED_FIGURES if not (RQ_FIG / f'{f}.pdf').exists()]
missing_t = [t for t in REQUIRED_TABLES if not (RQ_TAB / f'{t}.csv').exists()]
chk('All required figures present', not missing_f, missing_f or 'all present')
chk('All required tables present', not missing_t, missing_t or 'all present')
HEALTH = pd.DataFrame(health); (RQ_DIR / 'run_health.json').write_text(HEALTH.to_json(orient='records', indent=1))
display(HEALTH.style.apply(lambda r: ['background-color: #d4edda' if r.status == 'PASS' else 'background-color: #f8d7da'] * len(r), axis=1)
        if hasattr(HEALTH, 'style') else HEALTH)
n_fail = int((HEALTH.status != 'PASS').sum())
print('RUN HEALTH:', 'ALL CHECKS PASSED' if n_fail == 0 else f'{n_fail} check(s) need attention before reporting results')

In [ ]:
# Results at a glance: generated from the saved tables (quote these, not hand-copied numbers).
def tab(name):
    p = RQ_TAB / f'{name}.csv'
    return pd.read_csv(p) if p.exists() else None
L = ['# Results at a glance (generated)', '', f'Run directory: `{OUT.name}`; mode: {"SMOKE TEST — not thesis results" if TE6502_QUICK else "thesis configuration"}', '']
def add(title, fn):
    try:
        lines = fn()
        if lines: L.extend([f'**{title}**', ''] + [f'- {x}' for x in lines] + [''])
    except Exception as error:
        L.extend([f'**{title}**: unavailable ({type(error).__name__})', ''])
def rq1():
    d = tab('rq1_discrimination'); out = []
    for _, r in d.iterrows():
        if r.score in ('Pairwise TE', 'Pairwise TE, lag 0', 'Zero-lag MI'):
            out.append(f"{r.level}, {r.score}: AUC {r.AUC}; AP lift {r.AP_lift}")
    c = tab('rq1_paired_contrasts')
    for _, r in c[c.contrast == 'lag0−lag1'].iterrows():
        out.append(f"lag 0 − lag 1 ({r.level}, {r.metric}): {r['median']:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]")
    return out
def rq1b():
    out = []
    d = tab('rq1_decision_rules'); r = d[(d.rule == 'BH q ≤ α (bit-correct)') & (d.level == 'physical')].iloc[0]
    out.append(f"BH-significant pairs: {r.fraction_significant:.1%}; physical precision lift {r.precision_lift:.2f}; MCC {r.MCC:.4f}")
    d = tab('rq1_direction'); out += [f"{r.predictor}: {r.accuracy}" for _, r in d.iterrows()]
    d = tab('rq1_signed_effect'); out += [f"δ<0, {r.category}: {r.fraction_negative_group_pairs}" for _, r in d.iterrows()]
    d = tab('rq1_incremental_prediction'); out += [f"{r.model}: out-of-fold AUC {r.AUC_mean:.3f}; ΔAUC vs distance+rates {r.delta_AUC_vs_base:+.3f} ({int(r.folds_AUC_improved)}/5 folds)" for _, r in d.iterrows()]
    return out
def rq2():
    d = tab('rq2_network_comparison'); out = [f"{r.method} ({r.level}): precision {r.precision:.4f}, recall {r.recall:.3f}, MCC {r.MCC:.4f}, edges {int(r.edges)}" for _, r in d.iterrows()]
    p = tab('rq2_paired_differences')
    if p is not None:
        out += [f"{r.contrast} [{r.level}, {r.metric}]: {r.difference:+.4f} [{r.ci_low:+.4f}, {r.ci_high:+.4f}]" for _, r in p[p.metric.isin(['precision', 'MCC'])].iterrows()]
    return out
def gt():
    out = []
    d = tab('sim_representation_scores')
    for _, r in d[d.reference == 'wiring'].iterrows():
        out.append(f"simulated chip, {r.representation}, {r.score}: AUC vs wiring {r.AUC:.3f}")
    d = tab('sim_wiring_vs_intervention'); out += [f"{r.effect}: {r.wired_pairs_with_causal_effect:.0%} of wired pairs have a causal effect; {r.causal_pairs_that_are_wired:.1%} of causal pairs are wired" for _, r in d.iterrows()]
    d = tab('sim_lesion_criticality')
    if d is not None: out += [f"lesion criticality predicted by {r.predictor}: AUC {r.AUC_predicting_criticality:.2f} (n={int(r.lesions)})" for _, r in d.iterrows()]
    d = tab('modules'); out += [f"modules, {r.data}, {r.network}: within/between {r.within_between_ratio:.2f} (null 97.5% {r.null_97_5:.2f}, p={r.p_perm:.3f}); NMI {r.NMI:.2f}" for _, r in d.iterrows()]
    return out
add('RQ1 — discrimination (physical and group levels, 95% target-cluster CI)', rq1)
add('RQ1 — decisions, direction, sign and incremental value', rq1b)
add('RQ2 — networks on identical targets', rq2)
add('Ground-truth 6502', gt)
md_text = '\n'.join(L); (RQ_DIR / 'results_at_a_glance.md').write_text(md_text); display(Markdown(md_text))

## 18. Results ledger, figure and table index, provenance

Everything reported above is collected in `thesis_v5/results_ledger.json`; figures (PDF for the thesis, PNG for slides) are in `thesis_v5/figures`, and tables (CSV and booktabs LaTeX, ready for `\input{}`) are in `thesis_v5/tables`. The provenance record lists software versions and input hashes for the reproducibility appendix.

In [ ]:
import importlib.metadata as md_
def ver(p):
    try: return md_.version(p)
    except Exception: return 'not installed'
java = jpype.java.lang.System.getProperty('java.version') if jpype.isJVMStarted() else 'n/a'
prov = dict(python=platform.python_version(), platform=platform.platform(), java=str(java),
            packages={p: ver(p) for p in ['numpy', 'scipy', 'pandas', 'scikit-learn', 'matplotlib', 'JPype1', 'idtxl', 'statsmodels']},
            toolkit_version=tk.TOOLKIT_VERSION, toolkit_sha256=hashlib.sha256(Path(tk.__file__).read_bytes()).hexdigest(),
            manifest=json.loads((OUT / 'manifest.json').read_text()), configuration=RQ)
rq_record('provenance', prov)
(RQ_DIR / 'figure_index.md').write_text('\n'.join(f'- `{k}.pdf`: {v}' for k, v in FIGURES.items()))
(RQ_DIR / 'table_index.md').write_text('\n'.join(f'- `{k}.tex`: {v}' for k, v in TABLES.items()))
display(Markdown('**Figures**\n\n' + '\n'.join(f'- `{k}`: {v}' for k, v in FIGURES.items())))
display(Markdown('**Tables**\n\n' + '\n'.join(f'- `{k}`: {v}' for k, v in TABLES.items())))
print('Results ledger:', RQ_DIR / 'results_ledger.json')

## 19. Reproducible evidence package

Run this cell last, after saving the notebook. It archives the notebook, helper code, every numerical result, figure and table (including `thesis_v5/`), with SHA-256 hashes and software versions. The raw recording and the JIDT jar are referenced by hash rather than copied.

In [ ]:
checks_notebook = NOTEBOOK_PATH
checks_versions = {}
for package in ['numpy','scipy','pandas','matplotlib','scikit-learn','JPype1']:
    try: checks_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: checks_versions[package] = 'unavailable'
checks_summary = dict(created_utc=datetime.now(timezone.utc).isoformat(),python=sys.version,
    platform=platform.platform(),machine=platform.machine(),packages=checks_versions,
    java_version=str(jpype.JClass('java.lang.System').getProperty('java.version')),
    manifest=json.loads((OUT/'manifest.json').read_text()),configuration=CHECK_CFG,
    provenance=checks_provenance,clock=globals().get('checks_clock_metadata'),
    replication_status='computed' if globals().get('checks_replication') else 'not run (no second recording supplied)',
    mte_configuration=globals().get('ME_CFG'),
    external_inputs=[dict(path=str(path),sha256=sha256_file(path),bytes=path.stat().st_size)
                     for path in [DATA_PATH,JAR_PATH,ROOT/'data/visual6502/transdefs.js']])
checks_json(CHECK_DIR/'evidence_summary.json',checks_summary)
checks_files = {checks_notebook}
checks_files.update((ROOT/'notebooks').glob('*.py'))
checks_files.update(path for path in [ROOT/'notebooks/requirements.txt'] if path.exists())
checks_files.update(path for path in OUT.glob('*') if path.is_file() and path.suffix in {'.json','.npz','.csv'})
for directory in [OUT/'analysis_pairwise',CHECK_DIR,OUT/'thesis_v5',globals().get('ME_RUN',OUT/'no_mte_run')]:
    if directory.exists():
        checks_files.update(path for path in directory.rglob('*') if path.is_file()
            and path.suffix.lower() in {'.json','.csv','.png','.pdf','.npz'} and 'targets' not in path.parts)
checks_inventory = [dict(path=str(path.relative_to(ROOT)),bytes=path.stat().st_size,sha256=sha256_file(path))
                    for path in sorted(checks_files)]
checks_archive_dir = ROOT/'deliverables/evidence_packages'; checks_archive_dir.mkdir(parents=True, exist_ok=True)
checks_archive = checks_archive_dir/f"TE_evidence_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')}.zip"
with zipfile.ZipFile(checks_archive,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(checks_files): archive.write(path,str(path.relative_to(ROOT)))
    archive.writestr('inventory.json',json.dumps(checks_inventory,indent=2))
    archive.writestr('README.txt','Run the saved notebook in order with the input files identified in remaining_checks/evidence_summary.json (under the results directory).\nRaw data and JIDT jar are external inputs. No larger embedding range was tested.\nUnverified semantics, missing clock measurements and absent replication recordings remain limitations.\nThe notebook is the version saved on disk before packaging; save it before running the final cell.\n')
print('Evidence package:',checks_archive)
print('Files:',len(checks_inventory),'Archive SHA-256:',sha256_file(checks_archive))